# Import Packages

In [ ]:
# =============================================================================
# 1. Import Packages
# =============================================================================
# All imports used in this notebook.

# --- Standard library ---
import json
import os
import pickle
import warnings
from pathlib import Path

# --- Scientific computing & data analysis ---
import numpy as np
import pandas as pd
import pymannkendall as mk
import xarray as xr
from scipy import stats
from scipy.ndimage import uniform_filter1d

# --- Geospatial & cartography ---
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import regionmask
from shapely.geometry import Point

# --- Visualization & styling ---
import cmaps  # NCL colormaps
import cmcrameri.cm as cmc  # Scientific colormaps (Crameri)
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from matplotlib.colors import (
    BoundaryNorm,
    ListedColormap,
    TwoSlopeNorm,
    to_rgba,
)
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter, MaxNLocator, MultipleLocator
from matplotlib.transforms import blended_transform_factory

# --- Jupyter notebook configuration ---
%matplotlib inline
warnings.filterwarnings("ignore", category=UserWarning, module="cartopy")
pd.options.display.max_columns = None

In [ ]:
# Font embedding
mpl.rcParams["font.sans-serif"] = ["Arial"]
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

# =============================================================================
# 2. Project paths and plotting settings
# =============================================================================
# Set input/output folders relative to the notebook.

NOTEBOOK_DIR = Path.cwd()
FIGDATA_DIR = NOTEBOOK_DIR / "FigData"
FIGURE_DIR = NOTEBOOK_DIR / "Figure"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Configure matplotlib parameters for consistent figure styling.
DOUBLE_COLUMN_FIG_WIDTH = 7.09  # Double column
SINGLE_COLUMN_FIG_WIDTH = 3.54  # Single column
mpl.rcParams["figure.figsize"] = [3.54, 2.36]
mpl.rcParams["figure.constrained_layout.use"] = True
mpl.rcParams["figure.dpi"] = 150
mpl.rcParams["axes.labelsize"] = 10
mpl.rcParams["axes.titlesize"] = 10
mpl.rcParams["axes.titlepad"] = 4  # Padding for title
mpl.rcParams["axes.linewidth"] = 0.5  # Frame line width
mpl.rcParams["xtick.labelsize"] = 10
mpl.rcParams["ytick.labelsize"] = 10
mpl.rcParams["xtick.major.width"] = 1
mpl.rcParams["xtick.major.size"] = 6
mpl.rcParams["xtick.minor.width"] = 1
mpl.rcParams["xtick.minor.size"] = 3
mpl.rcParams["ytick.major.width"] = 1
mpl.rcParams["ytick.major.size"] = 6
mpl.rcParams["ytick.minor.width"] = 1
mpl.rcParams["ytick.minor.size"] = 3
mpl.rcParams["xtick.direction"] = "in"
mpl.rcParams["ytick.direction"] = "in"

# Mapping Helper Function

In [ ]:
def create_map_features_adder(
    extent=[-9, 27, 35.6, 71],
    land_color="#FFFFFF",
    ocean_color="#FFFFFF",
    coastline_color="#000000",
    coastline_linewidth=0.7,
):
    """
    Create a reusable function to add map features to any axis.

    This factory function returns a closure that can be used to add
    consistent map features across multiple figures.

    Parameters
    ----------
    extent : list of float
        Map extent as [lon_min, lon_max, lat_min, lat_max] in
        PlateCarree coordinates
    land_color : str, optional
        Hex color code for land areas. Default is '#e3e3e3'
    ocean_color : str, optional
        Hex color code for ocean areas. Default is white '#FFFFFF'
    coastline_color : str, optional
        Hex color code for coastline. Default is '#000000'
    coastline_linewidth : float, optional
        Linewidth for coastline. Default is 0.25

    Returns
    -------
    function
        A function that takes an axis and adds map features to it
    """

    def add_map_features(ax):
        """Add common map features to an axis."""
        global europe_outline

        ax.set_extent(extent, crs=ccrs.PlateCarree())

        # --- THE FIX ---
        # Flatten everything with a zorder less than 4.
        # This bakes ocean (0), land (1), data (2), and coastlines (3) into one image.
        ax.set_rasterization_zorder(4)

        # Add ocean with facecolor (zorder=0 - bottom layer)
        ax.add_feature(
            cfeature.OCEAN.with_scale("50m"), facecolor=ocean_color, zorder=0
        )

        # Add land features with fill (zorder=1 - middle layer)
        ax.add_geometries(
            europe_outline.geometry,
            ccrs.PlateCarree(),
            facecolor=land_color,
            edgecolor="none",
            zorder=1,
            # rasterized=True
        )

        # Add coastline/outline on top (zorder=3 - top layer)
        ax.add_geometries(
            europe_outline.boundary,
            ccrs.PlateCarree(),
            facecolor="none",
            edgecolor=coastline_color,
            linewidth=coastline_linewidth,
            zorder=3,
            # rasterized=True
        )

        # Add gridlines
        gl = ax.gridlines(draw_labels=False, linewidth=0.0)

    return add_map_features


def set_axis_linewidth(ax, linewidth=0.1):
    """Set the linewidth for all spines of an axis."""
    for spine in ax.spines.values():
        spine.set_linewidth(linewidth)

In [ ]:
# Now create your map features adder
add_map_features = create_map_features_adder(
    extent=[-9, 27, 35.6, 71],
    land_color="#FFFFFF",
    ocean_color="#FFFFFF",
    coastline_color="#000000",
    coastline_linewidth=0.8,
)

# Colorbar

In [ ]:
colors_hex_viridis = ["#440154", "#433e85", "#2a788e", "#2fb47c", "#8ed645", "#dfe318"]
viridis_mod = ListedColormap(colors_hex_viridis)
viridis_mod

# Europe Outline

In [ ]:
europe_outline = gpd.read_file(str(FIGDATA_DIR / "europe_outline.gpkg"))

# Plot to verify
europe_outline.boundary.plot(edgecolor="black", linewidth=0.5)
plt.show()

# Prudence Region

In [ ]:
# Load region mask
europe_PRUDENCE_regions = regionmask.defined_regions.prudence
print(europe_PRUDENCE_regions)
europe_PRUDENCE_regions.plot()
plt.show()

# Plotting Function

## Prudence Region stats

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  compute_region_stats
# ══════════════════════════════════════════════════════════════════════════════


def compute_region_stats(
    data_old,
    data_new,
    region_mask,
    var_name: str = "mean_annual_frequency",
    fill_value: float = -99999999,
    duration_vars: list = None,
    duration_unit: str = "seconds",
):
    """
    Compute per-region descriptive statistics from spatially masked xarray data.

    Compatible with both plot_dot_range_8x4 and plot_strip_8x4.

    Parameters
    ----------
    data_old : xr.Dataset or xr.DataArray
        Data for the old period (e.g. 1961–1990).
    data_new : xr.Dataset or xr.DataArray
        Data for the new period (e.g. 1991–2020).
    region_mask : regionmask.Regions
        The regionmask Regions object.
    var_name : str
        Variable to extract if a Dataset is passed.
    fill_value : float
        Fill value to exclude alongside NaNs (default -99999).
    duration_vars : list or None
        Variable names whose raw values should be converted to days.
    duration_unit : {'seconds', 'nanoseconds', 'hours', 'days'}
        Storage unit of duration variables (converted to days automatically).

    Returns
    -------
    dict
        Nested dict keyed by region abbreviation. Each region dict contains:

        Summary statistics (scalars) — used by plot_dot_range_8x4
        ─────────────────────────────────────────────────────────────
            med_old / med_new    — median (Q50)
            mean_old / mean_new  — arithmetic mean
            q5_old  / q5_new     — 5th  percentile
            q10_old / q10_new    — 10th percentile
            q25_old / q25_new    — 25th percentile
            q75_old / q75_new    — 75th percentile
            q90_old / q90_new    — 90th percentile
            q95_old / q95_new    — 95th percentile

        Raw arrays — used by plot_strip_8x4 (ignored by plot_dot_range_8x4)
        ─────────────────────────────────────────────────────────────────────
            raw_old  — 1-D numpy array of all valid grid-point values (old)
            raw_new  — 1-D numpy array of all valid grid-point values (new)
    """

    _to_days = {
        "seconds": 1 / 86400,
        "nanoseconds": 1 / 86400e9,
        "hours": 1 / 24,
        "days": 1,
    }
    if duration_unit not in _to_days:
        raise ValueError(
            f"duration_unit must be one of {list(_to_days)}, got '{duration_unit}'"
        )
    scale = _to_days[duration_unit]

    def _extract_var(data, var):
        return data[var] if isinstance(data, xr.Dataset) else data

    da_old = _extract_var(data_old, var_name)
    da_new = _extract_var(data_new, var_name)

    if duration_vars is not None and var_name in duration_vars:
        da_old = da_old * scale
        da_new = da_new * scale

    mask_3d_old = region_mask.mask_3D(da_old)
    mask_3d_new = region_mask.mask_3D(da_new)

    def _region_values(da, mask_3d, region_number):
        masked = da.where(mask_3d.sel(region=region_number))
        flat = masked.values.ravel()
        valid = flat[(~np.isnan(flat)) & (flat != fill_value) & (np.abs(flat) < 1e6)]
        return valid

    def _stats(arr):
        return {
            "q5": float(np.percentile(arr, 5)),
            "q10": float(np.percentile(arr, 10)),
            "q25": float(np.percentile(arr, 25)),
            "med": float(np.percentile(arr, 50)),
            "q75": float(np.percentile(arr, 75)),
            "q90": float(np.percentile(arr, 90)),
            "q95": float(np.percentile(arr, 95)),
            "mean": float(np.mean(arr)),
        }

    out = {}
    for region in region_mask:
        reg = region.abbrev
        region_number = region.number

        arr_o = _region_values(da_old, mask_3d_old, region_number)
        arr_n = _region_values(da_new, mask_3d_new, region_number)

        if arr_o.size == 0 or arr_n.size == 0:
            print(
                f"  ⚠ Warning: region {reg} (index {region_number}) "
                f"returned no valid grid points — check your mask."
            )
            continue

        so = _stats(arr_o)
        sn = _stats(arr_n)

        out[reg] = {
            # summary statistics (scalars) — used by plot_dot_range_8x4
            "med_old": so["med"],
            "med_new": sn["med"],
            "mean_old": so["mean"],
            "mean_new": sn["mean"],
            "q5_old": so["q5"],
            "q5_new": sn["q5"],
            "q10_old": so["q10"],
            "q10_new": sn["q10"],
            "q25_old": so["q25"],
            "q25_new": sn["q25"],
            "q75_old": so["q75"],
            "q75_new": sn["q75"],
            "q90_old": so["q90"],
            "q90_new": sn["q90"],
            "q95_old": so["q95"],
            "q95_new": sn["q95"],
            # raw arrays — used by plot_strip_8x4
            "raw_old": arr_o,
            "raw_new": arr_n,
        }

    return out

In [ ]:
def compute_region_stats_EARLS(
    df,
    col_old,
    col_new,
    region_mask,
    basin_type="ungauged",
    lon_col="lon",
    lat_col="lat",
    verbose=True,
):
    """
    Compute per-PRUDENCE-region statistics from an EARLS basin CSV DataFrame.

    Parameters
    ----------
    df : pd.DataFrame
        Output of basin_annual_freq_duration.py loaded via pd.read_csv().
        Must contain lon_col, lat_col, col_old, col_new.
    col_old : str
        Column name for the baseline period values,
        e.g. 'AnnualFreq_1961to1990'.
    col_new : str
        Column name for the recent period values,
        e.g. 'AnnualFreq_1991to2020'.
    region_mask : regionmask Regions object
        e.g. regionmask.defined_regions.prudence
    basin_type : str
        'gauged' or 'ungauged'.  When 'ungauged', the 'hybas_' prefix is
        stripped from the idx column (matching plot_compound_metrics_EARLS
        behaviour).  Does not affect region assignment — included for
        console output and future use.
    lon_col / lat_col : str
        Column names for basin longitude and latitude.
    verbose : bool
        Print per-region basin counts and mean values.

    Returns
    -------
    dict
        Same nested structure as compute_region_stats():
        keyed by PRUDENCE abbreviation, each value contains scalar
        summary statistics plus 'raw_old' / 'raw_new' arrays.
    """
    # ── Validate columns ──────────────────────────────────────────────────────
    for col in (lon_col, lat_col, col_old, col_new):
        if col not in df.columns:
            raise ValueError(
                f"Column '{col}' not found in DataFrame.\n"
                f"Available columns: {list(df.columns)}"
            )

    # ── Drop rows with missing coordinates or values ──────────────────────────
    df_clean = df.dropna(subset=[lon_col, lat_col, col_old, col_new]).copy()
    if verbose:
        print(f"[compute_region_stats_EARLS] {basin_type} | " f"{col_old} / {col_new}")
        print(f"  Input rows: {len(df)}  →  valid: {len(df_clean)}")

    # ── Point-in-polygon region assignment ───────────────────────────────────
    # regionmask PRUDENCE regions have overlap=True so mask_3D() raises an
    # error on basin point data.  Shapely point-in-polygon is used instead.
    region_polys = {r.number: r.polygon for r in region_mask}

    def _assign(lon, lat):
        pt = Point(lon, lat)
        for rnum, poly in region_polys.items():
            if poly.contains(pt):
                return rnum
        return -1  # outside all regions

    df_clean["_region_num"] = [
        _assign(lo, la)
        for lo, la in zip(df_clean[lon_col].values, df_clean[lat_col].values)
    ]

    n_assigned = (df_clean["_region_num"] >= 0).sum()
    if verbose:
        print(
            f"  Basins assigned to a PRUDENCE region: " f"{n_assigned}/{len(df_clean)}"
        )

    # ── Per-region statistics ─────────────────────────────────────────────────
    def _stats(arr):
        """Return scalar summary statistics for a 1-D array."""
        return {
            "q5": float(np.percentile(arr, 5)),
            "q10": float(np.percentile(arr, 10)),
            "q25": float(np.percentile(arr, 25)),
            "med": float(np.percentile(arr, 50)),
            "q75": float(np.percentile(arr, 75)),
            "q90": float(np.percentile(arr, 90)),
            "q95": float(np.percentile(arr, 95)),
            "mean": float(np.mean(arr)),
        }

    out = {}

    for region in region_mask:
        abbrev = region.abbrev
        rnum = region.number

        sub = df_clean[df_clean["_region_num"] == rnum]

        if len(sub) == 0:
            if verbose:
                print(f"  ⚠  {abbrev}: no basins assigned — skipping.")
            continue

        arr_o = sub[col_old].values.astype(float)
        arr_n = sub[col_new].values.astype(float)

        # Remove any residual NaNs (should already be clean, but be safe)
        arr_o = arr_o[~np.isnan(arr_o)]
        arr_n = arr_n[~np.isnan(arr_n)]

        if len(arr_o) == 0 or len(arr_n) == 0:
            if verbose:
                print(f"  ⚠  {abbrev}: all values NaN after filtering — skipping.")
            continue

        so = _stats(arr_o)
        sn = _stats(arr_n)

        out[abbrev] = {
            # ── scalar summaries (used by plot_dot_range_8x4) ──────────────
            "med_old": so["med"],
            "med_new": sn["med"],
            "mean_old": so["mean"],
            "mean_new": sn["mean"],
            "q5_old": so["q5"],
            "q5_new": sn["q5"],
            "q10_old": so["q10"],
            "q10_new": sn["q10"],
            "q25_old": so["q25"],
            "q25_new": sn["q25"],
            "q75_old": so["q75"],
            "q75_new": sn["q75"],
            "q90_old": so["q90"],
            "q90_new": sn["q90"],
            "q95_old": so["q95"],
            "q95_new": sn["q95"],
            # ── raw arrays (used by plot_strip_8x4_EARLS) ──────────────────
            "raw_old": arr_o,
            "raw_new": arr_n,
        }

        if verbose:
            print(
                f"  {abbrev:3s}  n={len(sub):4d}  "
                f"mean_old={so['mean']:.3f}  mean_new={sn['mean']:.3f}  "
                f"Δ={sn['mean']-so['mean']:+.3f}"
            )

    return out

## Regional Map 1x4

In [ ]:
def plot_four_panel_analysis(
    data_past_1,
    data_present_1,
    data_past_2,
    data_present_2,
    variable_name_1,
    variable_name_2,
    bins_map_1=np.arange(0, 5.5, 0.5),
    bins_map_2=np.arange(0, 5.5, 0.5),
    bins_delta_1=np.array([-1.5, -1.0, -0.5, -0.025, 0.025, 1.0, 1.5]),
    bins_delta_2=np.array([-1.5, -1.0, -0.5, -0.025, 0.025, 1.0, 1.5]),
    yellow_range_1=(-0.025, 0.025),
    yellow_range_2=(-0.025, 0.025),
    use_nondefault_cmap=True,
    cmap_sequential_1=None,
    cmap_sequential_2=None,
    cmap_sequential_range=(0.1, 0.9),
    extend_colorbar_1="max",
    extend_colorbar_2="max",
    tick_positions_delta_1=[-1.5, -1, -0.5, 0, 0.5, 1, 1.5],
    tick_positions_delta_2=[-1.5, -1, -0.5, 0, 0.5, 1, 1.5],
    titles=None,
    panel_labels=["a", "b", "c", "d"],
    save_dir=str(FIGURE_DIR) + os.sep,
    filename=None,
    save_formats=["jpg", "pdf"],
    dpi=300,
    figsize=(260, 120),  # mm (width, height)
    title_fontsize=12,
    cbar_tick_fontsize=12,
    tick_interval_1=1,
    tick_interval_2=1,
    add_map_features_func=None,
):
    """
    Create a four-panel figure:
      Panel 1: Present period map for dataset 1
      Panel 2: Relative delta map for dataset 1 (present vs past)
      Panel 3: Present period map for dataset 2
      Panel 4: Relative delta map for dataset 2 (present vs past)

    Parameters
    ----------
    data_past_1 : xarray.Dataset or DataArray
        Past period data for dataset 1 (e.g. 1961–1990).
    data_present_1 : xarray.Dataset or DataArray
        Present period data for dataset 1 (e.g. 1991–2020).
    data_past_2 : xarray.Dataset or DataArray
        Past period data for dataset 2.
    data_present_2 : xarray.Dataset or DataArray
        Present period data for dataset 2.
    variable_name_1 : str
        Variable name to extract from dataset 1 (if Dataset).
    variable_name_2 : str
        Variable name to extract from dataset 2 (if Dataset).
    bins_map_1 : array-like
        Colormap bin edges for the sequential map of dataset 1.
    bins_map_2 : array-like
        Colormap bin edges for the sequential map of dataset 2.
    bins_delta_1 : array-like
        Bin edges for the diverging delta colormap of panel 2 (dataset 1).
    bins_delta_2 : array-like
        Bin edges for the diverging delta colormap of panel 4 (dataset 2).
    yellow_range_1 : tuple
        (min, max) values mapped to neutral yellow in delta panel 2.
    yellow_range_2 : tuple
        (min, max) values mapped to neutral yellow in delta panel 4.
    ...
    tick_positions_delta_1 : list
        Tick positions for the delta colorbar of panel 2 (dataset 1).
    tick_positions_delta_2 : list
        Tick positions for the delta colorbar of panel 4 (dataset 2).
    titles : dict or None
        Custom panel titles. Keys: 'panel1', 'panel2', 'panel3', 'panel4'.
        Defaults to generic labels if None.
    save_dir : str
        Directory for saved figures.
    filename : str or None
        Base filename (no extension). Figure is not saved when None.
    save_formats : list of str
        File formats to write, e.g. ['jpg', 'pdf'].
    dpi : int
        Resolution for raster formats.
    figsize : tuple
        Figure size in millimetres (width, height).
    title_fontsize : int
        Font size for panel titles and annotations.
    cbar_tick_fontsize : int
        Font size for colorbar tick labels.
    tick_interval_1 : int
        Show every Nth tick on the dataset-1 sequential colorbar.
    tick_interval_2 : int
        Show every Nth tick on the dataset-2 sequential colorbar.
    add_map_features_func : callable or None
        Function ``f(ax)`` that adds cartopy features (coastlines, borders …).

    Returns
    -------
    fig : matplotlib.figure.Figure
    """

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------
    MM_TO_INCH = 1 / 25.4
    figsize_inches = (figsize[0] * MM_TO_INCH, figsize[1] * MM_TO_INCH)

    mpl.rcParams["xtick.direction"] = "in"
    mpl.rcParams["ytick.direction"] = "in"

    FILL_VALUE = -99999999
    panel_labels = panel_labels
    panel_label_fontsize = title_fontsize - 1

    def _extract(dataset, varname):
        if hasattr(dataset, varname):
            return dataset[varname]
        return dataset

    def _compute_delta(past, present):
        """Relative change: (present - past) / past, masked for FILL_VALUE."""
        valid = (past != FILL_VALUE) & (present != FILL_VALUE)
        delta = xr.full_like(past, FILL_VALUE, dtype=float)
        delta = xr.where(valid & (past != 0), (present - past) / past, delta)
        delta = xr.where(valid & (past == 0), 0, delta)
        return delta

    def _build_seq_norm(
        bins_map,
        extend_colorbar,
        cmap_sequential,
        use_nondefault_cmap,
        cmap_sequential_range,
    ):
        n_colors = len(bins_map) + 1
        if use_nondefault_cmap and cmap_sequential is not None:
            cmap = cmap_sequential
        else:
            base = plt.get_cmap(cmap_sequential if cmap_sequential else "plasma")
            colors = base(
                np.linspace(
                    cmap_sequential_range[0], cmap_sequential_range[1], n_colors
                )
            )
            cmap = ListedColormap(colors)

        if extend_colorbar == "max":
            boundaries = np.concatenate((bins_map, [1e6]))
        else:  # 'both'
            boundaries = np.concatenate(([-99998], bins_map, [1e6]))
            cmap.set_under("#4D4D4D")

        norm = mcolors.BoundaryNorm(boundaries, n_colors)
        return cmap, norm

    def _build_delta_norm(bins_delta, yellow_range):
        boundaries = np.concatenate(([-99998], bins_delta, [1e6]))
        n_colors = len(boundaries) - 1

        negative_bins = bins_delta[bins_delta < yellow_range[0]]
        positive_bins = bins_delta[bins_delta > yellow_range[1]]

        RdYlBu_or = plt.cm.RdYlBu
        extreme_blue = RdYlBu_or(np.linspace(0.95, 1.00, 1))
        blue_colors = RdYlBu_or(np.linspace(0.90, 0.70, len(negative_bins)))
        yellow_colors = np.tile(to_rgba("#FFDF20"), (1, 1))
        red_colors = RdYlBu_or(np.linspace(0.30, 0.10, len(positive_bins)))
        extreme_red = RdYlBu_or(np.linspace(0.00, 0.05, 1))

        colors = np.vstack(
            [extreme_blue, blue_colors, yellow_colors, red_colors, extreme_red]
        )
        cmap = ListedColormap(colors)
        cmap.set_under("#4D4D4D")
        norm = mcolors.BoundaryNorm(boundaries, n_colors)
        return cmap, norm

    def set_axis_linewidth(ax, linewidth=0.0):
        for spine in ax.spines.values():
            spine.set_linewidth(linewidth)

    def _add_seq_colorbar(fig, im, ax, bins_map, tick_interval, cbar_tick_fontsize):
        cbar = plt.colorbar(
            im, ax=ax, orientation="horizontal", pad=0.02, shrink=0.9, aspect=20
        )
        cbar.ax.tick_params(labelsize=cbar_tick_fontsize)
        cbar.set_ticks(bins_map[::tick_interval])
        cbar.set_ticklabels([f"{v:g}" for v in bins_map[::tick_interval]])
        cbar.outline.set_linewidth(0.8)
        cbar.outline.set_edgecolor("black")
        cbar.ax.tick_params(
            labelsize=cbar_tick_fontsize, color="black", labelcolor="black"
        )

    def _add_delta_colorbar(fig, im, ax, tick_positions_delta, cbar_tick_fontsize):
        cbar = plt.colorbar(
            im, ax=ax, orientation="horizontal", pad=0.02, shrink=0.9, aspect=20
        )
        cbar.ax.tick_params(labelsize=cbar_tick_fontsize)
        cbar.set_ticks(tick_positions_delta)
        cbar.set_ticklabels([f"{x:g}" for x in tick_positions_delta])
        cbar.outline.set_linewidth(0.8)
        cbar.outline.set_edgecolor("black")
        cbar.ax.tick_params(
            labelsize=cbar_tick_fontsize, color="black", labelcolor="black"
        )

    # ------------------------------------------------------------------
    # Extract data
    # ------------------------------------------------------------------
    past_1 = _extract(data_past_1, variable_name_1)
    present_1 = _extract(data_present_1, variable_name_1)
    past_2 = _extract(data_past_2, variable_name_2)
    present_2 = _extract(data_present_2, variable_name_2)

    delta_1 = _compute_delta(past_1, present_1)
    delta_2 = _compute_delta(past_2, present_2)

    # ------------------------------------------------------------------
    # Colormaps & norms
    # ------------------------------------------------------------------
    cmap_seq_1, norm_seq_1 = _build_seq_norm(
        bins_map_1,
        extend_colorbar_1,
        cmap_sequential_1,
        use_nondefault_cmap,
        cmap_sequential_range,
    )

    cmap_seq_2, norm_seq_2 = _build_seq_norm(
        bins_map_2,
        extend_colorbar_2,
        cmap_sequential_2,
        use_nondefault_cmap,
        cmap_sequential_range,
    )

    cmap_delta_1, norm_delta_1 = _build_delta_norm(bins_delta_1, yellow_range_1)
    cmap_delta_2, norm_delta_2 = _build_delta_norm(bins_delta_2, yellow_range_2)

    # ------------------------------------------------------------------
    # Default titles
    # ------------------------------------------------------------------
    if titles is None:
        titles = {
            "panel1": f"{variable_name_1} — Present",
            "panel2": f"{variable_name_1} — Rel. Δ",
            "panel3": f"{variable_name_2} — Present",
            "panel4": f"{variable_name_2} — Rel. Δ",
        }

    # ------------------------------------------------------------------
    # Figure layout
    # ------------------------------------------------------------------
    fig = plt.figure(figsize=figsize_inches)

    gs = GridSpec(1, 4, figure=fig, wspace=0.1, width_ratios=[1, 1, 1, 1])

    projection = ccrs.LambertConformal(
        central_latitude=53,
        central_longitude=12,
        standard_parallels=(45, 65),
    )

    common_plot_kw = dict(
        add_colorbar=False,
        transform=ccrs.PlateCarree(),
        zorder=2,
        rasterized=True,
        antialiased=False,
        edgecolors="none",
        linewidth=0,
    )

    # ------------------------------------------------------------------
    # Panel 1 – present, dataset 1
    # ------------------------------------------------------------------
    ax1 = fig.add_subplot(gs[0, 0], projection=projection)
    if add_map_features_func:
        add_map_features_func(ax1)

    im1 = present_1.plot(ax=ax1, cmap=cmap_seq_1, norm=norm_seq_1, **common_plot_kw)
    _add_seq_colorbar(fig, im1, ax1, bins_map_1, tick_interval_1, cbar_tick_fontsize)
    set_axis_linewidth(ax1)
    ax1.set_title(titles["panel1"], fontsize=title_fontsize, pad=4, fontweight="bold")

    # ------------------------------------------------------------------
    # Panel 2 – delta, dataset 1
    # ------------------------------------------------------------------
    ax2 = fig.add_subplot(gs[0, 1], projection=projection)
    if add_map_features_func:
        add_map_features_func(ax2)

    im2 = delta_1.plot(ax=ax2, cmap=cmap_delta_1, norm=norm_delta_1, **common_plot_kw)
    _add_delta_colorbar(fig, im2, ax2, tick_positions_delta_1, cbar_tick_fontsize)
    set_axis_linewidth(ax2)
    ax2.set_title(titles["panel2"], fontsize=title_fontsize, pad=4, fontweight="bold")

    # ------------------------------------------------------------------
    # Panel 3 – present, dataset 2
    # ------------------------------------------------------------------
    ax3 = fig.add_subplot(gs[0, 2], projection=projection)
    if add_map_features_func:
        add_map_features_func(ax3)

    im3 = present_2.plot(ax=ax3, cmap=cmap_seq_2, norm=norm_seq_2, **common_plot_kw)
    _add_seq_colorbar(fig, im3, ax3, bins_map_2, tick_interval_2, cbar_tick_fontsize)
    set_axis_linewidth(ax3)
    ax3.set_title(titles["panel3"], fontsize=title_fontsize, pad=4, fontweight="bold")

    # ------------------------------------------------------------------
    # Panel 4 – delta, dataset 2
    # ------------------------------------------------------------------
    ax4 = fig.add_subplot(gs[0, 3], projection=projection)
    if add_map_features_func:
        add_map_features_func(ax4)

    im4 = delta_2.plot(ax=ax4, cmap=cmap_delta_2, norm=norm_delta_2, **common_plot_kw)
    _add_delta_colorbar(fig, im4, ax4, tick_positions_delta_2, cbar_tick_fontsize)
    set_axis_linewidth(ax4)
    ax4.set_title(titles["panel4"], fontsize=title_fontsize, pad=4, fontweight="bold")

    # ------------------------------------------------------------------
    # Panel labels (a) … (d)
    # ------------------------------------------------------------------
    for ax, label in zip([ax1, ax2, ax3, ax4], panel_labels):
        ax.text(
            0.0,
            1.0,
            f"({label})",
            transform=ax.transAxes,
            fontsize=panel_label_fontsize,
            fontweight="bold",
            va="top",
            ha="left",
            zorder=100,
        )

    # ------------------------------------------------------------------
    # Save
    # ------------------------------------------------------------------
    if filename is not None:
        os.makedirs(save_dir, exist_ok=True)
        for fmt in save_formats:
            filepath = os.path.join(save_dir, f"{filename}.{fmt}")
            if fmt.lower() in ["jpg", "jpeg", "png", "tiff"]:
                fig.savefig(filepath, dpi=dpi, format=fmt, bbox_inches="tight")
            else:
                fig.savefig(filepath, format=fmt, bbox_inches="tight")
            print(f"✓ Saved: {filepath}")

    return fig

## Regional Map 1x3

In [ ]:
def plot_three_panel_analysis(
    data_1961to1990,
    data_1991to2020,
    variable_name,
    CE_MISSING_FLAG,
    bins_map=np.arange(0, 5.5, 0.5),
    bins_delta=np.array([-1.5, -1.0, -0.5, -0.025, 0.1, 0.025, 1.0, 1.5]),
    yellow_range=(-0.025, 0.025),
    use_nondefault_cmap=True,
    cmap_sequential=None,
    cmap_sequential_range=(0.1, 0.9),
    extend_colorbar="max",
    kde_ylim=(0, 1),
    kde_yticks=None,
    kde_xlim=None,
    kde_xticks=None,
    kde_bounded=False,
    kde_lower_bound=0,
    print_kde_integrals=True,
    tick_positions_delta=[-3, -1, 0, 1, 3],
    titles=None,
    panel_labels=["a", "b", "c"],
    save_dir=str(FIGURE_DIR) + os.sep,
    filename=None,
    save_formats=["jpg", "pdf"],
    dpi=120,
    figsize=(200, 120),
    title_fontsize=12,
    cbar_tick_fontsize=12,
    tick_interval=1,
    add_map_features_func=None,
    kde_position_adjust=(0.05, -0.078, 0.00, 0.01),
    # --- strip plot + IQR box parameters ---
    strip_jitter=0.012,  # half-width of uniform vertical y-jitter (axes coords)
    strip_jitter_x=0.015,  # half-width of uniform horizontal x-jitter (data coords)
    # set > 0 to add horizontal spread, e.g. 0.02
    strip_alpha=0.1,  # transparency of strip dots
    strip_size=2,  # dot size in points²
    mean_fontsize=10,  # font size of the mean value annotation near the diamond
    verbose=True,
):
    """
    Create three-panel figure: 1991-2020 map, delta map, and KDE + strip + IQR box.

    Design choices for panel 3
    --------------------------
    - Strip plot sits BELOW the x-axis spine (clip_on=False, negative axes-coord y).
    - Two strip rows are close together to read as one compact comparison unit.
    - IQR box: thin edge only — no fill, no whiskers, no median line, no caps.
      Height tightly wraps the jitter cloud (box_half_h = strip_jitter + small margin).
    - Mean diamond: transparent fill, black edge, centred inside the IQR box.
    - Mean annotation: past (1961-1990) label above the box, present (1991-2020) below.
    - x and y axes show integer ticks and labels only.

    Delta (relative-change) handling:
    ----------------------------------
    Panel 2 shows (data_1991 - data_1961) / data_1961, computed per grid
    cell with the following rules, applied in order:

    1. Invalid input: if EITHER period's value equals the module-level
       CE_MISSING_FLAG sentinel (-99999999, "never checked"), the grid is
       excluded from the delta calculation entirely and stays at
       CE_MISSING_FLAG in the output. This also applies to any other
       flag-derived value that isn't a real 0 or NaN by the time it
       reaches this function -- see point 4 below.
    2. Nonzero baseline (data_1961 != 0): the ordinary relative change is
       computed as (data_1991 - data_1961) / data_1961.
    3. Zero baseline, still zero (data_1961 == 0 and data_1991 == 0): set
       explicitly to 0. This is a genuine "no change" and avoids a 0/0
       division.
    4. Zero baseline, now nonzero (data_1961 == 0 and data_1991 != 0): set
       to NEWLY_EMERGED_FLAG (99999999, an arbitrary large sentinel
       defined locally in this function). A relative change from a zero
       baseline has no finite percentage representation ("0 events -> N
       events" is an infinite increase), so rather than defaulting to 0
       (which would silently hide the fact that a hazard has newly
       appeared in that grid), it is tagged so it can be identified and
       given distinct treatment. NOTE: as written, boundaries_delta /
       cmap_delta do not currently give this sentinel its own color --
       any value beyond the last bin edge (1e6) renders as the same
       "extreme high" color as an ordinary large percentage increase, so
       a 99999999 cell is visually indistinguishable from, say, a +500%
       cell unless the colormap/boundaries are extended to treat it
       separately.

    Note on the CE_ZERO_EVENTS_FLAG / CE_MISSING_FLAG / NOT_RIVER_CELL_FLAG
    flags from analyze_annual_summer_lowFlow_heatwave_events: by the time
    data reaches this function, grid_number has already been dropped and
    those flags have already been resolved into literal statistic values
    (CE_ZERO_EVENTS_FLAG -> 0.0, NOT_RIVER_CELL_FLAG -> NaN,
    CE_MISSING_FLAG -> -99999999 unchanged). This function cannot
    distinguish a "confirmed always-zero" grid from a grid that merely
    averaged to 0 for other reasons -- both are plain 0.0 and are handled
    identically by rule 3 or rule 4 above, whichever applies.

    Parameters
    ----------
    data_1961to1990 : xarray.Dataset or DataArray
    data_1991to2020 : xarray.Dataset or DataArray
    variable_name : str
    bins_map : array-like
        Bins for the sequential colormap on maps.
    bins_delta : array-like
        Bins for the diverging colormap on delta map.
    yellow_range : tuple
        (min, max) for yellow centre in delta plot.
    cmap_sequential : str or colormap
        Sequential colormap name (default: 'plasma').
    cmap_sequential_range : tuple
        (min, max) range to sample from sequential colormap.
    extend_colorbar : str
        'max' or 'both'.
    kde_ylim : tuple
    kde_yticks : array-like or None
    kde_xlim : tuple or None
    kde_bounded : bool
        Use bounded KDE with reflection method.
    kde_lower_bound : float
    print_kde_integrals : bool
    tick_positions_delta : list
    titles : dict or None
        Keys: 'panel1', 'panel2', 'panel3'.
    panel_labels : list of str
    save_dir : str
    filename : str or None
        If None the figure is not saved.
    save_formats : list of str
    dpi : int
    figsize : tuple
        (width, height) in millimetres.
    title_fontsize : int
    cbar_tick_fontsize : int
    tick_interval : int
    add_map_features_func : callable or None
    kde_position_adjust : tuple
        (x_offset, y_offset, w_offset, h_offset) for KDE panel position.
    strip_jitter : float
        Half-width of uniform vertical y-jitter for strip dots (axes coords).
        Keep small so the IQR box wraps tightly.
    strip_jitter_x : float
        Half-width of uniform horizontal x-jitter for strip dots (data coords).
        0 = no horizontal jitter (default). Units match the variable being plotted.
    strip_alpha : float
        Alpha of strip plot dots.
    strip_size : float
        Marker size of strip plot dots in points².
    mean_fontsize : int
        Font size of the mean value annotation near the diamond. Default 10.
    verbose : bool

    Returns
    -------
    fig : matplotlib.figure.Figure
    """

    # ------------------------------------------------------------------
    # Figure setup
    # ------------------------------------------------------------------
    MM_TO_INCH = 1 / 25.4
    figsize_inches = (figsize[0] * MM_TO_INCH, figsize[1] * MM_TO_INCH)

    mpl.rcParams["xtick.direction"] = "in"
    mpl.rcParams["ytick.direction"] = "in"

    panel_label_fontsize = title_fontsize - 1

    # ------------------------------------------------------------------
    # Extract xarray variables
    # ------------------------------------------------------------------
    data_1961 = (
        data_1961to1990[variable_name]
        if hasattr(data_1961to1990, variable_name)
        else data_1961to1990
    )
    data_1991 = (
        data_1991to2020[variable_name]
        if hasattr(data_1991to2020, variable_name)
        else data_1991to2020
    )

    # ------------------------------------------------------------------
    # Delta (relative change) -- see docstring "Delta (relative-change)
    # handling" above for the full rule-by-rule rationale.
    # ------------------------------------------------------------------
    CE_MISSING_FLAG = CE_MISSING_FLAG

    # Arbitrary "newly emerged" sentinel for the zero-baseline -> nonzero
    # case (rule 4). Deliberately chosen far outside the bins_delta /
    # boundaries_delta range so it is never confused with a genuine
    # bounded relative-change percentage.
    NEWLY_EMERGED_FLAG = 99999999

    # Valid only if BOTH periods are valid (rule 1)
    valid_mask = (data_1961 != CE_MISSING_FLAG) & (data_1991 != CE_MISSING_FLAG)

    # Initialize delta with CE_MISSING_FLAG everywhere
    delta_data = xr.full_like(data_1961, CE_MISSING_FLAG, dtype=float)

    # Rule 2: nonzero baseline -> ordinary relative change
    delta_data = xr.where(
        valid_mask & (data_1961 != 0),
        (data_1991 - data_1961) / data_1961,
        delta_data,
    )

    # Rule 3: zero baseline, still zero -> explicit 0 (genuine no-change,
    # avoids 0/0 division)
    delta_data = xr.where(
        valid_mask & (data_1961 == 0) & (data_1991 == 0),
        0,
        delta_data,
    )

    # Rule 4: zero baseline, now nonzero -> newly emerged hazard, flagged
    # distinctly rather than silently collapsed to 0
    delta_data = xr.where(
        valid_mask & (data_1961 == 0) & (data_1991 != 0),
        NEWLY_EMERGED_FLAG,
        delta_data,
    )

    # ------------------------------------------------------------------
    # Titles
    # ------------------------------------------------------------------
    if titles is None:
        titles = {
            "panel1": "1991–2020",
            "panel2": "Relative Δ",
            "panel3": "Pan-Europe Distribution",
        }

    # ------------------------------------------------------------------
    # Figure / GridSpec
    # ------------------------------------------------------------------
    fig = plt.figure(figsize=figsize_inches)
    gs = GridSpec(1, 3, figure=fig, wspace=0.1, width_ratios=[1, 1, 1])

    projection = ccrs.LambertConformal(
        central_latitude=53,
        central_longitude=12,
        standard_parallels=(45, 65),
    )

    # ------------------------------------------------------------------
    # Sequential colormap
    # ------------------------------------------------------------------
    n_colors_seq = len(bins_map) + 1
    if use_nondefault_cmap and cmap_sequential is not None:
        cmap_seq = cmap_sequential
    else:
        cmap_seq_base = plt.get_cmap(cmap_sequential if cmap_sequential else "plasma")
        colors_seq = cmap_seq_base(
            np.linspace(
                cmap_sequential_range[0], cmap_sequential_range[1], n_colors_seq
            )
        )
        cmap_seq = ListedColormap(colors_seq)

    if extend_colorbar == "max":
        boundaries_seq = np.concatenate((bins_map, [1e6]))
    elif extend_colorbar == "both":
        boundaries_seq = np.concatenate(([-99998], bins_map, [1e6]))
        cmap_seq.set_under("#4D4D4D")

    norm_seq = mcolors.BoundaryNorm(boundaries_seq, n_colors_seq)

    def set_axis_linewidth(ax, linewidth=0.0):
        for spine in ax.spines.values():
            spine.set_linewidth(linewidth)

    # ==================================================================
    # PANEL 1 — 1991-2020 map
    # ==================================================================
    ax1 = fig.add_subplot(gs[0, 0], projection=projection)
    # Rasterize everything at zorder < 3 (map data + boundary/outline features)
    ax1.set_rasterization_zorder(3)

    if add_map_features_func:
        add_map_features_func(ax1)

    im1 = data_1991.plot(
        ax=ax1,
        transform=ccrs.PlateCarree(),
        add_colorbar=False,
        cmap=cmap_seq,
        norm=norm_seq,
        zorder=2,
        rasterized=True,
        **{"antialiased": False, "edgecolors": "none", "linewidth": 0},
    )

    cbar1 = plt.colorbar(
        im1, ax=ax1, orientation="horizontal", pad=0.02, shrink=0.9, aspect=20
    )
    # Rasterize the colorbar filled patches
    cbar1.solids.set_rasterized(True)
    cbar1.ax.tick_params(labelsize=cbar_tick_fontsize)
    cbar1.set_ticks(bins_map[::tick_interval])
    cbar1.set_ticklabels([f"{v:g}" for v in bins_map[::tick_interval]])
    cbar1.outline.set_linewidth(0.8)
    cbar1.outline.set_edgecolor("black")
    cbar1.ax.tick_params(
        labelsize=cbar_tick_fontsize, color="black", labelcolor="black"
    )
    set_axis_linewidth(ax1, linewidth=0.0)
    ax1.set_title(titles["panel1"], fontsize=title_fontsize, pad=4, fontweight="bold")
    ax1.title.set_zorder(10)

    # ==================================================================
    # PANEL 2 — delta map
    # ==================================================================
    ax2 = fig.add_subplot(gs[0, 1], projection=projection)
    # Rasterize everything at zorder < 3 (map data + boundary/outline features)
    ax2.set_rasterization_zorder(3)

    boundaries_delta = np.concatenate(([-99998], bins_delta, [1e6]))
    n_colors_delta = len(boundaries_delta) - 1

    negative_bins = bins_delta[bins_delta < yellow_range[0]]
    positive_bins = bins_delta[bins_delta > yellow_range[1]]

    RdYlBu_or = plt.cm.RdYlBu
    extreme_blue = RdYlBu_or(np.linspace(0.95, 1.0, 1))
    blue_colors = RdYlBu_or(np.linspace(0.90, 0.70, len(negative_bins)))
    yellow_colors = np.tile(to_rgba("#FFDF20"), (1, 1))
    red_colors = RdYlBu_or(np.linspace(0.30, 0.10, len(positive_bins)))
    extreme_red = RdYlBu_or(np.linspace(0.0, 0.05, 1))

    colors_delta = np.vstack(
        [extreme_blue, blue_colors, yellow_colors, red_colors, extreme_red]
    )
    cmap_delta = ListedColormap(colors_delta)
    cmap_delta.set_under("#4D4D4D")
    norm_delta = mcolors.BoundaryNorm(boundaries_delta, n_colors_delta)

    if add_map_features_func:
        add_map_features_func(ax2)

    im2 = delta_data.plot(
        ax=ax2,
        transform=ccrs.PlateCarree(),
        add_colorbar=False,
        cmap=cmap_delta,
        norm=norm_delta,
        zorder=2,
        rasterized=True,
        **{"antialiased": False, "edgecolors": "none", "linewidth": 0},
    )

    cbar2 = plt.colorbar(
        im2, ax=ax2, orientation="horizontal", pad=0.02, shrink=0.9, aspect=20
    )
    # Rasterize the colorbar filled patches
    cbar2.solids.set_rasterized(True)
    cbar2.ax.tick_params(labelsize=cbar_tick_fontsize)
    cbar2.set_ticks(tick_positions_delta)
    cbar2.set_ticklabels([f"{x:g}" for x in tick_positions_delta])
    cbar2.outline.set_linewidth(0.8)
    cbar2.outline.set_edgecolor("black")
    cbar2.ax.tick_params(
        labelsize=cbar_tick_fontsize, color="black", labelcolor="black"
    )

    set_axis_linewidth(ax2, linewidth=0.0)
    ax2.set_title(titles["panel2"], fontsize=title_fontsize, pad=4, fontweight="bold")
    ax2.title.set_zorder(10)

    # ==================================================================
    # PANEL 3 — KDE + strip plot + IQR box
    # ==================================================================
    ax3 = fig.add_subplot(gs[0, 2])

    # --- clean flat arrays ---
    data_1991_flat = data_1991.values.flatten()
    data_1961_flat = data_1961.values.flatten()

    if verbose:
        print(f"CE_MISSING_FLAG={CE_MISSING_FLAG}")

    mask_1991 = (~np.isnan(data_1991_flat)) & (data_1991_flat != CE_MISSING_FLAG)
    mask_1961 = (~np.isnan(data_1961_flat)) & (data_1961_flat != CE_MISSING_FLAG)
    data_1991_flat = data_1991_flat[mask_1991]
    data_1961_flat = data_1961_flat[mask_1961]

    if verbose:
        print(f"Valid grid cells (1961–1990): {data_1961_flat.size}")
        print(f"Valid grid cells (1991–2020): {data_1991_flat.size}")

    # --- KDE ---
    if kde_xlim is not None:
        x_min, x_max = kde_xlim
    else:
        x_min = min(data_1961_flat.min(), data_1991_flat.min())
        x_max = max(data_1961_flat.max(), data_1991_flat.max())

    x_range = np.linspace(x_min, x_max, 300)

    if kde_bounded:
        r61 = np.concatenate([data_1961_flat, 2 * kde_lower_bound - data_1961_flat])
        r91 = np.concatenate([data_1991_flat, 2 * kde_lower_bound - data_1991_flat])
        kde_1961 = stats.gaussian_kde(r61)
        kde_1991 = stats.gaussian_kde(r91)
        density_1961 = kde_1961(x_range) * 2
        density_1991 = kde_1991(x_range) * 2
        kde_method_label = f"Bounded KDE (lower bound={kde_lower_bound})"
    else:
        kde_1961 = stats.gaussian_kde(data_1961_flat)
        kde_1991 = stats.gaussian_kde(data_1991_flat)
        density_1961 = kde_1961(x_range)
        density_1991 = kde_1991(x_range)
        kde_method_label = "Standard KDE"

    color_1961 = "#4575b4"
    color_1991 = "#d73027"
    mean_1961 = np.mean(data_1961_flat)
    mean_1991 = np.mean(data_1991_flat)

    # ------------------------------------------------------------------
    # KDE curves + subtle fill
    # ------------------------------------------------------------------
    ax3.plot(
        x_range,
        density_1961,
        color=color_1961,
        linewidth=1.8,
        label="1961–1990",
        zorder=5,
    )
    ax3.plot(
        x_range,
        density_1991,
        color=color_1991,
        linewidth=1.8,
        label="1991–2020",
        zorder=5,
    )
    ax3.fill_between(x_range, density_1961, alpha=0.12, color=color_1961, zorder=4)
    ax3.fill_between(x_range, density_1991, alpha=0.12, color=color_1991, zorder=4)

    ax3.set_ylim(kde_ylim)
    if kde_xlim is not None:
        ax3.set_xlim(kde_xlim)

    # ------------------------------------------------------------------
    # Integer-only ticks and labels on both axes.
    # MaxNLocator(integer=True) places ticks only at integer values;
    # the default FuncFormatter then renders them without decimals.
    # If kde_yticks is provided it overrides the y locator.
    # ------------------------------------------------------------------
    ax3.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax3.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax3.tick_params(axis="both", labelsize=12)

    # ------------------------------------------------------------------
    # Blended transform: x = data coords, y = axes coords.
    # Strip rows sit BELOW the x-axis spine (negative axes-coord y values).
    # clip_on=False lets elements render outside the axes bounding box.
    # ------------------------------------------------------------------
    trans = blended_transform_factory(ax3.transData, ax3.transAxes)

    # Negative y values → below the spine (outward-offset spine creates
    # the visual gap between the x-axis and the strip rows).
    y_centre_1961 = -0.03  # first row  (1961-1990, closer to spine)
    y_centre_1991 = -0.065  # second row (1991-2020, slightly lower)

    # IQR box height snugly wraps the jitter cloud
    box_half_h = strip_jitter + 0.008
    lw_box = 0.7

    rng = np.random.default_rng(42)
    MAX_PTS = 3000

    # ------------------------------------------------------------------
    # STRIP PLOT  (vertical + optional horizontal jitter)
    # Rasterized=True applied to each scatter call for PDF output size.
    # ------------------------------------------------------------------
    for data_flat, yc, col in [
        (data_1961_flat, y_centre_1961, color_1961),
        (data_1991_flat, y_centre_1991, color_1991),
    ]:
        idx = rng.choice(
            len(data_flat), size=min(MAX_PTS, len(data_flat)), replace=False
        )
        sample = data_flat[idx]

        # vertical jitter (axes coords)
        jitter_y = rng.uniform(-strip_jitter, strip_jitter, size=len(idx))

        # horizontal jitter (data coords) — zero by default
        jitter_x = (
            rng.uniform(-strip_jitter_x, strip_jitter_x, size=len(idx))
            if strip_jitter_x > 0
            else np.zeros(len(idx))
        )

        ax3.scatter(
            sample + jitter_x,
            yc + jitter_y,
            s=strip_size,
            color=col,
            alpha=strip_alpha,
            linewidths=0,
            zorder=3,
            clip_on=False,
            transform=trans,
            rasterized=False,  # rasterize strip plot dots in PDF output
        )

    # ------------------------------------------------------------------
    # IQR box — thin edge only, no fill, no whiskers, no median, no caps
    # Mean diamond: transparent fill, black edge, centred inside the box.
    # Mean annotation:
    #   1961–1990 (past)    → label ABOVE the box top edge  (va='bottom')
    #   1991–2020 (present) → label BELOW the box bottom edge (va='top')
    # ------------------------------------------------------------------
    for data_flat, yc, col in [
        (data_1961_flat, y_centre_1961, color_1961),
        (data_1991_flat, y_centre_1991, color_1991),
    ]:
        q25 = np.percentile(data_flat, 25)
        q75 = np.percentile(data_flat, 75)
        mean = np.mean(data_flat)

        # IQR box
        box_rect = plt.Rectangle(
            (q25, yc - box_half_h),
            q75 - q25,
            2 * box_half_h,
            transform=trans,
            linewidth=lw_box,
            edgecolor=col,
            facecolor="none",
            zorder=9,
            clip_on=False,
        )
        ax3.add_patch(box_rect)

        # Mean diamond — centred at (mean, yc), transparent fill, black edge
        ax3.scatter(
            [mean],
            [yc],
            marker="D",
            s=22,
            facecolors="none",
            edgecolors="black",
            linewidths=0.8,
            zorder=11,
            clip_on=False,
            transform=trans,
        )

        # Mean annotation placement
        if col == color_1961:
            text_y = yc + box_half_h  # above the box top edge
            va_side = "bottom"
        else:
            text_y = yc - box_half_h - 0.01  # below the box bottom edge
            va_side = "top"

        ax3.annotate(
            f"{mean:.2f}",
            xy=(mean, text_y),
            xytext=(mean, text_y),
            xycoords=trans,
            textcoords=trans,
            fontweight="normal",
            ha="center",
            va=va_side,
            fontsize=mean_fontsize,
            color=col,
            annotation_clip=False,
        )

    # ------------------------------------------------------------------
    # Formatting
    # ------------------------------------------------------------------
    ax3.set_ylabel("Density", fontsize=12, fontweight="normal")
    ax3.set_title(titles["panel3"], fontsize=title_fontsize, pad=8, fontweight="bold")

    # Override y locator if explicit yticks are requested
    if kde_yticks is not None:
        ax3.set_yticks(kde_yticks)

    # Override x locator if explicit yticks are requested
    if kde_xticks is not None:
        ax3.set_xticks(kde_xticks)

    ax3.spines["top"].set_visible(False)
    ax3.spines["right"].set_visible(False)
    ax3.spines["left"].set_linewidth(1)
    ax3.spines["left"].set_position(("outward", 5))
    ax3.spines["bottom"].set_linewidth(1)
    ax3.spines["bottom"].set_position(("outward", 28))
    ax3.set_aspect("auto")

    ax3.legend(
        ["1961–1990", "1991–2020"],
        loc="upper right",
        fontsize=11,
        frameon=False,
        bbox_to_anchor=(1.05, 0.98),
    )

    # ------------------------------------------------------------------
    # Align KDE panel height with map panels
    # ------------------------------------------------------------------
    plt.draw()
    pos0 = ax1.get_position()
    pos2 = ax3.get_position()
    ax3.set_position(
        [
            pos2.x0 + kde_position_adjust[0],
            pos0.y0 + kde_position_adjust[1],
            pos2.width + kde_position_adjust[2],
            pos0.height + kde_position_adjust[3],
        ]
    )

    # ------------------------------------------------------------------
    # Panel labels
    # ------------------------------------------------------------------
    for ax, label in zip([ax1, ax2, ax3], panel_labels):
        x_pos = -0.23 if ax is ax3 else 0.0
        ax.text(
            x_pos,
            1.0,
            f"({label})",
            transform=ax.transAxes,
            fontsize=panel_label_fontsize,
            fontweight="bold",
            va="top",
            ha="left",
            zorder=100,
        )

    # ==================================================================
    # PRINT KDE INTEGRALS AND STATISTICS
    # ==================================================================
    if print_kde_integrals and verbose:
        dx = x_range[1] - x_range[0]
        integral_1961 = np.trapezoid(density_1961, dx=dx)
        integral_1991 = np.trapezoid(density_1991, dx=dx)

        median_1961 = np.median(data_1961_flat)
        median_1991 = np.median(data_1991_flat)
        q25_1961 = np.percentile(data_1961_flat, 25)
        q75_1961 = np.percentile(data_1961_flat, 75)
        q25_1991 = np.percentile(data_1991_flat, 25)
        q75_1991 = np.percentile(data_1991_flat, 75)
        iqr_1961 = q75_1961 - q25_1961
        iqr_1991 = q75_1991 - q25_1991

        print("=" * 70)
        print(f"KDE Method: {kde_method_label}")
        print(f"Variable:   {variable_name}")
        print("-" * 70)
        print(f"X-range:              [{x_range[0]:.4f}, {x_range[-1]:.4f}]")
        print(f"Integral (1961-1990): {integral_1961:.4f}")
        print(f"Integral (1991-2020): {integral_1991:.4f}")

        if not kde_bounded and kde_lower_bound is not None:
            x_below = np.linspace(x_range[0], kde_lower_bound, 100)
            if x_range[0] < kde_lower_bound:
                leak_1961 = np.trapezoid(
                    kde_1961(x_below), dx=(x_below[1] - x_below[0])
                )
                leak_1991 = np.trapezoid(
                    kde_1991(x_below), dx=(x_below[1] - x_below[0])
                )
                print("-" * 70)
                print(f"Probability leaked below {kde_lower_bound}:")
                print(f"  1961-1990: {leak_1961:.4f} ({leak_1961*100:.2f}%)")
                print(f"  1991-2020: {leak_1991:.4f} ({leak_1991*100:.2f}%)")

        print("-" * 70)
        print("DESCRIPTIVE STATISTICS:")
        print("-" * 70)
        print(f"{'Statistic':<20} {'1961-1990':>15} {'1991-2020':>15} {'Change':>15}")
        print("-" * 70)
        print(
            f"{'Mean':<20} {mean_1961:>15.4f} {mean_1991:>15.4f} {(mean_1991-mean_1961):>15.4f}"
        )
        print(
            f"{'Median':<20} {median_1961:>15.4f} {median_1991:>15.4f} {(median_1991-median_1961):>15.4f}"
        )
        print(
            f"{'Min':<20} {data_1961_flat.min():>15.4f} {data_1991_flat.min():>15.4f} {(data_1991_flat.min()-data_1961_flat.min()):>15.4f}"
        )
        print(
            f"{'Max':<20} {data_1961_flat.max():>15.4f} {data_1991_flat.max():>15.4f} {(data_1991_flat.max()-data_1961_flat.max()):>15.4f}"
        )
        print(
            f"{'Q25 (25th %ile)':<20} {q25_1961:>15.4f} {q25_1991:>15.4f} {(q25_1991-q25_1961):>15.4f}"
        )
        print(
            f"{'Q75 (75th %ile)':<20} {q75_1961:>15.4f} {q75_1991:>15.4f} {(q75_1991-q75_1961):>15.4f}"
        )
        print(
            f"{'IQR (Q75-Q25)':<20} {iqr_1961:>15.4f} {iqr_1991:>15.4f} {(iqr_1991-iqr_1961):>15.4f}"
        )
        print("-" * 70)
        print(
            f"{'Mean Relative Change:':<40} {((mean_1991-mean_1961)/mean_1961):>15.4f}"
        )
        print(
            f"{'Median Relative Change:':<40} {((median_1991-median_1961)/median_1961):>15.4f}"
        )
        print("-" * 70)
        print(f"Bandwidth (1961-1990): {kde_1961.factor:.6f}")
        print(f"Bandwidth (1991-2020): {kde_1991.factor:.6f}")
        print("=" * 70)

    # ==================================================================
    # SAVE
    # ==================================================================
    if filename is not None:
        os.makedirs(save_dir, exist_ok=True)
        for fmt in save_formats:
            filepath = os.path.join(save_dir, f"{filename}.{fmt}")
            if fmt.lower() in ["jpg", "jpeg", "png", "tiff"]:
                fig.savefig(filepath, dpi=dpi, format=fmt, bbox_inches="tight")
            else:
                fig.savefig(filepath, format=fmt, bbox_inches="tight")
            if verbose:
                print(f"✓ Saved: {filepath}")

    return fig

## Regional Strip Plot 8x2

In [ ]:
# ── palette (identical to plot_strip_8x4) ────────────────────────────────────
C_PAST = "#4575b4"  # 1961–1990 dots + diamond (blue)
C_PRES = "#d73027"  # 1991–2020 dots + diamond (red)
C_GRID = "#e8e6df"  # vertical grid lines
C_SEP = "#dddbd4"  # column dividers
C_TEXT = "#2e2e2e"  # bold region abbreviation + column headers
C_MUTED = "#aaa8a2"  # italic region names
C_ANNOT_B = "#1a4f9c"  # past  value annotation (above strip)
C_ANNOT_R = "#9e1a1a"  # present value annotation (below strip)

REGIONS = ["FR", "ME", "AL", "EA", "MD", "SC", "IP", "BI"]
REGION_NAMES = {
    "FR": "France",
    "ME": "Mid-Europe",
    "SC": "Scandinavia",
    "MD": "Mediterranean",
    "EA": "Eastern Europe",
    "AL": "Alps",
    "IP": "Iberian Pen.",
    "BI": "British Isles",
}

_DOT_CHOICES = ("median", "mean")


def plot_strip_8x2(
    stats_lfe: dict,
    stats_hwe: dict,
    # ── column titles ─────────────────────────────────────────────────────────
    lfe_title: str = "Low Flow Events  (LFE)",
    hwe_title: str = "Heat Wave Events  (HWE)",
    # ── x-axis labels ─────────────────────────────────────────────────────────
    lfe_xlabel: str = None,
    hwe_xlabel: str = None,
    # ── x-axis limits ────────────────────────────────────────────────────────
    xmin_lfe: float = 0.0,
    xmax_lfe: float = 2.8,
    xmin_hwe: float = 0.0,
    xmax_hwe: float = 3.0,
    # ── major tick interval per column ───────────────────────────────────────
    major_tick_lfe: float = 0.5,
    major_tick_hwe: float = 0.5,
    # ── horizontal jitter per column (fraction of x-range, 0 = off) ──────────
    h_jitter_lfe: float = 0.0,
    h_jitter_hwe: float = 0.0,
    # ── centre statistic ─────────────────────────────────────────────────────
    dot_stat: str = "mean",  # 'mean' or 'median'
    # ── strip appearance ──────────────────────────────────────────────────────
    alpha: float = 0.05,  # global dot transparency (0–1)
    jitter_frac: float = 0.55,  # vertical spread: fraction of half-strip height
    dot_size: float = 2.5,  # scatter marker area (pts²)
    dot_lw: float = 0.0,  # dot edge linewidth (0 = no edge)
    diamond_size: float = 18,  # diamond marker area (pts²)
    diamond_lw: float = 0.8,  # diamond black outline linewidth
    # ── per-region alpha ──────────────────────────────────────────────────────
    # Dict keyed by region abbreviation, e.g. {'MD': 0.03, 'IP': 0.03}.
    # Regions not listed fall back to the global `alpha`.
    region_alpha: dict = None,
    # ── layout ────────────────────────────────────────────────────────────────
    figwidth_mm: float = 174,
    row_height_mm: float = 8.5,
    top_mm: float = 12,
    bottom_mm: float = 10,
    label_col_mm: float = 28,
    gap_col_mm: float = 4,
    strip_sep_frac: float = 0.30,  # fraction of row height between strip centres
    # ── font sizes ───────────────────────────────────────────────────────────
    col_header_fs: float = 9.0,
    region_abbrev_fs: float = None,
    region_name_fs: float = None,
    tick_fs: float = None,
    annot_fs: float = None,
    panel_label: str = "(a)",
    # ── regions ───────────────────────────────────────────────────────────────
    regions: list = None,
    # ── save ─────────────────────────────────────────────────────────────────
    save_dir: str = str(FIGURE_DIR) + os.sep,
    filename: str = None,
    save_formats: list = None,
    dpi: int = 300,
    seed: int = 42,
):
    """
    8×2 strip plot for PRUDENCE regions across two datasets (LFE and HWE).

    Drop-in replacement for plot_dot_range_8x2 — same call signature,
    strip+diamond rendering instead of dot+IQR boxes.

    Each region row has two horizontal strips:
      • upper strip — 1961–1990 (past,    blue)
      • lower strip — 1991–2020 (present, red)

    Parameters
    ----------
    stats_lfe, stats_hwe : dict
        Output of compute_region_stats(). Must contain 'raw_old', 'raw_new',
        and either 'mean_old'/'mean_new' or 'med_old'/'med_new' per region.
    dot_stat : {'mean', 'median'}
        Statistic shown by the diamond marker.
    alpha : float
        Global dot transparency in [0, 1].
    region_alpha : dict or None
        Per-region dot transparency. Keys = region abbreviations.
        Regions not listed fall back to global `alpha`.
        E.g. {'MD': 0.03, 'IP': 0.03}
    h_jitter_lfe, h_jitter_hwe : float
        Optional horizontal jitter as a fraction of that column's x-range.
        Default 0.0 (no horizontal jitter).
    lfe_xlabel, hwe_xlabel : str or None
        X-axis label per column. None = hidden (default).
    panel_label : str
        Short label at top-left, e.g. '(a)'.

    Returns
    -------
    matplotlib.figure.Figure

    Example
    -------
    >>> fig = plot_strip_8x2(
    ...     stats_lfe    = stats_lfe_freq,
    ...     stats_hwe    = stats_hwe_freq,
    ...     lfe_title    = 'LF Frequency',
    ...     hwe_title    = 'HW Frequency',
    ...     lfe_xlabel   = 'Events per year',
    ...     hwe_xlabel   = 'Events per year',
    ...     xmin_lfe=0.0, xmax_lfe=1.4, major_tick_lfe=0.5,
    ...     xmin_hwe=0.0, xmax_hwe=2.9, major_tick_hwe=1.0,
    ...     dot_stat     = 'mean',
    ...     alpha        = 0.05,
    ...     region_alpha = {'MD': 0.03, 'IP': 0.03},
    ...     panel_label  = '(a)',
    ...     filename     = 'Fig1_a_strip_8x2',
    ...     save_formats = ['pdf', 'jpg'],
    ... )
    """
    # ── validate ──────────────────────────────────────────────────────────────
    if dot_stat not in _DOT_CHOICES:
        raise ValueError(f"dot_stat must be one of {_DOT_CHOICES}, got '{dot_stat}'")

    # ── resolve font sizes ────────────────────────────────────────────────────
    region_abbrev_fs = (
        region_abbrev_fs if region_abbrev_fs is not None else col_header_fs
    )
    region_name_fs = region_name_fs if region_name_fs is not None else col_header_fs - 2
    tick_fs = tick_fs if tick_fs is not None else col_header_fs
    annot_fs = annot_fs if annot_fs is not None else (diamond_size**0.5) - 1

    # ── stat key ──────────────────────────────────────────────────────────────
    centre_key = "med" if dot_stat == "median" else "mean"

    region_list = regions if regions is not None else REGIONS
    nR = len(region_list)

    # ── per-region alpha lookup ───────────────────────────────────────────────
    _region_alpha = {reg: alpha for reg in region_list}
    if region_alpha is not None:
        for reg, a in region_alpha.items():
            if reg in _region_alpha:
                _region_alpha[reg] = a
            else:
                warnings.warn(
                    f"region_alpha key '{reg}' not found in region_list — ignored.",
                    UserWarning,
                    stacklevel=2,
                )

    # ── bundle columns ────────────────────────────────────────────────────────
    columns = [
        (
            "lfe",
            stats_lfe,
            lfe_title,
            lfe_xlabel,
            xmin_lfe,
            xmax_lfe,
            major_tick_lfe,
            h_jitter_lfe,
        ),
        (
            "hwe",
            stats_hwe,
            hwe_title,
            hwe_xlabel,
            xmin_hwe,
            xmax_hwe,
            major_tick_hwe,
            h_jitter_hwe,
        ),
    ]
    nC = len(columns)

    # ── figure ────────────────────────────────────────────────────────────────
    plt.close("all")
    total_h_mm = top_mm + nR * row_height_mm + bottom_mm
    fw = figwidth_mm / 25.4
    fh = total_h_mm / 25.4

    fig = plt.figure(figsize=(fw, fh), facecolor="white")
    fig.set_layout_engine("none")

    left_frac = label_col_mm / figwidth_mm
    right_frac = 1 - 8 / figwidth_mm
    top_frac = 1 - top_mm / total_h_mm
    bottom_frac = bottom_mm / total_h_mm
    mid_gap = gap_col_mm / figwidth_mm

    plot_w = (right_frac - left_frac - mid_gap) / 2
    col_starts = [left_frac, left_frac + plot_w + mid_gap]
    row_h_frac = (top_frac - bottom_frac) / nR

    # ── axes grid ─────────────────────────────────────────────────────────────
    axes = {"lfe": [], "hwe": []}
    for ri in range(nR):
        y0 = top_frac - (ri + 1) * row_h_frac
        for ci, (key, *_) in enumerate(columns):
            ax = fig.add_axes([col_starts[ci], y0, plot_w, row_h_frac])
            axes[key].append(ax)

    rng = np.random.default_rng(seed)

    # ── row drawing ───────────────────────────────────────────────────────────
    def draw_row(ax, st, xmin, xmax, ri, reg, major_tick_interval, h_jitter_frac):
        is_last = ri == nR - 1

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(0, 1)
        ax.set_yticks([])
        ax.set_facecolor("white")

        for side in ("top", "left", "right"):
            ax.spines[side].set_visible(False)

        if is_last:
            ax.spines["bottom"].set_linewidth(0.6)
            ax.spines["bottom"].set_color("black")
            ax.spines["bottom"].set_position(("outward", 2))
            ax.tick_params(
                axis="x",
                labelsize=tick_fs,
                length=3.5,
                width=0.5,
                pad=1.5,
                colors="black",
                labelcolor="black",
            )
            ax.xaxis.set_major_locator(MultipleLocator(major_tick_interval))
            ax.xaxis.set_major_formatter(
                FuncFormatter(lambda x, _: f"{int(x)}" if x == int(x) else f"{x:.1f}")
            )
            ax.xaxis.set_minor_locator(MultipleLocator(major_tick_interval / 2))
            ax.tick_params(
                axis="x", which="minor", length=2.5, width=0.4, colors="black"
            )
        else:
            ax.spines["bottom"].set_visible(False)
            ax.tick_params(axis="x", bottom=False, labelbottom=False)

        # Vertical grid lines
        for gv in np.arange(
            np.ceil((xmin + 1e-9) / major_tick_interval) * major_tick_interval,
            xmax + 1e-9,
            major_tick_interval,
        ):
            ax.axvline(gv, color=C_GRID, linewidth=0.35, zorder=0)

        # ── strip geometry ────────────────────────────────────────────────────
        cy = 0.5
        half_sep = strip_sep_frac / 2
        cy_past = cy + half_sep
        cy_pres = cy - half_sep
        jitter_amp = half_sep * jitter_frac
        h_jitter_amp = (xmax - xmin) * h_jitter_frac
        row_alpha = _region_alpha[reg]

        # ── retrieve data ─────────────────────────────────────────────────────
        raw_old = st.get("raw_old", None)
        raw_new = st.get("raw_new", None)
        centre_old = st.get(f"{centre_key}_old", None)
        centre_new = st.get(f"{centre_key}_new", None)

        # ── jittered dots ─────────────────────────────────────────────────────
        def draw_strip(raw, cy_strip, color):
            if raw is None or len(raw) == 0:
                return
            vals = np.asarray(raw, dtype=float)
            if h_jitter_amp > 0:
                vals = vals + rng.uniform(-h_jitter_amp, h_jitter_amp, size=len(vals))
            vals = vals[(vals >= xmin) & (vals <= xmax)]
            if len(vals) == 0:
                return
            jitter = rng.uniform(-jitter_amp, jitter_amp, size=len(vals))
            ax.scatter(
                vals,
                cy_strip + jitter,
                s=dot_size,
                color=color,
                alpha=row_alpha,
                linewidths=dot_lw,
                edgecolors="none" if dot_lw == 0 else color,
                zorder=3,
                rasterized=True,
                transform=ax.transData,
            )

        draw_strip(raw_old, cy_past, C_PAST)
        draw_strip(raw_new, cy_pres, C_PRES)

        # ── diamond markers ───────────────────────────────────────────────────
        def draw_diamond(x_val, cy_strip):
            if x_val is None or np.isnan(x_val):
                return
            ax.scatter(
                [x_val],
                [cy_strip],
                s=diamond_size,
                marker="D",
                facecolors=(0, 0, 0, 0),  # transparent fill
                edgecolors="black",
                linewidths=diamond_lw,
                zorder=7,
                transform=ax.transData,
            )

        draw_diamond(centre_old, cy_past)
        draw_diamond(centre_new, cy_pres)

        # ── defer annotations ─────────────────────────────────────────────────
        if centre_old is not None and not np.isnan(centre_old):
            _annots.append(
                dict(
                    ax=ax,
                    x_data=centre_old,
                    xmin=xmin,
                    xmax=xmax,
                    y_axes=cy_past + 0.18,
                    label=f"{centre_old:.2f}",
                    ha="right",
                    va="bottom",
                    color=C_ANNOT_B,
                )
            )
        if centre_new is not None and not np.isnan(centre_new):
            _annots.append(
                dict(
                    ax=ax,
                    x_data=centre_new,
                    xmin=xmin,
                    xmax=xmax,
                    y_axes=cy_pres - 0.18,
                    label=f"{centre_new:.2f}",
                    ha="left",
                    va="top",
                    color=C_ANNOT_R,
                )
            )

    # ── populate all panels ───────────────────────────────────────────────────
    _annots = []
    for key, stats, title, xlabel, xmin, xmax, mti, h_jit in columns:
        for ri, reg in enumerate(region_list):
            draw_row(
                axes[key][ri],
                stats[reg],
                xmin,
                xmax,
                ri,
                reg,
                major_tick_interval=mti,
                h_jitter_frac=h_jit,
            )

    # ── annotations on figure canvas (top layer) ──────────────────────────────
    fig.canvas.draw()
    for ann in _annots:
        ax = ann["ax"]
        pos = ax.get_position()
        x_norm = (ann["x_data"] - ann["xmin"]) / (ann["xmax"] - ann["xmin"])
        fig_x = pos.x0 + x_norm * pos.width
        fig_y = pos.y0 + ann["y_axes"] * pos.height
        fig.text(
            fig_x,
            fig_y,
            ann["label"],
            ha=ann["ha"],
            va=ann["va"],
            fontsize=annot_fs,
            color=ann["color"],
            zorder=999,
            clip_on=False,
        )

    # ── region labels ─────────────────────────────────────────────────────────
    for ri, reg in enumerate(region_list):
        pos = axes["lfe"][ri].get_position()
        cy_fig = (pos.y0 + pos.y1) / 2
        fig.text(
            left_frac - 4 / figwidth_mm,
            cy_fig + 0.008,
            reg,
            ha="right",
            va="center",
            fontsize=region_abbrev_fs,
            fontweight="bold",
            color=C_TEXT,
        )
        fig.text(
            left_frac - 4 / figwidth_mm,
            cy_fig - 0.012,
            REGION_NAMES.get(reg, reg),
            ha="right",
            va="center",
            fontsize=region_name_fs,
            style="italic",
            color=C_MUTED,
        )

    # ── column headers + panel label ──────────────────────────────────────────
    header_y = top_frac + 0.006
    fig.text(
        left_frac - 6 / figwidth_mm,
        header_y,
        panel_label,
        ha="left",
        va="bottom",
        fontsize=col_header_fs,
        fontweight="bold",
        color=C_TEXT,
    )
    for key, stats, title, xlabel, xmin, xmax, mti, h_jit in columns:
        pos = axes[key][0].get_position()
        cx = (pos.x0 + pos.x1) / 2
        fig.text(
            cx,
            header_y,
            title,
            ha="center",
            va="bottom",
            fontsize=col_header_fs,
            fontweight="bold",
            color="black",
        )

    # ── x-axis labels (None = hidden) ────────────────────────────────────────
    for key, stats, title, xlabel, xmin, xmax, mti, h_jit in columns:
        if xlabel is None:
            continue
        last_pos = axes[key][-1].get_position()
        cx = (last_pos.x0 + last_pos.x1) / 2
        fig.text(
            cx,
            last_pos.y0 - 0.04,
            xlabel,
            ha="center",
            va="top",
            fontsize=tick_fs + 0.5,
            color=C_MUTED,
        )

    # ── column separator line ─────────────────────────────────────────────────
    x_sep = col_starts[1] - mid_gap / 2
    fig.add_artist(
        plt.Line2D(
            [x_sep, x_sep],
            [bottom_frac, top_frac],
            transform=fig.transFigure,
            color=C_SEP,
            linewidth=0.4,
            zorder=0,
        )
    )

    # ── legend (commented out — uncomment to re-enable) ──────────────────────
    # legend_handles = [
    #     mpatches.Patch(facecolor=C_PAST, alpha=0.8, label='1961–1990'),
    #     mpatches.Patch(facecolor=C_PRES, alpha=0.8, label='1991–2020'),
    # ]
    # fig.legend(
    #     handles=legend_handles,
    #     loc='lower left',
    #     bbox_to_anchor=(left_frac, 0.0),
    #     bbox_transform=fig.transFigure,
    #     ncol=2, fontsize=col_header_fs - 1,
    #     frameon=False, handlelength=1.0,
    #     handletextpad=0.4, columnspacing=1.0,
    # )

    # ── save / display ────────────────────────────────────────────────────────
    display(fig)

    if filename is not None:
        os.makedirs(save_dir, exist_ok=True)
        formats = save_formats if save_formats is not None else ["pdf"]
        for fmt in formats:
            fmt = fmt.lower().lstrip(".")
            filepath = os.path.join(save_dir, f"{filename}.{fmt}")
            kw = dict(bbox_inches="tight", facecolor="white")
            if fmt in ("jpg", "jpeg", "png", "tiff"):
                kw["dpi"] = dpi
            fig.savefig(filepath, format=fmt, **kw)
            print(f"✓  Saved: {filepath}")

    plt.close(fig)
    return fig

## Regional Strip Plot 8x4

In [ ]:
# ── palette ───────────────────────────────────────────────────────────────────
C_PAST = "#4575b4"  # 1961–1990 dots + centre marker (blue)
C_PRES = "#d73027"  # 1991–2020 dots + centre marker (red)
C_GRID = "#e8e6df"  # vertical grid lines
C_SEP = "#dddbd4"  # row separators + column dividers
C_TEXT = "#2e2e2e"  # bold region abbreviation + column headers
C_MUTED = "#aaa8a2"  # italic region names
C_ANNOT_B = "#1a4f9c"  # past  value annotation (above strip)
C_ANNOT_R = "#9e1a1a"  # present value annotation (below strip)

REGIONS = ["FR", "ME", "AL", "EA", "MD", "SC", "IP", "BI"]
REGION_NAMES = {
    "FR": "France",
    "ME": "Mid-Europe",
    "SC": "Scandinavia",
    "MD": "Mediterranean",
    "EA": "Eastern Europe",
    "AL": "Alps",
    "IP": "Iberian Pen.",
    "BI": "British Isles",
}

_DOT_CHOICES = ("median", "mean")

# ══════════════════════════════════════════════════════════════════════════════
#  plot_strip_8x4
# ══════════════════════════════════════════════════════════════════════════════


def plot_strip_8x4(
    stats_col1: dict,
    stats_col2: dict,
    stats_col3: dict,
    stats_col4: dict,
    # ── column headers ────────────────────────────────────────────────────────
    col1_title: str = "Column 1",
    col2_title: str = "Column 2",
    col3_title: str = "Column 3",
    col4_title: str = "Column 4",
    # ── x-axis labels ────────────────────────────────────────────────────────
    col1_xlabel: str = "Events per year",
    col2_xlabel: str = "Events per year",
    col3_xlabel: str = "Days per year",
    col4_xlabel: str = "Days per year",
    # ── x-axis limits ────────────────────────────────────────────────────────
    xmin_col1: float = 0.0,
    xmax_col1: float = 2.8,
    xmin_col2: float = 0.0,
    xmax_col2: float = 3.0,
    xmin_col3: float = 0.0,
    xmax_col3: float = 10.0,
    xmin_col4: float = 0.0,
    xmax_col4: float = 10.0,
    # ── major tick interval per column ────────────────────────────────────────
    major_tick_col1: float = 1.0,
    major_tick_col2: float = 1.0,
    major_tick_col3: float = 2.0,
    major_tick_col4: float = 2.0,
    # ── horizontal jitter per column (0 = off, default) ─────────────────────
    h_jitter_col1: float = 0.0,
    h_jitter_col2: float = 0.0,
    h_jitter_col3: float = 0.0,
    h_jitter_col4: float = 0.0,
    # ── centre statistic ─────────────────────────────────────────────────────
    dot_stat: str = "mean",  # 'mean' or 'median' — controls the diamond marker
    # ── strip appearance ──────────────────────────────────────────────────────
    alpha: float = 0.05,  # global dot transparency fallback (0–1)
    jitter_frac: float = 0.55,  # vertical spread: fraction of half-strip height
    dot_size: float = 2.5,  # jittered scatter marker area (pts²)
    dot_lw: float = 0.0,  # jittered dot edge linewidth (0 = no edge)
    diamond_size: float = 18,  # diamond marker area (pts²)
    diamond_lw: float = 0.8,  # diamond black outline linewidth
    # ── per-region alpha ──────────────────────────────────────────────────────
    # Dict keyed by region abbreviation, e.g. {'FR': 0.08, 'MD': 0.03}.
    # Any region not listed falls back to the global `alpha` value above.
    # Default None → all regions use the global `alpha`.
    region_alpha: dict = None,
    # ── layout ────────────────────────────────────────────────────────────────
    figwidth_mm: float = 190,
    row_height_mm: float = 9.5,
    top_mm: float = 12,
    bottom_mm: float = 10,
    label_col_mm: float = 30,
    gap_col_mm: float = 3,
    strip_sep_frac: float = 0.30,
    # ── font sizes ───────────────────────────────────────────────────────────
    col_header_fs: float = 9.0,
    region_abbrev_fs: float = None,
    region_name_fs: float = None,
    tick_fs: float = None,
    annot_fs: float = None,
    panel_label: str = "(a)",
    # ── regions ───────────────────────────────────────────────────────────────
    regions: list = None,
    # ── save ─────────────────────────────────────────────────────────────────
    save_dir: str = str(FIGURE_DIR) + os.sep,
    filename: str = None,
    save_formats: list = None,
    dpi: int = 300,
    seed: int = 42,
):
    """
    8×4 strip plot for PRUDENCE regions across four datasets.

    Each region row has two horizontal strips:
      • upper strip — 1961–1990 (past,    blue)
      • lower strip — 1991–2020 (present, red)

    Individual grid-point values from raw_old / raw_new are drawn as jittered
    dots.  A hollow diamond marker (black outline) shows the chosen central
    statistic (mean or median), with a value annotation above/below drawn
    on the figure canvas so it is never occluded.

    Parameters
    ----------
    stats_col1 … stats_col4 : dict
        Output of compute_region_stats(). Must contain 'raw_old', 'raw_new',
        and either 'mean_old'/'mean_new' or 'med_old'/'med_new' per region.
    dot_stat : {'mean', 'median'}
        Which statistic the diamond marker represents.
    alpha : float
        Global dot transparency in [0, 1]. Used for any region not listed in
        region_alpha. Default 0.05.
    region_alpha : dict or None
        Per-region dot transparency, keyed by region abbreviation.
        E.g. {'FR': 0.08, 'MD': 0.03, 'IP': 0.03}.
        Regions not listed fall back to the global `alpha`.
        Default None → all regions use `alpha = 0.05`.

        Quick-start example (override two regions, rest stay at global alpha):
            region_alpha = {'MD': 0.03, 'IP': 0.03}

        Set every region explicitly:
            region_alpha = {
                'FR': 0.05, 'ME': 0.05, 'AL': 0.05, 'EA': 0.05,
                'MD': 0.03, 'SC': 0.05, 'IP': 0.03, 'BI': 0.05,
            }

    Returns
    -------
    matplotlib.figure.Figure

    Example
    -------
    >>> fig = plot_strip_8x4(
    ...     **{f'stats_col{i+1}':      c['stats']      for i, c in enumerate(columns_def)},
    ...     **{f'col{i+1}_title':      c['title']      for i, c in enumerate(columns_def)},
    ...     **{f'col{i+1}_xlabel':     c['xlabel']     for i, c in enumerate(columns_def)},
    ...     **{f'xmin_col{i+1}':       c['xmin']       for i, c in enumerate(columns_def)},
    ...     **{f'xmax_col{i+1}':       c['xmax']       for i, c in enumerate(columns_def)},
    ...     **{f'major_tick_col{i+1}': c['major_tick'] for i, c in enumerate(columns_def)},
    ...     dot_stat     = 'mean',
    ...     alpha        = 0.05,           # default for all regions
    ...     region_alpha = {'MD': 0.03, 'IP': 0.03},  # override specific regions
    ...     panel_label  = '(g)',
    ...     filename     = 'Fig1_g_strip_8x4',
    ...     save_formats = ['pdf', 'jpg'],
    ... )
    """
    # ── validate ──────────────────────────────────────────────────────────────
    if dot_stat not in _DOT_CHOICES:
        raise ValueError(f"dot_stat must be one of {_DOT_CHOICES}, got '{dot_stat}'")

    # ── resolve font sizes ────────────────────────────────────────────────────
    region_abbrev_fs = (
        region_abbrev_fs if region_abbrev_fs is not None else col_header_fs
    )
    region_name_fs = region_name_fs if region_name_fs is not None else col_header_fs - 2
    tick_fs = tick_fs if tick_fs is not None else col_header_fs
    annot_fs = annot_fs if annot_fs is not None else (diamond_size**0.5) - 1

    # ── stat key ──────────────────────────────────────────────────────────────
    centre_key = "med" if dot_stat == "median" else "mean"

    region_list = regions if regions is not None else REGIONS
    nR = len(region_list)

    # ── build per-region alpha lookup ─────────────────────────────────────────
    # For every region in region_list: use region_alpha[reg] if provided,
    # otherwise fall back to the global `alpha`.
    _region_alpha = {reg: alpha for reg in region_list}  # start with global
    if region_alpha is not None:
        for reg, a in region_alpha.items():
            if reg in _region_alpha:
                _region_alpha[reg] = a
            else:
                warnings.warn(
                    f"region_alpha key '{reg}' not found in region_list — ignored.",
                    UserWarning,
                    stacklevel=2,
                )

    columns = [
        (
            stats_col1,
            col1_title,
            col1_xlabel,
            xmin_col1,
            xmax_col1,
            major_tick_col1,
            h_jitter_col1,
        ),
        (
            stats_col2,
            col2_title,
            col2_xlabel,
            xmin_col2,
            xmax_col2,
            major_tick_col2,
            h_jitter_col2,
        ),
        (
            stats_col3,
            col3_title,
            col3_xlabel,
            xmin_col3,
            xmax_col3,
            major_tick_col3,
            h_jitter_col3,
        ),
        (
            stats_col4,
            col4_title,
            col4_xlabel,
            xmin_col4,
            xmax_col4,
            major_tick_col4,
            h_jitter_col4,
        ),
    ]
    nC = len(columns)

    # ── figure ────────────────────────────────────────────────────────────────
    plt.close("all")
    total_h_mm = top_mm + nR * row_height_mm + bottom_mm
    fw = figwidth_mm / 25.4
    fh = total_h_mm / 25.4

    fig = plt.figure(figsize=(fw, fh), facecolor="white")
    fig.set_layout_engine("none")

    left_frac = label_col_mm / figwidth_mm
    right_frac = 1 - 4 / figwidth_mm
    top_frac = 1 - top_mm / total_h_mm
    bottom_frac = bottom_mm / total_h_mm
    mid_gap = gap_col_mm / figwidth_mm

    total_plot_w = right_frac - left_frac - (nC - 1) * mid_gap
    col_w = total_plot_w / nC
    col_x0 = [left_frac + i * (col_w + mid_gap) for i in range(nC)]
    row_h_frac = (top_frac - bottom_frac) / nR

    # ── axes grid ─────────────────────────────────────────────────────────────
    axes = [[None] * nR for _ in range(nC)]
    for ri in range(nR):
        y0 = top_frac - (ri + 1) * row_h_frac
        for ci in range(nC):
            ax = fig.add_axes([col_x0[ci], y0, col_w, row_h_frac])
            axes[ci][ri] = ax

    rng = np.random.default_rng(seed)

    # ── row drawing ───────────────────────────────────────────────────────────
    def draw_row(ax, st, xmin, xmax, ri, reg, major_tick_interval, h_jitter_frac):
        is_last = ri == nR - 1

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(0, 1)
        ax.set_yticks([])
        ax.set_facecolor("white")

        for side in ("top", "left", "right"):
            ax.spines[side].set_visible(False)

        if is_last:
            ax.spines["bottom"].set_linewidth(0.6)
            ax.spines["bottom"].set_color("black")
            ax.spines["bottom"].set_position(("outward", 2))
            ax.tick_params(
                axis="x",
                labelsize=tick_fs,
                length=3.5,
                width=0.5,
                pad=1.5,
                colors="black",
                labelcolor="black",
            )
            ax.xaxis.set_major_locator(MultipleLocator(major_tick_interval))
            ax.xaxis.set_major_formatter(
                FuncFormatter(lambda x, _: f"{int(x)}" if x == int(x) else f"{x:.1f}")
            )
            ax.xaxis.set_minor_locator(MultipleLocator(major_tick_interval / 2))
            ax.tick_params(
                axis="x", which="minor", length=2.5, width=0.4, colors="black"
            )
        else:
            ax.spines["bottom"].set_visible(False)
            ax.tick_params(axis="x", bottom=False, labelbottom=False)

        for gv in np.arange(
            np.ceil((xmin + 1e-9) / major_tick_interval) * major_tick_interval,
            xmax + 1e-9,
            major_tick_interval,
        ):
            ax.axvline(gv, color=C_GRID, linewidth=0.35, zorder=0)

        # ── strip geometry ────────────────────────────────────────────────────
        cy = 0.5
        half_sep = strip_sep_frac / 2
        cy_past = cy + half_sep
        cy_pres = cy - half_sep
        jitter_amp = half_sep * jitter_frac
        h_jitter_amp = (xmax - xmin) * h_jitter_frac

        # ── per-region alpha for this row ─────────────────────────────────────
        row_alpha = _region_alpha[reg]

        # ── retrieve data ─────────────────────────────────────────────────────
        raw_old = st.get("raw_old", None)
        raw_new = st.get("raw_new", None)
        centre_old = st.get(f"{centre_key}_old", None)
        centre_new = st.get(f"{centre_key}_new", None)

        # ── jittered dots ─────────────────────────────────────────────────────
        def draw_strip(raw, cy_strip, color):
            if raw is None or len(raw) == 0:
                return
            vals = np.asarray(raw, dtype=float)
            if h_jitter_amp > 0:
                vals = vals + rng.uniform(-h_jitter_amp, h_jitter_amp, size=len(vals))
            vals = vals[(vals >= xmin) & (vals <= xmax)]
            if len(vals) == 0:
                return
            jitter = rng.uniform(-jitter_amp, jitter_amp, size=len(vals))
            ax.scatter(
                vals,
                cy_strip + jitter,
                s=dot_size,
                color=color,
                alpha=row_alpha,  # ← per-region alpha
                linewidths=dot_lw,
                edgecolors="none" if dot_lw == 0 else color,
                zorder=3,
                rasterized=True,
                transform=ax.transData,
            )

        draw_strip(raw_old, cy_past, C_PAST)
        draw_strip(raw_new, cy_pres, C_PRES)

        # ── diamond markers ───────────────────────────────────────────────────
        def draw_diamond(x_val, cy_strip, face_color):
            if x_val is None or np.isnan(x_val):
                return
            ax.scatter(
                [x_val],
                [cy_strip],
                s=diamond_size,
                marker="D",
                facecolors=(0, 0, 0, 0),  # transparent fill
                edgecolors="black",
                linewidths=diamond_lw,
                zorder=7,
                transform=ax.transData,
            )

        draw_diamond(centre_old, cy_past, C_PAST)
        draw_diamond(centre_new, cy_pres, C_PRES)

        # ── defer annotations ─────────────────────────────────────────────────
        if centre_old is not None and not np.isnan(centre_old):
            _annots.append(
                dict(
                    ax=ax,
                    x_data=centre_old,
                    xmin=xmin,
                    xmax=xmax,
                    y_axes=cy_past + 0.18,
                    label=f"{centre_old:.2f}",
                    ha="right",
                    va="bottom",
                    color=C_ANNOT_B,
                )
            )
        if centre_new is not None and not np.isnan(centre_new):
            _annots.append(
                dict(
                    ax=ax,
                    x_data=centre_new,
                    xmin=xmin,
                    xmax=xmax,
                    y_axes=cy_pres - 0.18,
                    label=f"{centre_new:.2f}",
                    ha="left",
                    va="top",
                    color=C_ANNOT_R,
                )
            )

    # ── populate all panels ───────────────────────────────────────────────────
    _annots = []
    for ci, (stats, title, xlabel, xmin, xmax, mti, h_jit) in enumerate(columns):
        for ri, reg in enumerate(region_list):
            draw_row(
                axes[ci][ri],
                stats[reg],
                xmin,
                xmax,
                ri,
                reg,
                major_tick_interval=mti,
                h_jitter_frac=h_jit,
            )

    # ── annotations on figure canvas (top layer) ──────────────────────────────
    fig.canvas.draw()
    for ann in _annots:
        ax = ann["ax"]
        pos = ax.get_position()
        x_norm = (ann["x_data"] - ann["xmin"]) / (ann["xmax"] - ann["xmin"])
        fig_x = pos.x0 + x_norm * pos.width
        fig_y = pos.y0 + ann["y_axes"] * pos.height
        fig.text(
            fig_x,
            fig_y,
            ann["label"],
            ha=ann["ha"],
            va=ann["va"],
            fontsize=annot_fs,
            color=ann["color"],
            zorder=999,
            clip_on=False,
        )

    # ── region labels ─────────────────────────────────────────────────────────
    for ri, reg in enumerate(region_list):
        pos = axes[0][ri].get_position()
        cy_fig = (pos.y0 + pos.y1) / 2
        fig.text(
            left_frac - 4 / figwidth_mm,
            cy_fig + 0.008,
            reg,
            ha="right",
            va="center",
            fontsize=region_abbrev_fs,
            fontweight="bold",
            color=C_TEXT,
        )
        fig.text(
            left_frac - 4 / figwidth_mm,
            cy_fig - 0.012,
            REGION_NAMES.get(reg, reg),
            ha="right",
            va="center",
            fontsize=region_name_fs,
            style="italic",
            color=C_MUTED,
        )

    # ── column headers + panel label ──────────────────────────────────────────
    header_y = top_frac + 0.02
    fig.text(
        left_frac - 18 / figwidth_mm,
        header_y,
        panel_label,
        ha="left",
        va="bottom",
        fontsize=col_header_fs,
        fontweight="bold",
        color=C_TEXT,
    )
    for ci, (stats, title, xlabel, xmin, xmax, mti, h_jit) in enumerate(columns):
        pos = axes[ci][0].get_position()
        cx = (pos.x0 + pos.x1) / 2
        fig.text(
            cx,
            header_y,
            title,
            ha="center",
            va="bottom",
            fontsize=col_header_fs,
            fontweight="bold",
            color="black",
        )

    # ── vertical column separator lines ──────────────────────────────────────
    for ci in range(1, nC):
        x_sep = col_x0[ci] - mid_gap / 2
        line = plt.Line2D(
            [x_sep, x_sep],
            [bottom_frac, top_frac],
            transform=fig.transFigure,
            color=C_SEP,
            linewidth=0.4,
            zorder=0,
        )
        fig.add_artist(line)

    # ── legend (commented out — uncomment to re-enable) ──────────────────────
    # legend_handles = [
    #     mpatches.Patch(facecolor=C_PAST, alpha=0.8, label='1961–1990'),
    #     mpatches.Patch(facecolor=C_PRES, alpha=0.8, label='1991–2020'),
    # ]
    # fig.legend(
    #     handles=legend_handles,
    #     loc='lower left',
    #     bbox_to_anchor=(left_frac, 0.0),
    #     bbox_transform=fig.transFigure,
    #     ncol=2,
    #     fontsize=col_header_fs - 1,
    #     frameon=False,
    #     handlelength=1.0,
    #     handletextpad=0.4,
    #     columnspacing=1.0,
    # )

    # ── save / display ────────────────────────────────────────────────────────
    display(fig)

    if filename is not None:
        os.makedirs(save_dir, exist_ok=True)
        formats = save_formats if save_formats is not None else ["pdf"]
        for fmt in formats:
            fmt = fmt.lower().lstrip(".")
            filepath = os.path.join(save_dir, f"{filename}.{fmt}")
            kw = dict(bbox_inches="tight", facecolor="white")
            if fmt in ("jpg", "jpeg", "png", "tiff"):
                kw["dpi"] = dpi
            fig.savefig(filepath, format=fmt, **kw)
            print(f"✓  Saved: {filepath}")

    plt.close(fig)
    return fig

# Fig S1

In [ ]:
def load_gridcell_data(proto_lon, proto_lat, load_dir=str(FIGDATA_DIR) + os.sep):
    """
    Load a previously saved grid-cell bundle and unpack it into the same
    variable names used at the call site (as a dict — unpack with **, or
    index directly).

    Returns
    -------
    dict with keys: proto_lon, proto_lat, dis_series, tx_series,
    q10_series, tx90_series, compound_df
    """
    fname = "FigS1_data.pkl"
    fpath = os.path.join(load_dir, fname)

    with open(fpath, "rb") as f:
        bundle = pickle.load(f)

    print(f"Loaded: {fpath}")
    return bundle


data_gridcell_HERA = load_gridcell_data(proto_lon=9.55, proto_lat=47.15)


def plot_compound_events(
    compound_df,
    dis_series,
    tx_series,
    q10_series,
    tx90_series,
    proto_lon=10.95,
    proto_lat=60.15,
    plot_years=None,
    shoulder_alpha=0.15,
):
    """
    Plot discharge and temperature on a single twin-axis panel for each
    requested year, showing how compound low-flow + heatwave events emerge
    from the co-occurrence of both hazards.

    Layout
    ------
    - Left y-axis  : discharge [m³/s] and Q10 threshold (blue tones, dashed
      threshold vs. solid observed — mirrors the temperature axis styling)
    - Right y-axis : temperature [°C] and TX90 threshold (red tones, dashed
      threshold vs. solid observed)
    - Shoulder months (Jan–May, Oct–Dec) are drawn at ``shoulder_alpha``
      to focus attention on the June–September hazard window.
    - Each compound event window is shaded purple with a duration label.
      Labels are placed in a strip above the plot and bumped to a higher
      tier only when they'd otherwise overlap a neighboring label, so an
      arbitrary number of events at arbitrary spacing never collide.
      A thin leader line connects each label to its window.
    - Legend is grouped by axis (discharge-axis items, then
      temperature-axis items, then the compound event window) and laid
      out in two columns so it reads top-to-bottom per axis rather than
      interleaving red/blue.

    Parameters
    ----------
    compound_df : pd.DataFrame
        Must contain: ``compound_year``, ``compound_start_date``, ``compound_duration``.
    dis_series : pd.Series
        Daily discharge [m³/s] with a DatetimeIndex.
    tx_series : pd.Series
        Daily maximum temperature [°C] with a DatetimeIndex.
    q10_series : pd.Series
        Q10 threshold indexed by day-of-year (1–366).
    tx90_series : pd.Series
        TX90 threshold indexed by day-of-year (1–366).
    proto_lon, proto_lat : float, optional
        Grid cell coordinates, shown in the title. Also the values you
        should use to slice the source datasets before calling this
        function, so the title always matches the data.
    plot_years : list[int], optional
        Years to plot. Each produces one figure. Defaults to [1961, 2003, 2018].
    shoulder_alpha : float, optional
        Alpha for lines and shading outside Jun–Sep. Defaults to 0.15.
    """
    # ══ GLOBAL TYPOGRAPHY & STYLE (set once) ═══════════════════════════════
    X = 10  # Global Typography Sizes
    mpl.rcParams["axes.titlesize"] = X  # Title
    mpl.rcParams["axes.labelsize"] = X  # X & Y Axis Titles
    mpl.rcParams["xtick.labelsize"] = X - 1  # X Tick Labels
    mpl.rcParams["ytick.labelsize"] = X - 1  # Y Tick Labels
    mpl.rcParams["legend.fontsize"] = X - 1  # Legend text
    mpl.rcParams["font.size"] = X - 1  # Default text (Annotations)

    # Global Colors & Spines
    mpl.rcParams["text.color"] = "#222222"
    mpl.rcParams["axes.labelcolor"] = "#222222"
    mpl.rcParams["xtick.labelcolor"] = "#222222"
    mpl.rcParams["ytick.labelcolor"] = "#222222"

    # Tick & Spine Color rules
    mpl.rcParams["axes.edgecolor"] = "#D9DAD9"
    mpl.rcParams["xtick.color"] = "#D9DAD9"
    mpl.rcParams["ytick.color"] = "#D9DAD9"
    mpl.rcParams["legend.frameon"] = False

    if plot_years is None:
        plot_years = [1961, 2003, 2018]

    summer_months = {6, 7, 8, 9}

    def _season_values(values, season_mask):
        """Return values with NaN outside the season so lines break cleanly."""
        out = values.astype(float).copy()
        out[~season_mask] = np.nan
        return out

    # Coordinate string — built once, reused across all years
    coord_str = f"lon: {proto_lon:.3f}°   lat: {proto_lat:.3f}°"

    for yr in plot_years:

        # ── Slice to this calendar year ───────────────────────────────────
        dis_yr = dis_series[dis_series.index.year == yr]
        tx_yr = tx_series[tx_series.index.year == yr]

        if dis_yr.empty or tx_yr.empty:
            print(f"[Skip] No data for {yr}")
            continue

        # ── Thresholds aligned to DOY ──────────────────────────────────────
        q10_yr = q10_series.loc[dis_yr.index.dayofyear].values
        tx90_yr = tx90_series.loc[tx_yr.index.dayofyear].values

        # ── Season masks ───────────────────────────────────────────────────
        dis_is_summer = np.isin(dis_yr.index.month, list(summer_months))
        tx_is_summer = np.isin(tx_yr.index.month, list(summer_months))

        # ── Figure + twin axes ─────────────────────────────────────────────
        fig, ax_dis = plt.subplots(figsize=(6.8, 3))
        ax_tx = ax_dis.twinx()  # right y-axis shares the same x-axis

        # ══ DISCHARGE (left axis) ══════════════════════════════════════════

        # Q10 threshold — dashed, mirroring TX90's "threshold" styling so
        # dashed = threshold and solid = observed on BOTH axes
        (line_q10,) = ax_dis.plot(
            dis_yr.index,
            _season_values(q10_yr, dis_is_summer),
            color="#1f77b4",
            lw=1.6,
            alpha=0.85,
            ls="--",
            label="Q10 threshold",
        )
        ax_dis.plot(
            dis_yr.index,
            _season_values(q10_yr, ~dis_is_summer),
            color="#1f77b4",
            lw=1.6,
            alpha=shoulder_alpha,
            ls="--",
        )

        # Observed discharge
        (line_dis,) = ax_dis.plot(
            dis_yr.index,
            _season_values(dis_yr.values, dis_is_summer),
            color="#08306b",
            lw=1.2,
            zorder=5,
            label="Daily discharge",
        )
        ax_dis.plot(
            dis_yr.index,
            _season_values(dis_yr.values, ~dis_is_summer),
            color="#08306b",
            lw=1.2,
            zorder=5,
            alpha=shoulder_alpha,
        )

        # Flow deficit shading
        deficit_cond = dis_yr.values < q10_yr
        patch_deficit = ax_dis.fill_between(
            dis_yr.index,
            dis_yr.values,
            q10_yr,
            where=deficit_cond & dis_is_summer,
            color="#DE8F05",
            alpha=0.45,
            label="Flow deficit",
            interpolate=True,
            zorder=4,
        )
        ax_dis.fill_between(
            dis_yr.index,
            dis_yr.values,
            q10_yr,
            where=deficit_cond & ~dis_is_summer,
            color="#DE8F05",
            alpha=shoulder_alpha,
            interpolate=True,
            zorder=4,
        )

        # ══ TEMPERATURE (right axis) ═══════════════════════════════════════

        # TX90 threshold
        (line_tx90,) = ax_tx.plot(
            tx_yr.index,
            _season_values(tx90_yr, tx_is_summer),
            color="#d62728",
            lw=1.6,
            alpha=0.85,
            ls="--",
            label="TX90 threshold",
        )
        ax_tx.plot(
            tx_yr.index,
            _season_values(tx90_yr, ~tx_is_summer),
            color="#d62728",
            lw=1.6,
            alpha=shoulder_alpha,
            ls="--",
        )

        # Observed temperature
        (line_tx,) = ax_tx.plot(
            tx_yr.index,
            _season_values(tx_yr.values, tx_is_summer),
            color="#67000d",
            lw=1.2,
            zorder=5,
            label="Daily TX",
        )
        ax_tx.plot(
            tx_yr.index,
            _season_values(tx_yr.values, ~tx_is_summer),
            color="#67000d",
            lw=1.2,
            zorder=5,
            alpha=shoulder_alpha,
        )

        # Temperature excess shading
        excess_cond = tx_yr.values > tx90_yr
        patch_excess = ax_tx.fill_between(
            tx_yr.index,
            tx_yr.values,
            tx90_yr,
            where=excess_cond & tx_is_summer,
            color="#fc9272",
            alpha=0.40,
            label="Temp excess",
            interpolate=True,
            zorder=4,
        )
        ax_tx.fill_between(
            tx_yr.index,
            tx_yr.values,
            tx90_yr,
            where=excess_cond & ~tx_is_summer,
            color="#fc9272",
            alpha=shoulder_alpha,
            interpolate=True,
            zorder=4,
        )

        # ══ COMPOUND EVENT WINDOWS + DURATION LABELS ═══════════════════════
        yr_events = compound_df[compound_df["compound_year"] == yr]
        yr_events = yr_events.sort_values("compound_start_date").reset_index(drop=True)
        n_events = len(yr_events)

        # Duration labels live in a dedicated strip ABOVE the plotted data
        # (axes-fraction y-coordinates, independent of the discharge
        # y-range) so they never compete for space with the legend or with
        # tall discharge/temperature peaks. Each label is greedily assigned
        # to the lowest tier where it won't overlap the label already
        # placed in that tier, so this stays robust to any number of
        # events at any spacing (not just alternating pairs) — a thin
        # leader line then drops from the label down to the window it
        # describes so the pairing stays legible however far it's bumped.
        trans = ax_dis.get_xaxis_transform()  # x in data coords, y in axes fraction
        tier_y = [1.06, 1.20, 1.34, 1.48]  # extra tiers used only if needed
        tier_step = tier_y[1] - tier_y[0]

        # Approximate half-width (in days) of a "NN days" label at the
        # current font size, used to decide whether two labels' text
        # would visually collide if placed on the same tier.
        chars_per_label = len("nn days")
        approx_char_days = 5.5  # empirical: label width in "days" units per character
        half_width_days = 0.5 * chars_per_label * approx_char_days

        events = []
        for _, cev in yr_events.iterrows():
            cs = pd.Timestamp(cev["compound_start_date"])
            dur = int(cev["compound_duration"])
            ce = cs + pd.Timedelta(days=dur - 1)
            mid = cs + pd.Timedelta(days=dur / 2)
            events.append({"cs": cs, "ce": ce, "dur": dur, "mid": mid})

        last_mid_per_tier = (
            []
        )  # last label center (in days-since-epoch) placed on each tier

        for ev in events:
            cs, ce, dur, mid = ev["cs"], ev["ce"], ev["dur"], ev["mid"]
            mid_num = mdates.date2num(mid)

            # Shaded window
            ax_dis.axvspan(cs, ce, color="purple", alpha=0.12, zorder=2)

            # Dashed edge markers
            for edge in (cs, ce):
                ax_dis.axvline(
                    edge, color="purple", lw=0.9, ls="--", alpha=0.5, zorder=3
                )

            # Find the lowest tier whose last label is far enough away
            # (in x) that this label's text won't overlap it. Add a new
            # tier on demand if every existing tier is still too close.
            tier_idx = None
            for t, last_mid in enumerate(last_mid_per_tier):
                if abs(mid_num - last_mid) >= 2 * half_width_days:
                    tier_idx = t
                    break
            if tier_idx is None:
                tier_idx = len(last_mid_per_tier)
                last_mid_per_tier.append(mid_num)
                if tier_idx >= len(tier_y):
                    tier_y.append(tier_y[-1] + tier_step)
            else:
                last_mid_per_tier[tier_idx] = mid_num

            label_y = tier_y[tier_idx]

            ax_dis.annotate(
                f"{dur} days",
                xy=(mid_num, 1.0),
                xycoords=trans,
                xytext=(mid_num, label_y),
                textcoords=trans,
                ha="center",
                va="bottom",
                fontweight="bold",
                color="purple",
                annotation_clip=False,
                arrowprops=dict(
                    arrowstyle="-",
                    color="purple",
                    alpha=0.5,
                    lw=0.9,
                    shrinkA=0,
                    shrinkB=2,
                ),
            )

        n_tiers_used = max(len(last_mid_per_tier), 1)

        # ══ AXIS FORMATTING ════════════════════════════════════════════════
        month_starts = [pd.Timestamp(f"{yr}-{m:02d}-01") for m in range(1, 13)]
        month_labels = [
            "Jan",
            "Feb",
            "Mar",
            "Apr",
            "May",
            "Jun",
            "Jul",
            "Aug",
            "Sep",
            "Oct",
            "Nov",
            "Dec",
        ]

        ax_dis.set_xticks(month_starts)
        ax_dis.set_xticklabels(month_labels)
        ax_dis.set_xlim(
            pd.Timestamp(f"{yr}-01-01"),
            pd.Timestamp(f"{yr}-12-31"),
        )

        ax_dis.set_ylabel("Discharge [m³/s]", labelpad=10, color="#08306b")
        ax_tx.set_ylabel("Temperature [°C]", labelpad=10, color="#67000d")

        # Remove all spines except the two y-axis spines
        ax_dis.spines["top"].set_visible(False)
        ax_tx.spines["top"].set_visible(False)

        # No grid
        ax_dis.grid(False)
        ax_tx.grid(False)

        # ══ COMBINED LEGEND (temp-over-discharge, 4 columns) ═══════════════
        # matplotlib fills a multi-column legend column-major (top-to-bottom
        # within a column before moving to the next), so pairing each column
        # as (temperature item, discharge item) — rather than grouping all
        # discharge items in one block and all temperature items in another —
        # gives a temp-over-discharge reading order within every column.

        compound_patch = mpatches.Patch(
            color="purple", alpha=0.4, label="Compound event window"
        )
        blank_patch = mpatches.Patch(alpha=0, label=" ")

        col1 = [line_tx, line_dis]  # Daily TX,        Daily discharge
        col2 = [line_tx90, line_q10]  # TX90 threshold,  Q10 threshold
        col3 = [patch_excess, patch_deficit]  # Temp excess,     Flow deficit
        col4 = [compound_patch, blank_patch]  # Compound window, (blank)

        handles = col1 + col2 + col3 + col4

        ax_dis.legend(
            handles=handles,
            framealpha=0.9,
            loc="upper left",
            ncols=4,
            bbox_to_anchor=(0.01, -0.2),
            handlelength=1.4,
            handletextpad=0.5,
            columnspacing=0.9,
            labelspacing=0.35,
            borderaxespad=0.3,
        )

        # ══ TITLE ══════════════════════════════════════════════════════════
        n_label = f"{n_events} compound event{'s' if n_events != 1 else ''}"
        title_lines = [
            f"Grid cell   —   {yr}   —   {n_label}",
            coord_str,
        ]

        fig.suptitle(
            "\n".join(title_lines),
            fontweight="bold",
            x=0.09,
            y=0.99,
            ha="left",
            va="top",
            fontsize=mpl.rcParams["axes.titlesize"],
        )

        # Reserve room at the top of the figure for the title (suptitle)
        # and, below it, the duration label strip (drawn in axes-fraction
        # space above y=1.0). The reserved space scales with however many
        # label tiers actually got used this year, so a dense year with
        # many close-together events doesn't clip its top tier while a
        # sparse year doesn't waste vertical space.
        top_margin = 0.68 - 0.05 * (n_tiers_used - 1)
        fig.subplots_adjust(top=max(top_margin, 0.42))

        # plt.tight_layout()
        # ── Save figure ───────────────────────────────────────────────────
        SAVE_DIR = str(FIGURE_DIR) + os.sep
        os.makedirs(SAVE_DIR, exist_ok=True)
        save_path = os.path.join(
            SAVE_DIR, f"FigS1_Compound_Visual_lon{proto_lon}_lat{proto_lat}_{yr}.pdf"
        )
        # fig.savefig(save_path, format="pdf", bbox_inches="tight", dpi=300)
        # print(f"Saved: {save_path}")

        plt.show()

In [ ]:
plot_compound_events(
    compound_df=data_gridcell_HERA["compound_df"],
    dis_series=data_gridcell_HERA["dis_series"],
    tx_series=data_gridcell_HERA["tx_series"],
    q10_series=data_gridcell_HERA["q10_series"],
    tx90_series=data_gridcell_HERA["tx90_series"],
    proto_lon=data_gridcell_HERA["proto_lon"],
    proto_lat=data_gridcell_HERA["proto_lat"],
    plot_years=[2003],
    shoulder_alpha=0.15,
)

# Fig S2

In [ ]:
load_dir = str(FIGDATA_DIR) + os.sep

CE_Q10_TX90_1991to2020_Intensity = xr.open_dataset(
    load_dir + "FigS2_1991to2020.nc"
)
CE_Q10_TX90_1961to1990_Intensity = xr.open_dataset(
    load_dir + "FigS2_1961to1990.nc"
)

## abc

In [ ]:
# Create map features adder
add_map_features = create_map_features_adder()

# Example: Save the figure
fig = plot_three_panel_analysis(
    data_1961to1990=CE_Q10_TX90_1961to1990_Intensity,
    data_1991to2020=CE_Q10_TX90_1991to2020_Intensity,
    CE_MISSING_FLAG=-99999,
    variable_name="mean_annual_flow_intensity_scaled",
    cmap_sequential=viridis_mod,
    extend_colorbar="both",
    bins_map=[2, 4, 5.5, 7.5, 11],  # np.arange(2, 12, 2),
    bins_delta=np.array([-3, -2, -1, -0.025, 0.025, 1, 2, 3]),
    yellow_range=(-0.025, 0.025),
    kde_bounded=True,  # NEW: Use bounded KDE (reflection method)
    kde_lower_bound=0,  # NEW: Lower bound for bounded KDE
    kde_ylim=(0, 0.31),
    kde_yticks=(0, 0.1, 0.2),
    kde_xlim=(-0, 25),
    kde_xticks=(0, 5, 10, 15, 20),
    tick_positions_delta=[-3, -2, -1, 0, 1, 2, 3],
    # filename='FigS2_abc',  # Save with this name
    save_formats=["pdf"],
    dpi=300,
    titles={
        "panel1": "Flow Deficit (%), 1991-2020",
        "panel2": "Relative \u0394",  # "Relative \u0394 (91-20 vs 61-90)"
        "panel3": "Pan-Europe Distribution",
    },
    tick_interval=1,
    add_map_features_func=add_map_features,
)
plt.show()

## def

In [ ]:
# Create map features adder
add_map_features = create_map_features_adder()

# Example: Save the figure
fig = plot_three_panel_analysis(
    data_1961to1990=CE_Q10_TX90_1961to1990_Intensity,
    data_1991to2020=CE_Q10_TX90_1991to2020_Intensity,
    CE_MISSING_FLAG=-99999,
    variable_name="mean_annual_temperature_intensity_scaled",
    cmap_sequential=viridis_mod,
    extend_colorbar="both",
    bins_map=[2.2, 3.0, 3.8, 4.6, 5.4],  # np.arange(1, 6, 1),
    bins_delta=np.array([-3, -2, -1, -0.025, 0.025, 1, 2, 3]),
    yellow_range=(-0.025, 0.025),
    kde_ylim=(0, 0.45),
    kde_yticks=(0, 0.2, 0.4),
    kde_xlim=(0, 11),
    kde_xticks=(0, 5, 10),
    tick_positions_delta=[-3, -2, -1, 0, 1, 2, 3],
    panel_labels=["d", "e", "f"],
    filename="FigS2_def",  # Save with this name
    save_formats=["pdf"],
    dpi=300,
    titles={
        "panel1": "Temp. Excess(%) 1991\u20132020",
        "panel2": "Relative \u0394",  # "Relative \u0394 (91-20 vs 61-90)"
        "panel3": "Pan-Europe Distribution",
    },
    tick_interval=1,
    title_fontsize=12,
    cbar_tick_fontsize=12,
    add_map_features_func=add_map_features,
)
plt.show()

# Fig S3

In [ ]:
def plot_prudence_regions(
    europe_IPCC_regions=None,
    add_features_fn=None,
    map_extent=[-10, 31, 35.6, 71],
    title="PRUDENCE Regions",
    title_fontsize=12,
    fig_width=4.5,
    fig_height=5.5,
    filename=None,
    save_dir=str(FIGURE_DIR) + os.sep,
    save_formats=["jpg"],
    dpi=150,
):
    """
    Plot a standalone PRUDENCE (or IPCC AR6) region map.

    Parameters
    ----------
    europe_IPCC_regions : geopandas.GeoDataFrame, optional
        GeoDataFrame containing region polygons with an 'abbrev' column
        used for labelling. If None, an empty styled basemap is rendered.
    add_features_fn : callable, optional
        Function with signature ``add_features_fn(ax)`` that decorates a
        GeoAxes with coastlines, borders, etc.  When None, the module-level
        ``create_map_features_adder`` factory is used to build a default
        Europe-scoped adder.
    map_extent : list of float
        Geographic bounding box as [lon_min, lon_max, lat_min, lat_max].
        Default covers continental Europe.
    title : str
        Title string rendered above the axes.
    title_fontsize : int
        Font size (pt) applied to the panel title.
    fig_width : float
        Figure width in inches.
    fig_height : float
        Figure height in inches.
    filename : str, optional
        Base filename (without extension) for saving.  When None the
        figure is not saved to disk.
    save_dir : str
        Absolute path to the output directory.  Created automatically if
        it does not exist.
    save_formats : list of str
        Container of format strings, e.g. ``['pdf', 'jpg']``.
    dpi : int
        Resolution used for raster output formats (jpg, png, tiff).

    Returns
    -------
    fig : matplotlib.figure.Figure
        The figure object.
    ax : cartopy.mpl.geoaxes.GeoAxes
        The single map axes.

    Examples
    --------
    >>> fig, ax = plot_prudence_regions(
    ...     europe_IPCC_regions=my_geodataframe,
    ...     filename='prudence_map',
    ...     save_formats=['pdf', 'jpg'],
    ... )
    >>> plt.show()
    """
    # ------------------------------------------------------------------
    # Projection — Lambert Conformal centred on Europe
    # ------------------------------------------------------------------
    projection = ccrs.LambertConformal(
        central_latitude=53,
        central_longitude=12,
        standard_parallels=(45, 65),
    )

    fig, ax = plt.subplots(
        figsize=(fig_width, fig_height),
        subplot_kw={"projection": projection},
    )

    # ------------------------------------------------------------------
    # Basemap features
    # Prefer the caller-supplied callable; fall back to the factory.
    # ------------------------------------------------------------------
    if add_features_fn is not None:
        add_features_fn(ax)
    else:
        # Factory builds a feature-adder pre-scoped to the target extent
        # and styled to match the other panels in the original figure.
        add_features_c = create_map_features_adder(
            extent=map_extent,
            land_color="#f5f5f5",
            coastline_color="#666666",
            coastline_linewidth=0.8,
        )
        add_features_c(ax)

    # ------------------------------------------------------------------
    # PRUDENCE / IPCC region polygons
    # ------------------------------------------------------------------
    if europe_IPCC_regions is not None:
        europe_IPCC_regions.plot(
            ax=ax,
            add_label=True,
            label="abbrev",
            line_kws={"linewidth": 1, "color": "#333333"},
            text_kws={
                "fontsize": 10,
                "weight": "bold",
                "color": "#000000",
            },
        )

    # ------------------------------------------------------------------
    # Cosmetics
    # ------------------------------------------------------------------
    # Remove the default geo-spine so the map has a clean, frameless look
    ax.spines["geo"].set_visible(False)
    set_axis_linewidth(ax, linewidth=0.0)

    ax.set_title(title, fontsize=title_fontsize, pad=4, fontweight="bold")

    # ------------------------------------------------------------------
    # Save to disk
    # ------------------------------------------------------------------
    if filename is not None:
        os.makedirs(save_dir, exist_ok=True)
        raster_formats = {"jpg", "jpeg", "png", "tiff"}
        for fmt in save_formats:
            filepath = os.path.join(save_dir, f"{filename}.{fmt}")
            fig.savefig(
                filepath,
                dpi=dpi if fmt.lower() in raster_formats else None,
                format=fmt,
                bbox_inches="tight",
            )
            print(f"✓ Saved: {filepath}")

    return fig, ax

In [ ]:
fig_prudence, ax_prudence = plot_prudence_regions(
    europe_IPCC_regions=europe_PRUDENCE_regions,
    map_extent=[-10, 31, 35.6, 71],
    title="PRUDENCE Regions",
    title_fontsize=12,
    fig_width=4,
    fig_height=5.5,
    # filename='FigS3',
    save_formats=["pdf"],
    dpi=150,
)

plt.show()

# Fig S4

In [ ]:
load_dir = str(FIGDATA_DIR) + os.sep

EARLS_CE_Q10_TX90_1961to2020_FD_Intensity_ungauged = pd.read_csv(
    load_dir + "FigS4_data.csv"
)
ungauged_shp = gpd.read_file(
    str(NOTEBOOK_DIR / "Data" / "EARLS_Shapefile" / "FigS4_basins.shp")
)

In [ ]:
# 1. Confirm the column names actually exist in your DataFrame
print(EARLS_CE_Q10_TX90_1961to2020_FD_Intensity_ungauged.columns.tolist())

# 2. Confirm the columns contain what you expect before passing them in
print(
    EARLS_CE_Q10_TX90_1961to2020_FD_Intensity_ungauged[
        ["FlowDeficit_scaled_1961to1990", "FlowDeficit_scaled_1991to2020"]
    ].describe()
)

In [ ]:
"""
plot_compound_metrics — updated Panel 3
========================================
Drop-in replacement for the original function.

Changes vs. the original
-------------------------
* Five new keyword parameters added to the signature (see "strip / IQR box"
  section below).
* Panel 3 re-implemented to match plot_three_panel_analysis:
    - Strip plot rendered BELOW the x-axis spine using a blended transform
      (x = data coords, y = axes coords), so dots never overlap the KDE area.
    - Thin-edge IQR box (no fill, no whiskers, no median line) wraps the
      jitter cloud snugly.
    - Mean diamond (transparent fill, black edge) sits inside each IQR box.
    - Mean annotation: 1961-1990 label appears ABOVE the box, 1991-2020
      label appears BELOW the box, preventing overlap.
    - Spines cleaned up: top/right hidden, left/bottom offset outward.
* The lollipop_style / lollipop_size parameters are retained in the
  signature for backwards-compatibility but are no longer used.  They can
  be safely removed in a future clean-up.
"""

# ─────────────────────────────────────────────────────────────────────────────
# Metric metadata — label, unit, default bins, KDE limits
# ─────────────────────────────────────────────────────────────────────────────
METRIC_META = {
    "AnnualFreq": {
        "label": "Mean annual frequency",
        "unit": "events year⁻¹",
        "bins_map": np.arange(0, 0.55, 0.05),
        "kde_xlim": (0, 1.0),
        "kde_ylim": (0, 8),
    },
    "AnnualDuration": {
        "label": "Mean annual duration",
        "unit": "days year⁻¹",
        "bins_map": np.arange(0, 5.5, 0.5),
        "kde_xlim": (0, 10),
        "kde_ylim": (0, 1),
    },
    "TempExcess_scaled": {
        "label": "Temp excess scaled",
        "unit": "% of JJAS TX90",
        "bins_map": np.arange(0, 22, 2),
        "kde_xlim": (0, 30),
        "kde_ylim": (0, 0.3),
    },
    "FlowDeficit_scaled": {
        "label": "Flow deficit scaled",
        "unit": "% of JJAS Q10",
        "bins_map": np.arange(0, 22, 2),
        "kde_xlim": (0, 30),
        "kde_ylim": (0, 0.3),
    },
    "CE_LFEdy": {
        "label": "CE/LFEdy — River heat stress",
        "unit": "CE days / LF days",
        "bins_map": np.arange(0, 0.55, 0.05),
        "kde_xlim": (0, 1.0),
        "kde_ylim": (0, 8),
    },
    "CE_HWEdy": {
        "label": "CE/HWEdy — Water vulnerability",
        "unit": "CE days / HW days",
        "bins_map": np.arange(0, 0.55, 0.05),
        "kde_xlim": (0, 1.0),
        "kde_ylim": (0, 8),
    },
}

DEFAULT_DELTA_BINS = np.array([-3.0, -2.0, -1.0, -0.5, -0.1, 0.1, 0.5, 1.0, 2.0, 3.0])
DEFAULT_DELTA_TICKS = [-3, -2, -1, 0, 1, 2, 3]


# ─────────────────────────────────────────────────────────────────────────────
# Convenience loader
# ─────────────────────────────────────────────────────────────────────────────
def prepare_europe_outline(
    europe_shp_path=str(FIGDATA_DIR / "European_Shapefile.shp"),
):
    """Load and dissolve the European shapefile into a single outline polygon."""
    print("Loading European shapefile...")
    shp = gpd.read_file(europe_shp_path)
    merged = shp.union_all()
    outline = gpd.GeoDataFrame(geometry=[merged], crs=shp.crs).to_crs("EPSG:4326")
    print(f"  Done. CRS: {outline.crs}")
    return outline


# ─────────────────────────────────────────────────────────────────────────────
# Main function
# ─────────────────────────────────────────────────────────────────────────────
def plot_compound_metrics(
    results_df,
    basin_shp,
    metric,
    # ── no event basins ──────────────────────────────────────────────────────
    no_ce_df=None,
    # ── boundary ─────────────────────────────────────────────────────────────
    europe_outline=None,
    europe_shp_path=str(FIGDATA_DIR / "European_Shapefile.shp"),
    basin_type="Gauged",
    # ── map extent ───────────────────────────────────────────────────────────
    lon_min=-12,
    lon_max=35,
    lat_min=35.6,
    lat_max=71,
    # ── sequential colormap (Panel 1) ────────────────────────────────────────
    bins_map=None,
    cmap_sequential="YlOrRd",
    cmap_sequential_range=(0.1, 0.95),
    # ── diverging colormap (Panel 2) ─────────────────────────────────────────
    bins_delta=None,
    delta_ticks=None,
    yellow_range=(-0.1, 0.1),
    extend_colorbar="neither",
    # ── KDE (Panel 3) ────────────────────────────────────────────────────────
    kde_xlim=None,
    kde_ylim=None,
    kde_xticks=None,
    kde_yticks=None,
    # ── strip plot + IQR box (Panel 3) ───────────────────────────────────────
    strip_jitter=0.012,
    strip_jitter_x=0.0,
    strip_alpha=0.10,
    strip_size=2,
    mean_fontsize=10,
    # ── kept for backwards compatibility — no longer used ────────────────────
    lollipop_style=True,
    lollipop_size=40,
    # ── basin boundaries ─────────────────────────────────────────────────────
    show_basin_boundaries=True,
    basin_boundary_color="black",
    basin_boundary_lw=0.01,
    # ── panel titles ─────────────────────────────────────────────────────────
    titles=None,
    panel_labels=("a", "b", "c"),
    # ── figure / font ────────────────────────────────────────────────────────
    figsize_mm=(200, 120),
    title_fontsize=12,
    cbar_tick_fontsize=12,
    kde_position_adjust=(0.05, -0.072, 0.00, -0.012),
    # ── save ─────────────────────────────────────────────────────────────────
    save_dir=str(FIGURE_DIR) + os.sep,
    filename=None,
    save_formats=["jpg", "pdf"],
    dpi=300,
):
    """
    Three-panel compound metric plot.

    Parameters
    ----------
    results_df : pd.DataFrame
        Must contain: idx, {metric}_1961to1990, {metric}_1991to2020.
    basin_shp : GeoDataFrame
        gauged_shp or ungauged_shp.
    metric : str
        One of: AnnualFreq | AnnualDuration | TempExcess_scaled |
                FlowDeficit_scaled | CE_LFEdy | CE_HWEdy
    no_ce_df : pd.DataFrame or None
        DataFrame of basins with zero compound events. Must contain an 'idx'
        column (or the first column is used). These basins are plotted in
        dark grey (#4D4D4D) on both maps.
    europe_outline : GeoDataFrame or None
        Pre-loaded single-polygon European boundary.
    basin_type : str
        "Gauged" or "Ungauged".
    lon_min / lon_max / lat_min / lat_max : float
        Map extent.
    bins_map : array-like or None
        Colormap bin edges for Panel 1. Defaults from METRIC_META.
    cmap_sequential : str or ListedColormap
        Sequential colormap for Panel 1. See original docstring for full
        description of the string vs. ListedColormap paths.
    cmap_sequential_range : tuple
        (lo, hi) fractions of cmap to sample when cmap_sequential is a string.
    bins_delta : array-like or None
        Bin edges for the diverging colormap in Panel 2.
    delta_ticks : list or None
        Tick positions on the Panel 2 colorbar.
    yellow_range : tuple
        (lo, hi) relative-change values shown in gold on Panel 2.
    extend_colorbar : str
        Controls Panel 1 colorbar extension for the string colormap path.
        "max" | "both" | "min" | "neither".
    kde_xlim / kde_ylim : tuple or None
        Axis limits for Panel 3.
    kde_xticks / kde_yticks : array-like or None
        Explicit tick positions for the x / y axis of Panel 3.
        When None (default) matplotlib chooses ticks automatically.
    strip_jitter : float
        Half-width of uniform vertical y-jitter for strip dots (axes coords).
        Keep small so the IQR box wraps tightly. Default 0.012.
    strip_jitter_x : float
        Half-width of uniform horizontal x-jitter for strip dots (data coords).
        0 = no horizontal jitter. Default 0.0.
    strip_alpha : float
        Alpha of strip plot dots. Default 0.10.
    strip_size : float
        Marker size of strip plot dots in points². Default 2.
    mean_fontsize : int
        Font size of the mean value annotation near the diamond. Default 10.
    lollipop_style / lollipop_size : kept for backwards compatibility only.
    show_basin_boundaries : bool
    basin_boundary_color / basin_boundary_lw : str / float
    titles : dict or None
        Override panel titles. Keys: 'panel1', 'panel2', 'panel3'.
    panel_labels : tuple or list of str
        Labels for the three panel annotations. Default ("a", "b", "c").
    figsize_mm : tuple
        Figure size in millimetres (width, height).
    title_fontsize / cbar_tick_fontsize : int
    kde_position_adjust : tuple
        (dx, dy, dw, dh) fine-tuning of KDE panel position.
    save_dir / filename / save_formats / dpi : save options.

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    # ── Panel titles ─────────────────────────────────────────────────────────
    if titles is None:
        titles = {
            "panel1": "Events/Yr, 1991\u20132020",
            "panel2": "Relative \u0394",
            "panel3": "Pan-Europe Distribution",
        }

    # ── Column names ─────────────────────────────────────────────────────────
    col_61 = f"{metric}_1961to1990"
    col_91 = f"{metric}_1991to2020"
    for col in [col_61, col_91]:
        if col not in results_df.columns:
            raise ValueError(
                f"Column '{col}' not found.\n" f"Available: {list(results_df.columns)}"
            )

    # ── Metadata defaults ─────────────────────────────────────────────────────
    meta = METRIC_META.get(metric, {})
    label = meta.get("label", metric)
    unit = meta.get("unit", "")
    bins_map = np.asarray(
        bins_map
        if bins_map is not None
        else meta.get("bins_map", np.arange(0, 1.1, 0.1))
    )
    bins_delta = np.asarray(
        bins_delta if bins_delta is not None else DEFAULT_DELTA_BINS
    )
    delta_ticks = delta_ticks if delta_ticks is not None else DEFAULT_DELTA_TICKS
    kde_xlim = kde_xlim if kde_xlim is not None else meta.get("kde_xlim")
    kde_ylim = kde_ylim if kde_ylim is not None else meta.get("kde_ylim", (0, 5))

    # ── Relative change ───────────────────────────────────────────────────────
    df = results_df.copy()
    df["rel_change"] = np.where(
        df[col_61] != 0,
        (df[col_91] - df[col_61]) / df[col_61],
        np.nan,
    )

    # ── European boundary ─────────────────────────────────────────────────────
    if europe_outline is None:
        print(
            "  Loading European shapefile "
            "(pass europe_outline= to avoid this on every call)..."
        )
        _shp = gpd.read_file(europe_shp_path)
        europe_outline = gpd.GeoDataFrame(
            geometry=[_shp.union_all()], crs=_shp.crs
        ).to_crs("EPSG:4326")

    # ── Merge results onto shapefile ──────────────────────────────────────────
    geo = basin_shp.copy()
    if "index" in geo.columns:
        geo = geo.rename(columns={"index": "idx"})
    if basin_type.lower() == "ungauged":
        geo["idx"] = geo["idx"].astype(str).str.replace(r"^hybas_", "", regex=True)
    geo = geo.to_crs("EPSG:4326")
    geo["idx"] = geo["idx"].astype(str)
    df["idx"] = df["idx"].astype(str)

    merged = geo.merge(df[["idx", col_61, col_91, "rel_change"]], on="idx", how="left")
    clipped = gpd.clip(merged, europe_outline)

    # ── Identify No-CE basins ─────────────────────────────────────────────────
    if no_ce_df is not None and not no_ce_df.empty:
        id_col = "idx" if "idx" in no_ce_df.columns else no_ce_df.columns[0]
        no_ce_ids = no_ce_df[id_col].astype(str).tolist()
        if basin_type.lower() == "ungauged":
            no_ce_ids = [s.replace("hybas_", "") for s in no_ce_ids]
        clipped["is_no_ce"] = clipped["idx"].isin(no_ce_ids)
    else:
        clipped["is_no_ce"] = False

    n_no_ce = clipped["is_no_ce"].sum()
    print(
        f"[{basin_type}] Matched: {merged[col_91].notna().sum()}/{len(merged)}  |  "
        f"Clipped to Europe: {len(clipped)}  |  No-CE basins marked: {n_no_ce}"
    )
    print(f"  {col_91}: {clipped[col_91].min():.4f} – {clipped[col_91].max():.4f}")
    print(
        f"  rel_change: {clipped['rel_change'].min():.3f} – "
        f"{clipped['rel_change'].max():.3f}"
    )

    # ── Sequential colormap (Panel 1) ─────────────────────────────────────────
    cbar1_extend = "neither"

    if isinstance(cmap_sequential, ListedColormap):
        colors_list = list(cmap_sequential.colors)
        n_needed = len(bins_map) + 1

        if len(colors_list) != n_needed:
            raise ValueError(
                f"ListedColormap must have exactly len(bins_map)+1 = {n_needed} "
                f"colors.\n"
                f"  colors[0]    → values below {bins_map[0]}\n"
                f"  colors[1:-1] → the {len(bins_map)-1} interior intervals\n"
                f"  colors[-1]   → values above {bins_map[-1]}\n"
                f"Got {len(colors_list)} colors."
            )

        bounds_seq = np.concatenate([[-1e9], bins_map, [1e9]])
        cmap_seq = ListedColormap(colors_list)
        norm_seq = BoundaryNorm(bounds_seq, ncolors=cmap_seq.N)
        cbar1_extend = "neither"

    elif isinstance(cmap_sequential, str):
        _cmap_base = plt.get_cmap(cmap_sequential)
        _extra = {"neither": 0, "min": 1, "max": 1, "both": 2}
        n_colors = (len(bins_map) - 1) + _extra.get(extend_colorbar, 1)

        colors_seq = _cmap_base(
            np.linspace(cmap_sequential_range[0], cmap_sequential_range[1], n_colors)
        )
        cmap_seq = ListedColormap(colors_seq)

        if extend_colorbar in ("min", "both"):
            cmap_seq.set_under("#4D4D4D")
        if extend_colorbar in ("max", "both"):
            cmap_seq.set_over(_cmap_base(cmap_sequential_range[1]))

        norm_seq = BoundaryNorm(bins_map, ncolors=cmap_seq.N, extend=extend_colorbar)
        cbar1_extend = extend_colorbar

    else:
        raise ValueError("cmap_sequential must be a string or ListedColormap.")

    # ── Diverging colormap (Panel 2) ──────────────────────────────────────────
    neg_bins = bins_delta[bins_delta < yellow_range[0]]
    pos_bins = bins_delta[bins_delta > yellow_range[1]]
    RdYlBu = plt.cm.RdYlBu
    colors_div = np.vstack(
        [
            RdYlBu(np.linspace(0.95, 1.0, 1)),
            RdYlBu(np.linspace(0.90, 0.65, len(neg_bins))),
            np.tile(to_rgba("#FFDF20"), (1, 1)),
            RdYlBu(np.linspace(0.35, 0.05, len(pos_bins))),
            RdYlBu(np.linspace(0.0, 0.05, 1)),
        ]
    )
    cmap_div = ListedColormap(colors_div)
    cmap_div.set_under("#4D4D4D")
    norm_div = BoundaryNorm(
        np.concatenate([[-1e9], bins_delta, [1e9]]),
        ncolors=cmap_div.N,
    )

    # ── Figure ────────────────────────────────────────────────────────────────
    MM2IN = 1 / 25.4
    fig = plt.figure(figsize=(figsize_mm[0] * MM2IN, figsize_mm[1] * MM2IN))
    gs = GridSpec(1, 3, figure=fig, wspace=0.12, width_ratios=[1, 1, 1])
    mpl.rcParams["xtick.direction"] = "in"
    mpl.rcParams["ytick.direction"] = "in"

    def _setup_map_ax(ax):
        """Light-grey land background, fixed extent, no ticks/labels/spines."""
        europe_outline.plot(ax=ax, facecolor="#f0f0f0", edgecolor="none", zorder=0)
        ax.set_xlim(lon_min, lon_max)
        ax.set_ylim(lat_min, lat_max)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlabel("")
        ax.set_ylabel("")
        for spine in ax.spines.values():
            spine.set_visible(False)

    poly_render_kwargs = dict(
        rasterized=True,
        zorder=1,
        antialiased=True,
        edgecolors="none",
        linewidth=0.025,
    )

    mask_no_ce = clipped["is_no_ce"]
    mask_p1_na = clipped[col_91].isna() & ~mask_no_ce
    mask_p1_ok = clipped[col_91].notna() & ~mask_no_ce
    mask_p2_na = clipped["rel_change"].isna() & ~mask_no_ce
    mask_p2_ok = clipped["rel_change"].notna() & ~mask_no_ce

    # ── Panel 1: 1991-2020 map ────────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, 0])
    _setup_map_ax(ax1)

    if mask_p1_na.any():
        clipped[mask_p1_na].plot(ax=ax1, color="lightgrey", **poly_render_kwargs)
    if mask_no_ce.any():
        clipped[mask_no_ce].plot(ax=ax1, color="#4D4D4D", **poly_render_kwargs)
    if mask_p1_ok.any():
        clipped[mask_p1_ok].plot(
            ax=ax1, column=col_91, cmap=cmap_seq, norm=norm_seq, **poly_render_kwargs
        )

    if show_basin_boundaries:
        clipped.boundary.plot(
            ax=ax1,
            edgecolor=basin_boundary_color,
            linewidth=basin_boundary_lw,
            zorder=2,
            rasterized=True,
        )

    europe_outline.boundary.plot(
        ax=ax1, edgecolor="black", linewidth=0.5, zorder=3, rasterized=True
    )

    sm1 = plt.cm.ScalarMappable(cmap=cmap_seq, norm=norm_seq)
    sm1.set_array([])
    cbar1 = plt.colorbar(
        sm1,
        ax=ax1,
        orientation="horizontal",
        pad=0.01,
        shrink=0.9,
        aspect=20,
        extend=cbar1_extend,
    )
    cbar1.set_ticks(bins_map[::])
    cbar1.set_ticklabels([f"{v:g}" for v in bins_map[::]])
    cbar1.ax.tick_params(labelsize=cbar_tick_fontsize)
    cbar1.set_label("")
    cbar1.outline.set_linewidth(0.8)

    ax1.set_title(titles["panel1"], fontsize=title_fontsize, fontweight="bold", pad=4)
    ax1.text(
        0.0,
        1.0,
        f"({panel_labels[0]})",
        transform=ax1.transAxes,
        fontsize=title_fontsize - 1,
        fontweight="bold",
        va="top",
    )

    # ── Panel 2: Relative change map ──────────────────────────────────────────
    ax2 = fig.add_subplot(gs[0, 1])
    _setup_map_ax(ax2)

    if mask_p2_na.any():
        clipped[mask_p2_na].plot(ax=ax2, color="lightgrey", **poly_render_kwargs)
    if mask_no_ce.any():
        clipped[mask_no_ce].plot(ax=ax2, color="#4D4D4D", **poly_render_kwargs)
    if mask_p2_ok.any():
        clipped[mask_p2_ok].plot(
            ax=ax2,
            column="rel_change",
            cmap=cmap_div,
            norm=norm_div,
            **poly_render_kwargs,
        )

    if show_basin_boundaries:
        clipped.boundary.plot(
            ax=ax2,
            edgecolor=basin_boundary_color,
            linewidth=basin_boundary_lw,
            zorder=2,
            rasterized=True,
        )

    europe_outline.boundary.plot(
        ax=ax2, edgecolor="black", linewidth=0.5, zorder=3, rasterized=True
    )

    sm2 = plt.cm.ScalarMappable(cmap=cmap_div, norm=norm_div)
    sm2.set_array([])
    cbar2 = plt.colorbar(
        sm2,
        ax=ax2,
        orientation="horizontal",
        pad=0.01,
        shrink=0.9,
        aspect=20,
        extend="neither",
    )
    cbar2.set_ticks(delta_ticks)
    cbar2.set_ticklabels([f"{v:g}" for v in delta_ticks])
    cbar2.ax.tick_params(labelsize=cbar_tick_fontsize)
    cbar2.set_label("")
    cbar2.outline.set_linewidth(0.8)

    ax2.set_title(titles["panel2"], fontsize=title_fontsize, fontweight="bold", pad=4)
    ax2.text(
        0.0,
        1.0,
        f"({panel_labels[1]})",
        transform=ax2.transAxes,
        fontsize=title_fontsize - 1,
        fontweight="bold",
        va="top",
    )

    # ── Panel 3: KDE + strip plot + IQR box ───────────────────────────────────
    ax3 = fig.add_subplot(gs[0, 2])

    # Use the full merged set for the KDE (not clipped) to stay consistent
    # with the original behaviour.
    v61 = merged[col_61].dropna().values
    v91 = merged[col_91].dropna().values

    if len(v61) < 2 or len(v91) < 2:
        ax3.text(
            0.5,
            0.5,
            "Insufficient data",
            ha="center",
            va="center",
            transform=ax3.transAxes,
        )
    else:
        # ── KDE curves + subtle fill ──────────────────────────────────────────
        x_lo = kde_xlim[0] if kde_xlim else min(v61.min(), v91.min())
        x_hi = kde_xlim[1] if kde_xlim else max(v61.max(), v91.max())
        x_rng = np.linspace(x_lo, x_hi, 300)

        kde61 = stats.gaussian_kde(v61)
        kde91 = stats.gaussian_kde(v91)
        d61 = kde61(x_rng)
        d91 = kde91(x_rng)

        c61, c91 = "#4575b4", "#d73027"  # blue (past) / red (present)
        dc61, dc91 = "#1a4d80", "#8b1a1a"  # darker accents for IQR box edges

        ax3.plot(x_rng, d61, color=c61, lw=1.8, label="1961\u20131990", zorder=5)
        ax3.plot(x_rng, d91, color=c91, lw=1.8, label="1991\u20132020", zorder=5)
        ax3.fill_between(x_rng, d61, alpha=0.12, color=c61, zorder=4)
        ax3.fill_between(x_rng, d91, alpha=0.12, color=c91, zorder=4)

        mean61, mean91 = v61.mean(), v91.mean()

        # Dashed vertical mean lines through the KDE area
        ax3.axvline(mean61, color=c61, ls="--", lw=1.4, alpha=0.55, zorder=6)
        ax3.axvline(mean91, color=c91, ls="--", lw=1.4, alpha=0.55, zorder=6)

        if kde_xlim:
            ax3.set_xlim(kde_xlim)
        if kde_ylim:
            ax3.set_ylim(kde_ylim)
        if kde_xticks is not None:
            ax3.set_xticks(kde_xticks)
        if kde_yticks is not None:
            ax3.set_yticks(kde_yticks)

        # ── Spine style ───────────────────────────────────────────────────────
        ax3.spines["top"].set_visible(False)
        ax3.spines["right"].set_visible(False)
        ax3.spines["left"].set_linewidth(1)
        ax3.spines["left"].set_position(("outward", 5))
        ax3.spines["bottom"].set_linewidth(1)
        # Outward offset creates visual gap between spine and strip rows.
        ax3.spines["bottom"].set_position(("outward", 28))

        ax3.tick_params(axis="both", labelsize=cbar_tick_fontsize)

        def _fmt(val, pos):
            return f"{int(val)}" if val == int(val) else f"{val:.2f}"

        ax3.xaxis.set_major_formatter(FuncFormatter(_fmt))
        ax3.yaxis.set_major_formatter(FuncFormatter(_fmt))

        # ── Blended transform: x = data coords, y = axes coords ──────────────
        # Strip rows sit BELOW the spine (negative axes-coord y values).
        # clip_on=False allows rendering outside the axes bounding box.
        trans = blended_transform_factory(ax3.transData, ax3.transAxes)

        y_centre_61 = -0.030  # first row  — closer to spine  (1961-1990)
        y_centre_91 = -0.065  # second row — slightly lower   (1991-2020)
        box_half_h = strip_jitter + 0.008  # IQR box snugly wraps jitter
        lw_box = 0.7

        rng = np.random.default_rng(42)  # fixed seed → reproducible layout
        MAX_PTS = 3000

        # ── Strip plot ────────────────────────────────────────────────────────
        # Dots are filtered to [x_lo, x_hi] BEFORE plotting so they never
        # bleed outside the KDE x-limits.  clip_on=False is still required
        # in the y-direction so dots below the spine remain visible, but
        # because we pre-filter in x there is nothing to show outside xlim.
        for data_flat, yc, col in [
            (v61, y_centre_61, c61),
            (v91, y_centre_91, c91),
        ]:
            idx = rng.choice(
                len(data_flat), size=min(MAX_PTS, len(data_flat)), replace=False
            )
            sample = data_flat[idx]

            # Optional horizontal jitter in data units (default = 0 → off)
            jitter_x = (
                rng.uniform(-strip_jitter_x, strip_jitter_x, size=len(idx))
                if strip_jitter_x > 0
                else np.zeros(len(idx))
            )
            sample_x = sample + jitter_x

            # Vertical jitter keeps dots within the IQR box height (axes coords)
            jitter_y = rng.uniform(-strip_jitter, strip_jitter, size=len(idx))

            # Pre-filter: drop any dot whose x-value falls outside the plot
            # limits.  This prevents bleed-through even when clip_on=False.
            in_xlim = (sample_x >= x_lo) & (sample_x <= x_hi)
            sample_x = sample_x[in_xlim]
            jitter_y = jitter_y[in_xlim]

            ax3.scatter(
                sample_x,
                yc + jitter_y,
                s=strip_size,
                color=col,
                alpha=strip_alpha,
                linewidths=0,
                zorder=3,
                clip_on=False,
                transform=trans,
                rasterized=True,  # keeps PDF file size small
            )

        # ── IQR box + mean diamond + mean annotation ──────────────────────────
        for data_flat, yc, col, dc in [
            (v61, y_centre_61, c61, dc61),
            (v91, y_centre_91, c91, dc91),
        ]:
            q25 = np.percentile(data_flat, 25)
            q75 = np.percentile(data_flat, 75)
            mean = data_flat.mean()

            # Thin-edge IQR box — no fill, no whiskers, no median line, no caps
            box_rect = plt.Rectangle(
                (q25, yc - box_half_h),
                q75 - q25,
                2 * box_half_h,
                transform=trans,
                linewidth=lw_box,
                edgecolor=dc,
                facecolor="none",
                zorder=9,
                clip_on=False,
            )
            ax3.add_patch(box_rect)

            # Mean diamond — transparent fill, black edge, centred inside box
            ax3.scatter(
                [mean],
                [yc],
                marker="D",
                s=22,
                facecolors="none",
                edgecolors="black",
                linewidths=0.8,
                zorder=11,
                clip_on=False,
                transform=trans,
            )

            # Annotation placement:
            #   1961-1990 (past)    → ABOVE the box top edge  (va='bottom')
            #   1991-2020 (present) → BELOW the box bottom edge (va='top')
            if col == c61:
                text_y = yc + box_half_h
                va_side = "bottom"
            else:
                text_y = yc - box_half_h - 0.01
                va_side = "top"

            ax3.annotate(
                f"{mean:.3f}",
                xy=(mean, text_y),
                xytext=(mean, text_y),
                xycoords=trans,
                textcoords=trans,
                ha="center",
                va=va_side,
                fontsize=mean_fontsize,
                color=col,
                annotation_clip=False,
            )

        # ── Axes labels and legend ────────────────────────────────────────────
        ax3.set_ylabel("Density", fontsize=title_fontsize - 1)
        ax3.legend(
            ["1961\u20131990", "1991\u20132020"],
            loc="upper right",
            fontsize=title_fontsize,
            frameon=False,
            bbox_to_anchor=(1.05, 0.98),
        )
        ax3.text(
            -0.15,
            1.0,
            f"({panel_labels[2]})",
            transform=ax3.transAxes,
            fontsize=title_fontsize - 1,
            fontweight="bold",
            va="top",
        )

        # # ── Statistics printout ───────────────────────────────────────────────
        # q25_61, q75_61 = np.percentile(v61, [25, 75])
        # q25_91, q75_91 = np.percentile(v91, [25, 75])

        # print(f"\n{'='*60}")
        # print(f"  {basin_type} | {label} ({unit})")
        # print(f"  yellow_range = {yellow_range}  "
        #       f"(rel-change values shown in gold)")
        # print(f"{'─'*60}")
        # print(f"  {'Statistic':<22} {'1961-1990':>12} {'1991-2020':>12} "
        #       f"{'Δ':>10}")
        # print(f"{'─'*60}")
        # for name, v1, v2 in [
        #     ("Mean",   mean61,          mean91),
        #     ("Median", np.median(v61),  np.median(v91)),
        #     ("Q25",    q25_61,          q25_91),
        #     ("Q75",    q75_61,          q75_91),
        #     ("Min",    v61.min(),       v91.min()),
        #     ("Max",    v61.max(),       v91.max()),
        # ]:
        #     print(f"  {name:<22} {v1:>12.4f} {v2:>12.4f} {v2-v1:>10.4f}")
        # print(f"{'='*60}\n")

    ax3.set_title(titles["panel3"], fontsize=title_fontsize, fontweight="bold", pad=8)

    # ── Align KDE panel to map height ─────────────────────────────────────────
    plt.draw()
    pos0 = ax1.get_position()
    pos2 = ax3.get_position()
    ax3.set_position(
        [
            pos2.x0 + kde_position_adjust[0],
            pos0.y0 + kde_position_adjust[1],
            pos2.width + kde_position_adjust[2],
            pos0.height + kde_position_adjust[3],
        ]
    )

    # ── Save ──────────────────────────────────────────────────────────────────
    if save_dir and filename:
        os.makedirs(save_dir, exist_ok=True)
        for fmt in save_formats:
            fpath = os.path.join(save_dir, f"{filename}.{fmt}")
            fig.savefig(fpath, dpi=dpi, bbox_inches="tight", format=fmt)
            print(f"  Saved: {fpath}")

    return fig

## abc

In [ ]:
# ── Annual frequency ─────────────────────────────────────────────────────
plot_compound_metrics(
    results_df=EARLS_CE_Q10_TX90_1961to2020_FD_Intensity_ungauged,
    basin_shp=ungauged_shp,
    metric="AnnualFreq",
    basin_type="UnGauged",
    cmap_sequential=viridis_mod,
    bins_map=[0.2, 0.3, 0.4, 0.5, 0.6],  # np.arange(0.5, 3.0, 0.5),
    strip_jitter_x=0.05,
    extend_colorbar="both",
    kde_xticks=[0, 0.5, 1.0],
    kde_ylim=(0, 6.1),
    kde_yticks=[0, 5],
    # filename    = "FigS4_abc",
    save_formats=["pdf"],
    bins_delta=np.array([-3, -2, -1, -0.025, 0.025, 1, 2, 3]),
    yellow_range=(-0.025, 0.025),
)
plt.show()

## def

In [ ]:
plot_compound_metrics(
    results_df=EARLS_CE_Q10_TX90_1961to2020_FD_Intensity_ungauged,
    basin_shp=ungauged_shp,
    metric="AnnualDuration",
    basin_type="UnGauged",
    cmap_sequential=viridis_mod,
    bins_map=[0.4, 0.6, 0.8, 1.0, 1.2],  # np.arange(0.5, 3.0, 0.5),
    extend_colorbar="both",
    # filename    = "FigS4_def",
    bins_delta=np.array([-3, -2, -1, -0.025, 0.025, 1, 2, 3]),
    yellow_range=(-0.025, 0.025),
    kde_xlim=(0, 2.8),
    kde_ylim=(0, 1.25),
    kde_yticks=[0, 1],
    strip_jitter_x=0.05,
    panel_labels=("d", "e", "f"),
    save_formats=["pdf"],
    titles={
        "panel1": "Days/Event, 1991\u20132020",
        "panel2": "Relative \u0394",
        "panel3": "Pan-Europe Distribution",
    },
)
plt.show()

## g

In [ ]:
# ── Compute stats — one call per column ──────────────────────────────────────
stats_ce_freq = compute_region_stats_EARLS(
    df=EARLS_CE_Q10_TX90_1961to2020_FD_Intensity_ungauged,
    col_old="AnnualFreq_1961to1990",
    col_new="AnnualFreq_1991to2020",
    region_mask=europe_PRUDENCE_regions,
    basin_type="ungauged",
)
stats_ce_dur = compute_region_stats_EARLS(
    df=EARLS_CE_Q10_TX90_1961to2020_FD_Intensity_ungauged,
    col_old="AnnualDuration_1961to1990",
    col_new="AnnualDuration_1991to2020",
    region_mask=europe_PRUDENCE_regions,
    basin_type="ungauged",
)
stats_fd = compute_region_stats_EARLS(
    df=EARLS_CE_Q10_TX90_1961to2020_FD_Intensity_ungauged,
    col_old="FlowDeficit_scaled_1961to1990",
    col_new="FlowDeficit_scaled_1991to2020",
    region_mask=europe_PRUDENCE_regions,
    basin_type="ungauged",
)
stats_te = compute_region_stats_EARLS(
    df=EARLS_CE_Q10_TX90_1961to2020_FD_Intensity_ungauged,
    col_old="TempExcess_scaled_1961to1990",
    col_new="TempExcess_scaled_1991to2020",
    region_mask=europe_PRUDENCE_regions,
    basin_type="ungauged",
)

# ── Define columns ────────────────────────────────────────────────────────────
columns_def = [
    dict(
        stats=stats_ce_freq,
        title="CE Events/year",
        xlabel="Events per year",
        xmin=0.0,
        xmax=1.1,
        h_jitter=0.05,
        major_tick=0.5,
    ),
    dict(
        stats=stats_ce_dur,
        title="CE Days/year",
        xlabel="Days per year",
        xmin=0.0,
        xmax=3.1,
        major_tick=2.0,
    ),
    dict(
        stats=stats_fd,
        title="Flow Deficit",
        xlabel="% of JJAS Q10",
        xmin=-1.0,
        xmax=6.2,
        major_tick=4.0,
    ),
    dict(
        stats=stats_te,
        title="Temp Excess",
        xlabel="% of JJAS TX90",
        xmin=-0.2,
        xmax=5.2,
        major_tick=2.0,
    ),
]

fig = plot_strip_8x4(
    **{f"stats_col{i+1}": c["stats"] for i, c in enumerate(columns_def)},
    **{f"col{i+1}_title": c["title"] for i, c in enumerate(columns_def)},
    **{f"col{i+1}_xlabel": c["xlabel"] for i, c in enumerate(columns_def)},
    **{f"xmin_col{i+1}": c["xmin"] for i, c in enumerate(columns_def)},
    **{f"xmax_col{i+1}": c["xmax"] for i, c in enumerate(columns_def)},
    **{f"major_tick_col{i+1}": c["major_tick"] for i, c in enumerate(columns_def)},
    **{f"h_jitter_col{i+1}": c.get("h_jitter", 0.0) for i, c in enumerate(columns_def)},
    dot_stat="mean",
    jitter_frac=0.50,
    col_header_fs=11,
    annot_fs=9,
    row_height_mm=10,
    region_alpha={
        "FR": 0.1,
        "ME": 0.1,
        "AL": 0.1,
        "EA": 0.1,
        "MD": 0.1,
        "SC": 0.1,
        "IP": 0.1,
        "BI": 0.1,
    },
    panel_label="(g)",
    # filename     = 'FigS4_i',
    save_formats=["pdf"],
)
plt.show()

# Fig S5

In [ ]:
CE_Q10_TX90_Frequency_Change = xr.open_dataset(
    str(FIGDATA_DIR / "FigS5_a.nc")
)
LMDI_validGrids = xr.open_dataset(str(FIGDATA_DIR / "FigS5_b.nc"))

## a

In [ ]:
# =========================
# CONFIG
# =========================

figsize = (3.54, 3.54)
title = r"$\Delta N_{CE}$ (Present-Historical)"
title_fontsize = 10
linewidth = 0.0
vmin = -26
vcenter = 0
vmax = 26
cmap = cmaps.NCV_jaisnd
dpi = 150  # ← lower dpi = smaller PDF file
save_dir = str(FIGURE_DIR)
filename = "FigS5_a"
save_formats = ["pdf"]

# =========================
# PROJECTION
# =========================
projection = ccrs.LambertConformal(
    central_latitude=53, central_longitude=12, standard_parallels=(45, 65)
)

# =========================
# FIGURE
# =========================
fig, ax = plt.subplots(figsize=figsize, dpi=dpi, subplot_kw={"projection": projection})

# =========================
# MAP FEATURES
# =========================
add_map_features = create_map_features_adder(
    extent=[-9, 27, 35.6, 71],
    land_color="#FFFFFF",
    ocean_color="#FFFFFF",
    coastline_color="#000000",
    coastline_linewidth=1,
)
add_map_features(ax)

# =========================
# DATA PLOT
# =========================
threshold = -99998
data = CE_Q10_TX90_Frequency_Change["ce_change_observed"]
norm = TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)

# --- Main plot (normal range only)
data_main = data.where(data >= threshold)
data_main.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap=cmap,
    norm=norm,
    add_colorbar=True,
    zorder=2,
    rasterized=True,
    **{"antialiased": False, "edgecolors": "none", "linewidth": 0},
    cbar_kwargs={"label": r"$\Delta N_{CE}$", "aspect": 30},
)

# --- Overlay: values < -99998 in fixed color
data_under = data.where(data < threshold)
data_under.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap=plt.cm.colors.ListedColormap(["#4D4D4D"]),
    add_colorbar=False,
    rasterized=True,
    **{"antialiased": False, "edgecolors": "none", "linewidth": 0},
)

# =========================
# TITLE / AXES
# =========================
ax.set_title(title, fontsize=title_fontsize, pad=4, fontweight="bold")

set_axis_linewidth(ax, linewidth=linewidth)

# Panel label — stored as variable so we can un-rasterize it
panel_label = ax.text(
    0.0,
    1.0,
    "(a)",
    transform=ax.transAxes,
    fontsize=12,
    fontweight="bold",
    va="top",
    ha="left",
)

# =========================
# SAVE
# =========================
if filename is not None:
    os.makedirs(save_dir, exist_ok=True)
    for fmt in save_formats:
        filepath = os.path.join(save_dir, f"{filename}.{fmt}")
        if fmt.lower() in ["jpg", "jpeg", "png", "tiff"]:
            fig.savefig(filepath, dpi=dpi, format=fmt, bbox_inches="tight")
        else:
            fig.savefig(filepath, format=fmt, bbox_inches="tight", dpi=dpi)
        print(f"✓ Saved: {filepath}")

plt.show()

## b

In [ ]:
# =========================
# PROJECTION
# =========================
projection = ccrs.LambertConformal(
    central_latitude=53, central_longitude=12, standard_parallels=(45, 65)
)

# --- Create figure ---
fig, ax = plt.subplots(
    figsize=(3.54, 3.54),
    subplot_kw={"projection": projection},
)

# --- Colormap and normalization ---
# 0 = no data/ocean (white), 1 = valid grids (grey), 2 = masked grids (red)
colors = ["#FFFFFF", "#D3D3D3", "#DC143C"]
cmap_discrete = ListedColormap(colors)
boundaries = [-0.5, 0.5, 1.5, 2.5]
norm = BoundaryNorm(boundaries, cmap_discrete.N)

# --- Map features ---
add_map_features = create_map_features_adder(
    extent=[-9, 27, 35.6, 71],
    land_color="#FFFFFF",
    ocean_color="#FFFFFF",
    coastline_color="#000000",
    coastline_linewidth=1,
)
add_map_features(ax)

# --- Create categorical array ---
data = LMDI_validGrids.ce_change_observed
categorical = xr.where(
    data == -99999, 2, xr.where(np.isfinite(data), 1, 0)  # masked → red
)  # valid  → grey, NaN → white

# Count grids by category
valid_count = (categorical == 1).sum().item()
masked_count = (categorical == 2).sum().item()
nan_count = (categorical == 0).sum().item()

print(f"Valid grids  : {valid_count}")
print(f"Masked grids : {masked_count}")
print(f"NaN grids    : {nan_count}")

# --- Plot ---
categorical.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap=cmap_discrete,
    norm=norm,
    add_colorbar=False,
    rasterized=True,
    **{"antialiased": False, "edgecolors": "none", "linewidth": 0},  # Add comma here
)

# --- Custom legend ---
# legend_elements = [
#     #Patch(facecolor='#DC143C', label=r'$\Delta CE > 5,\ HWE_{\text{overlap}} \leq 2$'),
#     #Patch(facecolor='#D3D3D3', label='Valid grids'),
#     #Patch(facecolor='#FFFFFF', edgecolor='black', linewidth=0.5, label='No data'),
# ]
# ax.legend(
#     handles=legend_elements,
#     loc='upper left',
#     fontsize=9,
#     frameon=False,
# )

fig.text(
    0.378,
    1.06,  # centered below the map
    r"$\blacksquare$  $\Delta N_{CE}< 5,\ HWE_{\text{overlap}} \geq 3$",
    ha="center",
    va="bottom",
    fontsize=10,
    color="#DC143C",
)

# Panel label
ax.text(
    0.0,
    1,
    "(b)",
    transform=ax.transAxes,
    fontsize=12,
    fontweight="bold",
    va="top",
    ha="left",
)

ax.spines["geo"].set_visible(False)

# ax.set_title('Observed CE Change — Masked Grids', fontsize=11, fontweight='bold')
# plt.savefig('ce_change_masked.png', dpi=150, bbox_inches='tight')
# =========================
# SAVE
# =========================
dpi = 300
save_dir = str(FIGURE_DIR)
# filename = 'FigS5_b'          # e.g. "ce_change_observed"
save_formats = ["pdf"]

if filename is not None:
    os.makedirs(save_dir, exist_ok=True)
    for fmt in save_formats:
        filepath = os.path.join(save_dir, f"{filename}.{fmt}")
        if fmt.lower() in ["jpg", "jpeg", "png", "tiff"]:
            fig.savefig(filepath, dpi=dpi, format=fmt, bbox_inches="tight")
        else:
            fig.savefig(filepath, format=fmt, bbox_inches="tight")
        print(f"✓ Saved: {filepath}")

plt.show()

# Fig S6

In [ ]:
load_dir = str(FIGDATA_DIR) + os.sep

lf_Q10_1991to2020 = xr.open_dataset(
    load_dir + "FigS6_lowflow_1991to2020.nc"
)
lf_Q10_1961to1990 = xr.open_dataset(
    load_dir + "FigS6_lowflow_1961to1990.nc"
)
hw_TX90_1991to2020 = xr.open_dataset(
    load_dir + "FigS6_heatwave_1991to2020.nc"
)
hw_TX90_1961to1990 = xr.open_dataset(
    load_dir + "FigS6_heatwave_1961to1990.nc"
)

In [ ]:
def convert_timedelta_var(ds, variable_name):
    """
    Convert a timedelta64 variable in an xarray Dataset to float64 (days),
    preserving NaT as NaN.

    Parameters:
    -----------
    ds : xarray.Dataset
        Input dataset
    variable_name : str
        Name of the timedelta64 variable to convert

    Returns:
    --------
    xarray.Dataset
        Dataset with the specified variable converted to float64 days,
        with NaT values replaced by NaN
    """
    if np.issubdtype(ds[variable_name].dtype, np.timedelta64):
        ds = ds.copy()
        # Convert to float days; NaT becomes NaN automatically
        ds[variable_name] = ds[variable_name] / np.timedelta64(1, "D")
    return ds

## abcd

In [ ]:
fig = plot_four_panel_analysis(
    data_past_1=lf_Q10_1961to1990,
    data_present_1=lf_Q10_1991to2020,
    data_past_2=convert_timedelta_var(lf_Q10_1961to1990, "mean_annual_duration"),
    data_present_2=convert_timedelta_var(lf_Q10_1991to2020, "mean_annual_duration"),
    variable_name_1="mean_annual_frequency",
    variable_name_2="mean_annual_duration",
    bins_map_1=[0.8, 1.1, 1.4, 1.7, 2],
    bins_map_2=[4, 5, 6, 7, 8],
    bins_delta_1=np.array([-2, -1, -0.025, 0.025, 1.0, 2]),
    bins_delta_2=np.array([-1, -0.5, -0.025, 0.025, 0.5, 1]),
    yellow_range_1=(-0.025, 0.025),
    yellow_range_2=(-0.025, 0.025),
    use_nondefault_cmap=True,
    cmap_sequential_1=viridis_mod,
    cmap_sequential_2=viridis_mod,
    cmap_sequential_range=(0.05, 0.9),
    extend_colorbar_1="both",
    extend_colorbar_2="both",
    tick_positions_delta_1=[
        -2,
        -1,
        0,
        1,
        2,
    ],
    tick_positions_delta_2=[-1, -0.5, 0, 0.5, 1],
    titles={
        "panel1": "LFE (Event/yr), 1991\u20132020",
        "panel2": "Relative \u0394",
        "panel3": "LFE (Days/Event), 1991\u20132020",
        "panel4": "Relative \u0394",
    },
    # filename='FigS6_abcd',
    save_formats=["pdf"],
    dpi=300,
    figsize=(180, 75),  # mm (width, height)
    title_fontsize=11,
    cbar_tick_fontsize=10,
    tick_interval_1=1,
    tick_interval_2=1,
    add_map_features_func=add_map_features,
)

## efgh

In [ ]:
fig = plot_four_panel_analysis(
    data_past_1=hw_TX90_1961to1990,
    data_present_1=hw_TX90_1991to2020,
    data_past_2=convert_timedelta_var(hw_TX90_1961to1990, "mean_annual_duration"),
    data_present_2=convert_timedelta_var(hw_TX90_1991to2020, "mean_annual_duration"),
    variable_name_1="mean_annual_frequency",
    variable_name_2="mean_annual_duration",
    bins_map_1=[1.6, 1.8, 2, 2.2, 2.4],
    bins_map_2=[3, 3.25, 3.5, 3.75, 4],
    bins_delta_1=np.array([-2, -1, -0.025, 0.025, 1.0, 2]),
    bins_delta_2=np.array([-1, -0.5, -0.025, 0.025, 0.5, 1.0]),
    yellow_range_1=(-0.025, 0.025),
    yellow_range_2=(-0.025, 0.025),
    use_nondefault_cmap=True,
    cmap_sequential_1=viridis_mod,
    cmap_sequential_2=viridis_mod,
    cmap_sequential_range=(0.05, 0.9),
    extend_colorbar_1="both",
    extend_colorbar_2="both",
    tick_positions_delta_1=[
        -2,
        -1,
        0,
        1,
        2,
    ],
    tick_positions_delta_2=[-1, -0.5, 0, 0.5, 1],
    titles={
        "panel1": "HWE (Event/yr), 1991\u20132020",
        "panel2": "Relative \u0394",
        "panel3": "HWE (Days/Event), 1991\u20132020",
        "panel4": "Relative \u0394",
    },
    # filename='FigS6_efgh',
    panel_labels=["e", "f", "g", "h"],
    save_formats=["pdf"],
    dpi=300,
    figsize=(180, 75),  # mm (width, height)
    title_fontsize=11,
    cbar_tick_fontsize=10,
    tick_interval_1=1,
    tick_interval_2=1,
    add_map_features_func=add_map_features,
)

## i

In [ ]:
# ── four separate stat dicts, one per column ──────────────────────────────
stats_lfe_freq = compute_region_stats(
    data_old=lf_Q10_1961to1990,
    data_new=lf_Q10_1991to2020,
    region_mask=europe_PRUDENCE_regions,
    fill_value=-99999,
    var_name="mean_annual_frequency",
)
stats_hwe_freq = compute_region_stats(
    data_old=hw_TX90_1961to1990,
    data_new=hw_TX90_1991to2020,
    region_mask=europe_PRUDENCE_regions,
    fill_value=-99999,
    var_name="mean_annual_frequency",
)
stats_lfe_dur = compute_region_stats(
    data_old=convert_timedelta_var(lf_Q10_1961to1990, "mean_annual_duration"),
    data_new=convert_timedelta_var(lf_Q10_1991to2020, "mean_annual_duration"),
    region_mask=europe_PRUDENCE_regions,
    fill_value=-99999,
    var_name="mean_annual_duration",
)
stats_hwe_dur = compute_region_stats(
    data_old=convert_timedelta_var(hw_TX90_1961to1990, "mean_annual_duration"),
    data_new=convert_timedelta_var(hw_TX90_1991to2020, "mean_annual_duration"),
    region_mask=europe_PRUDENCE_regions,
    fill_value=-99999,
    var_name="mean_annual_duration",
)

# ── define columns as a list — easy to reorder, comment out, or swap ─────
columns_def = [
    dict(
        stats=stats_lfe_freq,
        title="LFE Events/year",
        xlabel="Events per year",
        xmin=0.0,
        xmax=3,
        h_jitter=0.05,
        major_tick=1.0,
    ),
    dict(
        stats=stats_hwe_freq,
        title="HWE Events/year",
        xlabel="Events per year",
        xmin=0.0,
        xmax=3.5,
        h_jitter=0.05,
        major_tick=1.0,
    ),
    dict(
        stats=stats_lfe_dur,
        title="LFE Days/Event",
        xlabel="Days per event",
        xmin=0.0,
        xmax=10.1,
        major_tick=4.0,
    ),
    dict(
        stats=stats_hwe_dur,
        title="HWE Days/Event",
        xlabel="Days per event",
        xmin=0.0,
        xmax=6,
        major_tick=3.0,
    ),
]

# ── reorder by simply changing the list order ─────────────────────────────
# e.g. swap frequency and duration:
# columns_def = [columns_def[2], columns_def[3], columns_def[0], columns_def[1]]

In [ ]:
# for name, stats in [('lfe_freq', stats_lfe_freq), ('hwe_freq', stats_hwe_freq),
#                     ('lfe_dur', stats_lfe_dur), ('hwe_dur', stats_hwe_dur)]:
#     for reg in REGIONS:
#         mo, mn = stats[reg].get('mean_old'), stats[reg].get('mean_new')
#         print(f'{name:9s} {reg}: mean_old={mo!r}  mean_new={mn!r}')

In [ ]:
fig = plot_strip_8x4(
    **{f"stats_col{i+1}": c["stats"] for i, c in enumerate(columns_def)},
    **{f"col{i+1}_title": c["title"] for i, c in enumerate(columns_def)},
    **{f"col{i+1}_xlabel": c["xlabel"] for i, c in enumerate(columns_def)},
    **{f"xmin_col{i+1}": c["xmin"] for i, c in enumerate(columns_def)},
    **{f"xmax_col{i+1}": c["xmax"] for i, c in enumerate(columns_def)},
    **{f"major_tick_col{i+1}": c["major_tick"] for i, c in enumerate(columns_def)},
    **{f"h_jitter_col{i+1}": c.get("h_jitter", 0.0) for i, c in enumerate(columns_def)},
    dot_stat="mean",  # or 'median'
    jitter_frac=0.50,  # vertical jitter, same for all columns
    col_header_fs=10,
    annot_fs=9,
    row_height_mm=10,
    region_alpha={
        "FR": 0.05,
        "ME": 0.05,
        "AL": 0.05,
        "EA": 0.05,
        "MD": 0.05,
        "SC": 0.015,
        "IP": 0.05,
        "BI": 0.05,
    },
    panel_label="(g)",
    # filename    = 'FigS6_i',
    save_formats=[
        "pdf",
    ],
)

# Fig S7

## abc

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RELOAD FIGURE DATA (skip recomputation)
# ─────────────────────────────────────────────────────────────────────────────

basin_type = "ungauged"  # used for titles/labels below

FILL_VALUE = -999999999999
PCT_COLS = ["Pct_N", "Pct_D", "Pct_R"]

figS7_abc_path = os.path.join(FIGDATA_DIR, "FigS7_abc.gpkg")
clipped = gpd.read_file(figS7_abc_path)

europe_boundary = europe_outline.boundary

In [ ]:
def plot_lmdi_map(clipped, europe_outline, basin_type, save_dir=None, show=True):
    """
    Replot the LMDI Attribution Map (Pct_N / Pct_D / Pct_R) from saved figure data.

    Parameters
    ----------
    clipped : GeoDataFrame
        Reloaded from FigS7_abc_{basin_type}.gpkg — must contain columns
        'idx', 'Pct_N', 'Pct_D', 'Pct_R', 'geometry'.
    europe_outline : GeoDataFrame
        Europe landmass polygon(s), used for background fill and boundary line.
    basin_type : str
        "gauged" or "ungauged" — used only for the title.
    save_dir : str or None
        If given, saves jpg + pdf into this directory. If None, does not save.
    show : bool
        Whether to call plt.show().

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    FILL_VALUE = -999999999999
    PCT_COLS = ["Pct_N", "Pct_D", "Pct_R"]
    COL_TITLES = [
        "Low–Flow Event frequency",
        "Low–Flow Event duration",
        "Heatwave Event Rate",
    ]
    PANEL_LABELS = ["(a)", "(b)", "(c)"]

    lon_min, lon_max = -12, 35
    lat_min, lat_max = 35.6, 71

    figsize_mm = (240, 90)
    dpi = 300
    title_fontsize = 12
    cbar_tick_fontsize = 8
    GRAY_NA = "lightgrey"

    # ── Colormap (identical bins/palette to original) ──────────────────────
    bins = np.array([-20, -10, -2, 2, 10, 20, 30, 40, 50, 60, 70, 80])
    yellow_range = (-2, 2)

    bounds_inner = bins.astype(float)
    boundaries = np.concatenate([[-1e9], bounds_inner, [1e9]])

    negative_bins = bins[bins < yellow_range[0]]
    positive_bins = bins[bins > yellow_range[1]]
    n_extreme_low = 1
    n_blue = len(negative_bins)
    n_yellow = 1
    n_red = len(positive_bins)
    n_extreme_high = 1

    base_cmap = plt.cm.PiYG
    extreme_blue = base_cmap(np.linspace(0.90, 1.0, n_extreme_low))
    blue_colors = base_cmap(np.linspace(0.80, 0.60, n_blue))
    yellow_colors = np.tile(to_rgba("#FFDF20"), (n_yellow, 1))
    red_colors = base_cmap(np.linspace(0.40, 0.10, n_red))
    extreme_red = base_cmap(np.linspace(0.0, 0.05, n_extreme_high))

    cmap_lmdi = ListedColormap(
        np.vstack([extreme_blue, blue_colors, yellow_colors, red_colors, extreme_red])
    )
    norm_lmdi = BoundaryNorm(boundaries, ncolors=cmap_lmdi.N)
    cbar_ticks = bins

    europe_boundary = europe_outline.boundary

    # ── Figure ───────────────────────────────────────────────────────────
    MM2IN = 1 / 25.4
    fig = plt.figure(figsize=(figsize_mm[0] * MM2IN, figsize_mm[1] * MM2IN))
    gs = GridSpec(
        1,
        3,
        figure=fig,
        left=0.01,
        right=0.99,
        top=0.90,
        bottom=0.22,
        wspace=0.06,
        hspace=0,
    )

    poly_kw = dict(
        rasterized=True, zorder=1, antialiased=False, edgecolors="none", linewidth=0.0
    )

    def _setup_ax(ax):
        europe_outline.plot(ax=ax, facecolor="#f0f0f0", edgecolor="none", zorder=0)
        ax.set_xlim(lon_min, lon_max)
        ax.set_ylim(lat_min, lat_max)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)

    sm = plt.cm.ScalarMappable(cmap=cmap_lmdi, norm=norm_lmdi)
    sm.set_array([])

    for col_idx, pct_col in enumerate(PCT_COLS):
        ax = fig.add_subplot(gs[0, col_idx])
        _setup_ax(ax)

        mask_below = clipped[pct_col] == FILL_VALUE
        mask_na = clipped[pct_col].isna()
        mask_grey = mask_below | mask_na
        mask_ok = ~mask_grey

        if mask_grey.any():
            clipped[mask_grey].plot(ax=ax, color=GRAY_NA, **poly_kw)

        if mask_ok.any():
            clipped[mask_ok].plot(
                ax=ax,
                column=pct_col,
                cmap=cmap_lmdi,
                norm=norm_lmdi,
                **poly_kw,
            )

        europe_boundary.plot(
            ax=ax, edgecolor="black", linewidth=0.5, rasterized=True, zorder=3
        )

        ax.set_title(
            COL_TITLES[col_idx], fontsize=title_fontsize, fontweight="bold", pad=4
        )

        ax.text(
            0.01,
            1.01,
            PANEL_LABELS[col_idx],
            transform=ax.transAxes,
            fontsize=title_fontsize - 1,
            fontweight="bold",
            va="bottom",
            ha="left",
        )

    cbar_ax = fig.add_axes([0.15, -0.04, 0.70, 0.030])
    cbar = fig.colorbar(
        sm,
        cax=cbar_ax,
        orientation="horizontal",
        ticks=cbar_ticks,
        boundaries=boundaries,
        extend="neither",
    )
    cbar.set_ticklabels([f"{v:g}" for v in cbar_ticks])
    cbar.ax.tick_params(labelsize=cbar_tick_fontsize)
    cbar.outline.set_linewidth(0.5)
    cbar.set_label(
        "Attribution to compound event frequency change (%)",
        fontsize=cbar_tick_fontsize + 1,
        labelpad=4,
    )

    gray_patch = mpatches.Patch(
        facecolor=GRAY_NA,
        edgecolor="0.5",
        linewidth=0.4,
        label="Below threshold (ΔCE < 5)",
    )
    fig.legend(
        handles=[gray_patch],
        loc="lower center",
        bbox_to_anchor=(0.50, -0.02),
        ncol=1,
        fontsize=cbar_tick_fontsize,
        framealpha=0.85,
        edgecolor="0.6",
        handlelength=1.2,
    )

    # fig.suptitle(f"LMDI Attribution — {basin_type.capitalize()} Basins",
    #              fontsize=title_fontsize + 1, fontweight="bold", y=1.05)

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        filename = "FigS7_abc"
        for fmt in ["pdf"]:
            fpath = os.path.join(save_dir, f"{filename}.{fmt}")
            fig.savefig(
                fpath,
                dpi=dpi if fmt != "pdf" else None,
                format=fmt,
                bbox_inches="tight",
            )
            print(f"✓ Saved: {fpath}")

    if show:
        plt.show()

    return fig

basin_type = "ungauged"

clipped = gpd.read_file(os.path.join(FIGDATA_DIR, "FigS7_abc.gpkg"))
# europe_outline must already be loaded in your session

fig = plot_lmdi_map(
    clipped, europe_outline, basin_type, save_dir=str(FIGURE_DIR) + os.sep
)

## d

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RELOAD FIGURE DATA (skip recomputation)
# ─────────────────────────────────────────────────────────────────────────────
basin_type = "ungauged"  # used for titles/labels below

figS7_d_path = os.path.join(FIGDATA_DIR, "FigS7_d.csv")
res = pd.read_csv(figS7_d_path)
print(f"Reloaded {len(res)} regions from {figS7_d_path}")

In [ ]:
def plot_LMDIprudence_bars(res, basin_type, save_dir=None, show=True):
    """
    Replot the PRUDENCE Regional Mean LMDI Attribution stacked bar chart
    from saved figure data.

    Parameters
    ----------
    res : DataFrame
        Reloaded from FigS7_d_{basin_type}.csv — must contain columns
        'abbrev', 'full', 'n', 'N_mean', 'D_mean', 'R_mean', 'N_se', 'D_se', 'R_se'.
    basin_type : str
        "gauged" or "ungauged" — used only for the title.
    save_dir : str or None
        If given, saves jpg + pdf into this directory. If None, does not save.
    show : bool
        Whether to call plt.show().

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    colors = {
        "N": "#4472C4",  # Blue   — LFE frequency
        "D": "#029E73",  # Teal   — LFE duration
        "R": "#ED7D31",  # Orange — HWE rate
    }

    N_means = res["N_mean"].values
    D_means = res["D_mean"].values
    R_means = res["R_mean"].values
    N_se = res["N_se"].values
    D_se = res["D_se"].values
    R_se = res["R_se"].values
    total_mean = N_means + D_means + R_means
    total_se = np.sqrt(N_se**2 + D_se**2 + R_se**2)
    region_labels = res["abbrev"].tolist()
    n_counts = res["n"].tolist()

    fig, ax = plt.subplots(figsize=(7.09, 3.2))

    x = np.arange(len(res))
    bar_width = 0.65

    N_pos = np.where(N_means > 0, N_means, 0)
    N_neg = np.where(N_means < 0, N_means, 0)
    D_pos = np.where(D_means > 0, D_means, 0)
    D_neg = np.where(D_means < 0, D_means, 0)
    R_pos = np.where(R_means > 0, R_means, 0)
    R_neg = np.where(R_means < 0, R_means, 0)

    bar_kw = dict(width=bar_width, edgecolor="black", linewidth=0.5)

    ax.bar(x, N_pos, color=colors["N"], label="Low–Flow Event frequency", **bar_kw)
    ax.bar(
        x,
        D_pos,
        bottom=N_pos,
        color=colors["D"],
        label="Low–Flow Event duration",
        **bar_kw,
    )
    ax.bar(
        x,
        R_pos,
        bottom=N_pos + D_pos,
        color=colors["R"],
        label="Heatwave Event Rate",
        **bar_kw,
    )

    ax.bar(x, N_neg, color=colors["N"], **bar_kw)
    ax.bar(x, D_neg, bottom=N_neg, color=colors["D"], **bar_kw)
    ax.bar(x, R_neg, bottom=N_neg + D_neg, color=colors["R"], **bar_kw)

    for i in range(len(res)):
        y_off = 0
        for val in [N_means[i], D_means[i], R_means[i]]:
            if val > 5:
                ax.text(
                    x[i],
                    y_off + val / 2,
                    f"{val:.0f}%",
                    ha="center",
                    va="center",
                    fontsize=6.5,
                    color="white",
                    fontweight="bold",
                )
            y_off += max(val, 0)

        y_off = 0
        for val in [N_means[i], D_means[i], R_means[i]]:
            if val < -5:
                ax.text(
                    x[i],
                    y_off + val / 2,
                    f"{val:.0f}%",
                    ha="center",
                    va="center",
                    fontsize=6.5,
                    color="black",
                    fontweight="bold",
                )
            y_off += min(val, 0)

    y_lim_bottom = min(np.nanmin(N_neg + D_neg + R_neg), 0) - 20
    for i, n in enumerate(n_counts):
        ax.text(
            x[i],
            y_lim_bottom + 10,
            f"n={n}",
            ha="center",
            va="top",
            fontsize=7,
            color="gray",
            style="italic",
        )

    ax.axhline(0, color="black", linewidth=0.8, zorder=5)

    ax.set_xticks(x)
    ax.set_xticklabels(region_labels, fontsize=10, fontweight="bold")
    ax.set_ylabel("Attribution to CE change (%)", fontsize=10, fontweight="bold")

    y_max = max(np.nanmax(total_mean + total_se), 0) + 65
    ax.set_ylim(y_lim_bottom, y_max)

    ax.grid(axis="y", alpha=0.3, linestyle="--", linewidth=0.5, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)
    ax.tick_params(axis="both", labelsize=9, width=0.8)

    ax.legend(frameon=False, fontsize=9, loc="upper left", ncols=3)

    ax.text(
        -0.08,
        1.05,
        "(d)",
        transform=ax.transAxes,
        fontsize=12,
        fontweight="bold",
        va="top",
        ha="left",
    )

    ax.set_title(
        f"Regional mean attribution to CE frequency change",
        fontsize=10,
        fontweight="bold",
        pad=10,
    )

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        fname = "FigS7_d"
        for fmt in ["pdf"]:
            fpath = os.path.join(save_dir, f"{fname}.{fmt}")
            fig.savefig(
                fpath,
                dpi=300 if fmt != "pdf" else None,
                format=fmt,
                bbox_inches="tight",
            )
            print(f"✓ Saved: {fpath}")

    if show:
        plt.show()

    return fig

basin_type = "ungauged"

res = pd.read_csv(os.path.join(FIGDATA_DIR, "FigS7_d.csv"))

fig = plot_LMDIprudence_bars(res, basin_type, save_dir=str(FIGURE_DIR) + os.sep)

# Fig S8

In [ ]:
HERA_LMF_ds = xr.open_dataset(str(FIGDATA_DIR / "FigS8_a.nc"))
EARLS_LMF_df = pd.read_csv(str(FIGDATA_DIR / "FigS8_b.csv"))

## a

In [ ]:
fig = plot_three_panel_analysis(
    data_1961to1990=HERA_LMF_ds["LMF_1961to1990"],  # DataArray directly
    data_1991to2020=HERA_LMF_ds["LMF_1991to2020"],  # DataArray directly
    CE_MISSING_FLAG=-99999,
    variable_name="_",  # placeholder — not found, so DataArray is used as-is
    add_map_features_func=add_map_features,
    cmap_sequential=viridis_mod,
    bins_map=[1, 1.5, 2, 2.5, 3],
    extend_colorbar="both",
    bins_delta=np.array([-1.0, -0.5, -0.25, -0.025, 0.025, 0.25, 0.5, 1.0]),
    tick_positions_delta=[-1.0, -0.5, -0.25, 0, 0.25, 0.5, 1.0],
    yellow_range=(-0.025, 0.025),
    kde_xlim=(0, 10),
    kde_ylim=(0, 0.6),
    kde_yticks=(0, 0.2, 0.4),
    kde_xticks=(0, 4, 8),
    cbar_tick_fontsize=9,
    # filename        = "FigS8_a",
    save_formats=["pdf"],
)
plt.show()

## b

In [ ]:
def plot_compound_metrics(
    results_df,
    basin_shp,
    metric,
    no_ce_df=None,
    europe_outline=None,
    europe_shp_path=str(FIGDATA_DIR / "European_Shapefile.shp"),
    basin_type="Gauged",
    lon_min=-12,
    lon_max=35,
    lat_min=35.6,
    lat_max=71,
    bins_map=None,
    cmap_sequential="YlOrRd",
    cmap_sequential_range=(0.1, 0.95),
    bins_delta=None,
    delta_ticks=None,
    yellow_range=(-0.1, 0.1),
    extend_colorbar="neither",
    kde_xlim=None,
    kde_ylim=None,
    kde_yticks=None,
    kde_xticks=None,
    # --- strip plot + IQR parameters (new) ---
    strip_jitter=0.012,
    strip_jitter_x=0.0,
    strip_alpha=0.1,
    strip_size=2,
    mean_fontsize=10,
    lollipop_style=True,
    lollipop_size=40,
    show_basin_boundaries=True,
    basin_boundary_color="black",
    basin_boundary_lw=0.01,
    titles=None,
    figsize_mm=(200, 120),
    title_fontsize=12,
    cbar_tick_fontsize=9,
    kde_position_adjust=(0.05, -0.078, 0.00, 0.01),
    save_dir=str(FIGURE_DIR) + os.sep,
    filename=None,
    save_formats=["jpg", "pdf"],
    dpi=300,
):

    # ── Panel titles ─────────────────────────────────────────────────────────
    if titles is None:
        titles = {
            "panel1": "Events/Yr, 1991–2020",
            "panel2": "Relative Δ",
            "panel3": "Pan-Europe Distribution",
        }

    # ── Column names ─────────────────────────────────────────────────────────
    col_61 = f"{metric}_1961to1990"
    col_91 = f"{metric}_1991to2020"
    for col in [col_61, col_91]:
        if col not in results_df.columns:
            raise ValueError(
                f"Column '{col}' not found.\nAvailable: {list(results_df.columns)}"
            )

    # ── Metadata defaults ─────────────────────────────────────────────────────
    meta = METRIC_META.get(metric, {})
    label = meta.get("label", metric)
    unit = meta.get("unit", "")
    bins_map = np.asarray(
        bins_map
        if bins_map is not None
        else meta.get("bins_map", np.arange(0, 1.1, 0.1))
    )
    bins_delta = np.asarray(
        bins_delta if bins_delta is not None else DEFAULT_DELTA_BINS
    )
    delta_ticks = delta_ticks if delta_ticks is not None else DEFAULT_DELTA_TICKS
    kde_xlim = kde_xlim if kde_xlim is not None else meta.get("kde_xlim")
    kde_ylim = kde_ylim if kde_ylim is not None else meta.get("kde_ylim", (0, 5))

    # ── Relative change ───────────────────────────────────────────────────────
    df = results_df.copy()
    df["rel_change"] = np.where(
        df[col_61] != 0, (df[col_91] - df[col_61]) / df[col_61], np.nan
    )

    # ── European boundary ─────────────────────────────────────────────────────
    if europe_outline is None:
        _shp = gpd.read_file(europe_shp_path)
        europe_outline = gpd.GeoDataFrame(
            geometry=[_shp.union_all()], crs=_shp.crs
        ).to_crs("EPSG:4326")

    # ── Merge results onto shapefile ──────────────────────────────────────────
    geo = basin_shp.copy()
    if "index" in geo.columns:
        geo = geo.rename(columns={"index": "idx"})
    if basin_type.lower() == "ungauged":
        geo["idx"] = geo["idx"].astype(str).str.replace(r"^hybas_", "", regex=True)
    geo = geo.to_crs("EPSG:4326")
    geo["idx"] = geo["idx"].astype(str)
    df["idx"] = df["idx"].astype(str)

    merged = geo.merge(df[["idx", col_61, col_91, "rel_change"]], on="idx", how="left")
    clipped = gpd.clip(merged, europe_outline)

    if no_ce_df is not None and not no_ce_df.empty:
        id_col = "idx" if "idx" in no_ce_df.columns else no_ce_df.columns[0]
        no_ce_ids = no_ce_df[id_col].astype(str).tolist()
        if basin_type.lower() == "ungauged":
            no_ce_ids = [s.replace("hybas_", "") for s in no_ce_ids]
        clipped["is_no_ce"] = clipped["idx"].isin(no_ce_ids)
    else:
        clipped["is_no_ce"] = False

    n_no_ce = clipped["is_no_ce"].sum()
    print(
        f"[{basin_type}] Matched: {merged[col_91].notna().sum()}/{len(merged)}  |  "
        f"Clipped: {len(clipped)}  |  No-CE basins: {n_no_ce}"
    )

    # ── Sequential colormap ───────────────────────────────────────────────────
    cbar1_extend = "neither"
    if isinstance(cmap_sequential, ListedColormap):
        colors_list = list(cmap_sequential.colors)
        n_needed = len(bins_map) + 1
        if len(colors_list) != n_needed:
            raise ValueError(f"ListedColormap must have exactly {n_needed} colors.")
        bounds_seq = np.concatenate([[-1e9], bins_map, [1e9]])
        cmap_seq = ListedColormap(colors_list)
        norm_seq = mcolors.BoundaryNorm(bounds_seq, ncolors=cmap_seq.N)
        cbar1_extend = "neither"
    elif isinstance(cmap_sequential, str):
        _cmap_base = plt.get_cmap(cmap_sequential)
        _extra = {"neither": 0, "min": 1, "max": 1, "both": 2}
        n_colors = (len(bins_map) - 1) + _extra.get(extend_colorbar, 1)
        colors_seq = _cmap_base(
            np.linspace(cmap_sequential_range[0], cmap_sequential_range[1], n_colors)
        )
        cmap_seq = ListedColormap(colors_seq)
        if extend_colorbar in ("min", "both"):
            cmap_seq.set_under("#4D4D4D")
        if extend_colorbar in ("max", "both"):
            cmap_seq.set_over(_cmap_base(cmap_sequential_range[1]))
        norm_seq = mcolors.BoundaryNorm(
            bins_map, ncolors=cmap_seq.N, extend=extend_colorbar
        )
        cbar1_extend = extend_colorbar
    else:
        raise ValueError("cmap_sequential must be a string or ListedColormap.")

    # ── Diverging colormap ────────────────────────────────────────────────────
    neg_bins = bins_delta[bins_delta < yellow_range[0]]
    pos_bins = bins_delta[bins_delta > yellow_range[1]]
    RdYlBu = plt.cm.RdYlBu
    colors_div = np.vstack(
        [
            RdYlBu(np.linspace(0.95, 1.0, 1)),
            RdYlBu(np.linspace(0.90, 0.65, len(neg_bins))),
            np.tile(to_rgba("#FFDF20"), (1, 1)),
            RdYlBu(np.linspace(0.35, 0.05, len(pos_bins))),
            RdYlBu(np.linspace(0.0, 0.05, 1)),
        ]
    )
    cmap_div = ListedColormap(colors_div)
    cmap_div.set_under("#4D4D4D")
    norm_div = mcolors.BoundaryNorm(
        np.concatenate([[-1e9], bins_delta, [1e9]]), ncolors=cmap_div.N
    )

    # ── Figure ────────────────────────────────────────────────────────────────
    MM2IN = 1 / 25.4
    fig = plt.figure(figsize=(figsize_mm[0] * MM2IN, figsize_mm[1] * MM2IN))
    gs = GridSpec(1, 3, figure=fig, wspace=0.12, width_ratios=[1, 1, 1])
    mpl.rcParams["xtick.direction"] = "in"
    mpl.rcParams["ytick.direction"] = "in"

    def _setup_map_ax(ax):
        europe_outline.plot(ax=ax, facecolor="#f0f0f0", edgecolor="none", zorder=0)
        ax.set_xlim(lon_min, lon_max)
        ax.set_ylim(lat_min, lat_max)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlabel("")
        ax.set_ylabel("")
        for spine in ax.spines.values():
            spine.set_visible(False)

    poly_render_kwargs = dict(
        rasterized=True, zorder=1, antialiased=True, edgecolors="k", linewidth=0.025
    )

    mask_no_ce = clipped["is_no_ce"]
    mask_p1_na = clipped[col_91].isna() & ~mask_no_ce
    mask_p1_ok = clipped[col_91].notna() & ~mask_no_ce
    mask_p2_na = clipped["rel_change"].isna() & ~mask_no_ce
    mask_p2_ok = clipped["rel_change"].notna() & ~mask_no_ce

    # ── Panel 1 ───────────────────────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, 0])
    _setup_map_ax(ax1)
    if mask_p1_na.any():
        clipped[mask_p1_na].plot(ax=ax1, color="lightgrey", **poly_render_kwargs)
    if mask_no_ce.any():
        clipped[mask_no_ce].plot(ax=ax1, color="#4D4D4D", **poly_render_kwargs)
    if mask_p1_ok.any():
        clipped[mask_p1_ok].plot(
            ax=ax1, column=col_91, cmap=cmap_seq, norm=norm_seq, **poly_render_kwargs
        )
    europe_outline.boundary.plot(
        ax=ax1, edgecolor="black", linewidth=0.5, zorder=3, rasterized=True
    )

    sm1 = plt.cm.ScalarMappable(cmap=cmap_seq, norm=norm_seq)
    sm1.set_array([])
    cbar1 = plt.colorbar(
        sm1,
        ax=ax1,
        orientation="horizontal",
        pad=0.01,
        shrink=0.9,
        aspect=20,
        extend=cbar1_extend,
    )
    cbar1.set_ticks(bins_map)
    cbar1.set_ticklabels([f"{v:g}" for v in bins_map])
    cbar1.ax.tick_params(labelsize=cbar_tick_fontsize)
    cbar1.outline.set_linewidth(0.8)
    ax1.set_title(titles["panel1"], fontsize=title_fontsize, fontweight="bold", pad=4)
    ax1.text(
        0.0,
        1.0,
        "(d)",
        transform=ax1.transAxes,
        fontsize=title_fontsize,
        fontweight="bold",
        va="top",
    )

    # ── Panel 2 ───────────────────────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[0, 1])
    _setup_map_ax(ax2)
    if mask_p2_na.any():
        clipped[mask_p2_na].plot(ax=ax2, color="lightgrey", **poly_render_kwargs)
    if mask_no_ce.any():
        clipped[mask_no_ce].plot(ax=ax2, color="#4D4D4D", **poly_render_kwargs)
    if mask_p2_ok.any():
        clipped[mask_p2_ok].plot(
            ax=ax2,
            column="rel_change",
            cmap=cmap_div,
            norm=norm_div,
            **poly_render_kwargs,
        )
    europe_outline.boundary.plot(
        ax=ax2, edgecolor="black", linewidth=0.5, zorder=3, rasterized=True
    )

    sm2 = plt.cm.ScalarMappable(cmap=cmap_div, norm=norm_div)
    sm2.set_array([])
    cbar2 = plt.colorbar(
        sm2,
        ax=ax2,
        orientation="horizontal",
        pad=0.01,
        shrink=0.9,
        aspect=20,
        extend="neither",
    )
    cbar2.set_ticks(delta_ticks)
    cbar2.set_ticklabels([f"{v:g}" for v in delta_ticks])
    cbar2.ax.tick_params(labelsize=cbar_tick_fontsize)
    cbar2.outline.set_linewidth(0.8)
    ax2.set_title(titles["panel2"], fontsize=title_fontsize, fontweight="bold", pad=4)
    ax2.text(
        0.0,
        1.0,
        "(e)",
        transform=ax2.transAxes,
        fontsize=title_fontsize,
        fontweight="bold",
        va="top",
    )

    # ── Panel 3: KDE + strip plot + IQR box ───────────────────────────────────
    ax3 = fig.add_subplot(gs[0, 2])

    v61 = merged[col_61].dropna().values
    v91 = merged[col_91].dropna().values

    color_1961 = "#4575b4"
    color_1991 = "#d73027"

    if len(v61) >= 2 and len(v91) >= 2:
        x_lo = kde_xlim[0] if kde_xlim else min(v61.min(), v91.min())
        x_hi = kde_xlim[1] if kde_xlim else max(v61.max(), v91.max())
        x_rng = np.linspace(x_lo, x_hi, 300)

        kde61 = stats.gaussian_kde(v61)
        kde91 = stats.gaussian_kde(v91)
        density_61 = kde61(x_rng)
        density_91 = kde91(x_rng)

        # KDE curves + fill
        ax3.plot(
            x_rng, density_61, color=color_1961, lw=1.8, label="1961–1990", zorder=5
        )
        ax3.plot(
            x_rng, density_91, color=color_1991, lw=1.8, label="1991–2020", zorder=5
        )
        ax3.fill_between(x_rng, density_61, alpha=0.12, color=color_1961, zorder=4)
        ax3.fill_between(x_rng, density_91, alpha=0.12, color=color_1991, zorder=4)

        # Mean vertical lines
        mean_61 = np.mean(v61)
        mean_91 = np.mean(v91)
        # ax3.axvline(mean_61, color=color_1961, ls="--", lw=1.5, alpha=0.85)
        # ax3.axvline(mean_91, color=color_1991, ls="--", lw=1.5, alpha=0.85)

        # ── Strip plot + IQR box (blended transform) ──────────────────────────
        trans = blended_transform_factory(ax3.transData, ax3.transAxes)

        y_centre_61 = -0.03  # first row  — closer to spine
        y_centre_91 = -0.065  # second row — slightly lower
        box_half_h = strip_jitter + 0.008
        lw_box = 0.7
        rng = np.random.default_rng(42)
        MAX_PTS = 3000

        for data_flat, yc, col in [
            (v61, y_centre_61, color_1961),
            (v91, y_centre_91, color_1991),
        ]:
            idx = rng.choice(
                len(data_flat), size=min(MAX_PTS, len(data_flat)), replace=False
            )
            sample = data_flat[idx]
            jitter_y = rng.uniform(-strip_jitter, strip_jitter, size=len(idx))
            jitter_x = (
                rng.uniform(-strip_jitter_x, strip_jitter_x, size=len(idx))
                if strip_jitter_x > 0
                else np.zeros(len(idx))
            )

            ax3.scatter(
                sample + jitter_x,
                yc + jitter_y,
                s=strip_size,
                color=col,
                alpha=strip_alpha,
                linewidths=0,
                zorder=3,
                clip_on=False,
                transform=trans,
                rasterized=True,
            )

            # IQR box
            q25 = np.percentile(data_flat, 25)
            q75 = np.percentile(data_flat, 75)
            mean = np.mean(data_flat)

            box_rect = plt.Rectangle(
                (q25, yc - box_half_h),
                q75 - q25,
                2 * box_half_h,
                transform=trans,
                linewidth=lw_box,
                edgecolor=col,
                facecolor="none",
                zorder=9,
                clip_on=False,
            )
            ax3.add_patch(box_rect)

            # Mean diamond
            ax3.scatter(
                [mean],
                [yc],
                marker="D",
                s=22,
                facecolors="none",
                edgecolors="black",
                linewidths=0.8,
                zorder=11,
                clip_on=False,
                transform=trans,
            )

            # Mean annotation — past above box, present below
            if col == color_1961:
                text_y, va_side = yc + box_half_h, "bottom"
            else:
                text_y, va_side = yc - box_half_h - 0.01, "top"

            ax3.annotate(
                f"{mean:.2f}",
                xy=(mean, text_y),
                xytext=(mean, text_y),
                xycoords=trans,
                textcoords=trans,
                fontweight="normal",
                ha="center",
                va=va_side,
                fontsize=mean_fontsize,
                color=col,
                annotation_clip=False,
            )

        # ── Axes formatting ───────────────────────────────────────────────────
        ax3.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax3.yaxis.set_major_locator(MaxNLocator(integer=True))
        ax3.tick_params(axis="both", labelsize=12)

        if kde_yticks is not None:
            ax3.set_yticks(kde_yticks)
        if kde_xticks is not None:
            ax3.set_xticks(kde_xticks)
        if kde_xlim:
            ax3.set_xlim(kde_xlim)
        if kde_ylim:
            ax3.set_ylim(kde_ylim)

        ax3.spines["top"].set_visible(False)
        ax3.spines["right"].set_visible(False)
        ax3.spines["left"].set_linewidth(0.8)
        ax3.spines["left"].set_position(("outward", 5))
        ax3.spines["bottom"].set_linewidth(0.8)
        ax3.spines["bottom"].set_position(("outward", 28))

        ax3.set_ylabel("Density", fontsize=title_fontsize)
        ax3.legend(
            loc="upper right",
            fontsize=title_fontsize,
            frameon=False,
            bbox_to_anchor=(1.05, 0.98),
        )

    ax3.set_title(titles["panel3"], fontsize=title_fontsize, fontweight="bold", pad=8)
    ax3.text(
        -0.18,
        1.0,
        "(f)",
        transform=ax3.transAxes,
        fontsize=title_fontsize,
        fontweight="bold",
        va="top",
    )

    # ── Align KDE panel height with map panels ────────────────────────────────
    plt.draw()
    pos0 = ax1.get_position()
    pos2 = ax3.get_position()
    ax3.set_position(
        [
            pos2.x0 + kde_position_adjust[0],
            pos0.y0 + kde_position_adjust[1],
            pos2.width + kde_position_adjust[2],
            pos0.height + kde_position_adjust[3],
        ]
    )

    # ── Save ──────────────────────────────────────────────────────────────────
    if save_dir and filename:
        os.makedirs(save_dir, exist_ok=True)
        for fmt in save_formats:
            fpath = os.path.join(save_dir, f"{filename}.{fmt}")
            fig.savefig(fpath, dpi=dpi, bbox_inches="tight", format=fmt)
            print(f"  ✓ Saved: {fpath}")

    return fig

In [ ]:
fig = plot_compound_metrics(
    results_df=EARLS_LMF_df,
    basin_shp=ungauged_shp,
    metric="LMF",
    europe_outline=europe_outline,
    basin_type="Ungauged",
    bins_map=[1, 1.5, 2, 2.5, 3],
    cmap_sequential=viridis_mod,
    bins_delta=np.array([-1, -0.5, -0.25, -0.025, 0.025, 0.25, 0.5, 1]),
    delta_ticks=[-1, -0.5, -0.25, 0, 0.25, 0.5, 1],
    yellow_range=(-0.025, 0.025),
    titles={
        "panel1": "LMF (1991\u20132020)",
        "panel2": "Relative \u0394",  # "Relative \u0394 (91-20 vs 61-90)"
        "panel3": "Pan-Europe Distribution",
    },
    kde_xlim=(0, 10),
    kde_ylim=(0, 0.62),
    kde_yticks=(0, 0.2, 0.4),
    kde_xticks=(0, 2, 4, 6, 8),
    kde_position_adjust=(0.05, -0.07, 0.00, 0.0),
    # filename        = "FigS8_b",
    save_formats=["pdf"],
)
plt.show()

# Fig S9

In [ ]:
def GRDC_HERA_LowFlowEvents(
    savename,
    figdata_dir=str(FIGDATA_DIR) + os.sep,
    save_dir=None,
    figsize=(18, 8),
    show=True,
):
    """
    Reload saved figure data and replot the 6-panel GRDC vs HERA
    qualifying low-flow events figure (scatter + KDE + bar chart rows).

    Parameters
    ----------
    savename : str
        Base name used when saving, e.g. 'GRDC_HERA_1991to2020_AllStations'.
        Loads '{savename}_annual_metrics.json' from figdata_dir.
    figdata_dir : str
        Directory containing the saved JSON figure data.
    save_dir : str or None
        If given, saves pdf + jpg into this directory. If None, does not save.
    figsize : tuple
        Figure size.
    show : bool
        Whether to call plt.show().

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    # ── Load data ────────────────────────────────────────────────────────
    figdata_path = os.path.join(figdata_dir, f"{savename}_data.json")
    with open(figdata_path, "r") as f:
        d = json.load(f)

    grdc_freq_aligned = d["grdc_freq_aligned"]
    hera_freq_aligned = d["hera_freq_aligned"]
    grdc_duration_aligned = d["grdc_duration_aligned"]
    hera_duration_aligned = d["hera_duration_aligned"]
    grdc_thresholds_clean = np.array(d["grdc_thresholds_clean"])
    hera_thresholds_clean = np.array(d["hera_thresholds_clean"])
    grdc_dur_nonzero = d["grdc_dur_nonzero"]
    hera_dur_nonzero = d["hera_dur_nonzero"]
    all_years = d["all_years"]
    grdc_annual_aligned = d["grdc_annual_aligned"]
    hera_annual_aligned = d["hera_annual_aligned"]
    qualifying_duration = d["qualifying_duration"]
    efficiency_criteria_description = d["efficiency_criteria_description"]
    common_stations_n = d["common_stations"]
    color_scheme = d.get("color_scheme", "steel_coral")

    print(f"Reloaded figure data: {figdata_path}")

    # ── Fonts / style (same as original) ────────────────────────────────
    mpl.rcParams["font.sans-serif"] = ["Arial"]
    mpl.rcParams["font.family"] = "sans-serif"
    mpl.rcParams["pdf.fonttype"] = 42
    mpl.rcParams["ps.fonttype"] = 42
    plt.style.use("default")
    mpl.rcParams["font.sans-serif"] = ["Arial"]
    mpl.rcParams["font.family"] = "sans-serif"
    mpl.rcParams["pdf.fonttype"] = 42
    mpl.rcParams["ps.fonttype"] = 42
    sns.set_palette("husl")

    color_schemes = {
        "steel_coral": {"grdc": "#4682b4", "hera": "#ff6b6b"},
        "classic_blue_orange": {"grdc": "#1f77b4", "hera": "#ff7f0e"},
    }
    colors = color_schemes.get(color_scheme, color_schemes["steel_coral"])
    grdc_color = colors["grdc"]
    hera_color = colors["hera"]

    # ── Figure ───────────────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 3, figsize=figsize, gridspec_kw={"hspace": 0.30})
    fig.suptitle(
        f"GRDC vs HERA Low Flow Events Analysis, {common_stations_n} Stations",
        fontsize=18,
        fontweight="bold",
    )

    # Plot 1: Mean Annual Frequency comparison
    axes[0, 0].scatter(
        grdc_freq_aligned,
        hera_freq_aligned,
        alpha=0.7,
        s=50,
        color="#1B4F72",
        edgecolors="white",
        linewidth=0.5,
    )
    max_freq = max(max(grdc_freq_aligned), max(hera_freq_aligned))
    axes[0, 0].plot(
        [0, max_freq],
        [0, max_freq],
        "--",
        color="#CB4335",
        alpha=0.8,
        linewidth=2,
        label="1:1 line",
    )
    if len(grdc_freq_aligned) > 0:
        pearson_r, pearson_p = stats.pearsonr(grdc_freq_aligned, hera_freq_aligned)
        spearman_r, spearman_p = stats.spearmanr(grdc_freq_aligned, hera_freq_aligned)
        corr_text = f"Pearson r={pearson_r:.3f}\nSpearman ρ={spearman_r:.3f}"
        axes[0, 0].text(
            0.05,
            0.95,
            corr_text,
            transform=axes[0, 0].transAxes,
            verticalalignment="top",
            fontsize=11,
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
        )
    axes[0, 0].set_xlabel("GRDC (events/year)", fontsize=14)
    axes[0, 0].set_ylabel("HERA (events/year)", fontsize=14)
    axes[0, 0].set_title(
        "Mean Annual Frequency Comparison", fontsize=15, fontweight="normal"
    )
    axes[0, 0].tick_params(axis="both", labelsize=12)
    axes[0, 0].legend(fontsize=12)

    # Plot 2: Mean Annual Duration comparison
    duration_mask = ~(
        (np.array(grdc_duration_aligned) == 0) | (np.array(hera_duration_aligned) == 0)
    )
    grdc_duration_clean = np.array(grdc_duration_aligned)[duration_mask]
    hera_duration_clean = np.array(hera_duration_aligned)[duration_mask]
    if len(grdc_duration_clean) > 0:
        axes[0, 1].scatter(
            grdc_duration_clean,
            hera_duration_clean,
            alpha=0.7,
            s=50,
            color="#1B4F72",
            edgecolors="white",
            linewidth=0.5,
        )
        max_duration = max(
            np.nanmax(grdc_duration_clean), np.nanmax(hera_duration_clean)
        )
        min_duration = min(
            np.nanmin(grdc_duration_clean), np.nanmin(hera_duration_clean)
        )
        axes[0, 1].plot(
            [min_duration, max_duration],
            [min_duration, max_duration],
            "--",
            color="#CB4335",
            alpha=0.8,
            linewidth=2,
            label="1:1 line",
        )
        pearson_r, pearson_p = stats.pearsonr(grdc_duration_clean, hera_duration_clean)
        spearman_r, spearman_p = stats.spearmanr(
            grdc_duration_clean, hera_duration_clean
        )
        corr_text = f"Pearson r={pearson_r:.3f}\nSpearman ρ={spearman_r:.3f}"
        axes[0, 1].text(
            0.05,
            0.95,
            corr_text,
            transform=axes[0, 1].transAxes,
            verticalalignment="top",
            fontsize=11,
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
        )
    axes[0, 1].set_xlabel("GRDC (days)", fontsize=14)
    axes[0, 1].set_ylabel("HERA (days)", fontsize=14)
    axes[0, 1].set_title(
        "Mean Annual Duration Comparison", fontsize=15, fontweight="normal"
    )
    axes[0, 1].tick_params(axis="both", labelsize=12)
    axes[0, 1].legend(fontsize=12)

    # Plot 3: Threshold comparison
    if len(grdc_thresholds_clean) > 0:
        axes[0, 2].scatter(
            grdc_thresholds_clean,
            hera_thresholds_clean,
            alpha=0.7,
            s=50,
            color="#1B4F72",
            edgecolors="white",
            linewidth=0.5,
        )
        max_threshold = max(
            np.nanmax(grdc_thresholds_clean), np.nanmax(hera_thresholds_clean)
        )
        min_threshold = min(
            np.nanmin(grdc_thresholds_clean), np.nanmin(hera_thresholds_clean)
        )
        axes[0, 2].plot(
            [min_threshold, max_threshold],
            [min_threshold, max_threshold],
            "--",
            color="#CB4335",
            alpha=0.8,
            linewidth=2,
            label="1:1 line",
        )
        pearson_r, pearson_p = stats.pearsonr(
            grdc_thresholds_clean, hera_thresholds_clean
        )
        spearman_r, spearman_p = stats.spearmanr(
            grdc_thresholds_clean, hera_thresholds_clean
        )
        corr_text = f"Pearson r={pearson_r:.3f}\nSpearman ρ={spearman_r:.3f}"
        axes[0, 2].text(
            0.05,
            0.95,
            corr_text,
            transform=axes[0, 2].transAxes,
            verticalalignment="top",
            fontsize=11,
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
        )
    axes[0, 2].set_xlabel("GRDC Average Q10 Threshold", fontsize=14)
    axes[0, 2].set_ylabel("HERA Average Q10 Threshold", fontsize=14)
    axes[0, 2].set_title("Threshold Comparison", fontsize=15, fontweight="normal")
    axes[0, 2].tick_params(axis="both", labelsize=12)
    axes[0, 2].legend(fontsize=12)

    # Plot 4: Frequency KDE
    if len(grdc_freq_aligned) > 0 and len(hera_freq_aligned) > 0:
        sns.kdeplot(
            data=grdc_freq_aligned,
            ax=axes[1, 0],
            label="GRDC",
            color=grdc_color,
            linewidth=2.5,
            alpha=0.4,
            fill=True,
        )
        sns.kdeplot(
            data=hera_freq_aligned,
            ax=axes[1, 0],
            label="HERA",
            color=hera_color,
            linewidth=2.5,
            alpha=0.4,
            fill=True,
        )
    axes[1, 0].set_xlabel("Mean Annual Frequency (events/year)", fontsize=14)
    axes[1, 0].set_ylabel("Density", fontsize=14)
    axes[1, 0].set_title(
        "Mean Annual Frequency Distribution", fontsize=15, fontweight="normal"
    )
    axes[1, 0].tick_params(axis="both", labelsize=12)
    axes[1, 0].legend(fontsize=12)

    # Plot 5: Duration KDE
    if len(grdc_dur_nonzero) > 0 and len(hera_dur_nonzero) > 0:
        sns.kdeplot(
            data=grdc_dur_nonzero,
            ax=axes[1, 1],
            label="GRDC",
            color=grdc_color,
            linewidth=2.5,
            alpha=0.4,
            fill=True,
        )
        sns.kdeplot(
            data=hera_dur_nonzero,
            ax=axes[1, 1],
            label="HERA",
            color=hera_color,
            linewidth=2.5,
            alpha=0.4,
            fill=True,
        )
    axes[1, 1].set_xlabel("Mean Annual Duration (days)", fontsize=14)
    axes[1, 1].set_ylabel("Density", fontsize=14)
    axes[1, 1].set_title(
        "Mean Annual Duration Distribution", fontsize=15, fontweight="normal"
    )
    axes[1, 1].tick_params(axis="both", labelsize=12)
    axes[1, 1].legend(fontsize=12)

    # Plot 6: Annual event distribution
    x = np.arange(len(all_years))
    width = 0.35
    axes[1, 2].bar(
        x - width / 2,
        grdc_annual_aligned,
        width,
        alpha=0.8,
        label="GRDC",
        color=grdc_color,
    )
    axes[1, 2].bar(
        x + width / 2,
        hera_annual_aligned,
        width,
        alpha=0.8,
        label="HERA",
        color=hera_color,
    )
    # axes[1, 2].set_xlabel('Year', fontsize=14)
    axes[1, 2].set_ylabel("Number of Events", fontsize=14)
    axes[1, 2].set_title("Annual Event Distribution", fontsize=15, fontweight="normal")

    specific_years = [1991, 1994, 1997, 2000, 2003, 2006, 2009, 2012, 2015, 2018, 2020]
    tick_positions, tick_labels = [], []
    for i, year in enumerate(all_years):
        if year in specific_years:
            tick_positions.append(i)
            tick_labels.append(str(year))
    axes[1, 2].set_xticks(tick_positions)
    axes[1, 2].set_xticklabels(tick_labels, rotation=45)
    axes[1, 2].tick_params(axis="both", labelsize=12)
    axes[1, 2].legend(fontsize=12)

    # ── Save ─────────────────────────────────────────────────────────────
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        pdf_path = os.path.join(save_dir, f"{savename}.pdf")
        fig.savefig(pdf_path, format="pdf", bbox_inches="tight", dpi=300)
        print(f"✓ Saved: {pdf_path}")

    if show:
        plt.show()

    return fig


fig = GRDC_HERA_LowFlowEvents(savename="FigS9", save_dir=str(FIGURE_DIR) + os.sep)

# Fig S10

In [ ]:
def GRDC_HERA_compare_map(
    savename,
    figdata_dir=str(FIGDATA_DIR) + os.sep,
    europe_outline=None,
    figsize=(12, 5),
    scatter_size=7,
    cbar_ticksize=12,
    save_dir=None,
    show=True,
):
    """
    Reload saved spatial figure data and replot the GRDC vs HERA vs Difference
    maps for mean annual frequency (fig1) and mean annual duration (fig2).

    Parameters
    ----------
    savename : str
        Base name used when saving, e.g. 'GRDC_HERA_1991to2020_AllStations'.
        Loads '{savename}_spatial.json' from figdata_dir.
    figdata_dir : str
        Directory containing the saved JSON figure data.
    europe_outline : GeoDataFrame
        Europe landmass outline, required for map background/boundary.
    figsize : tuple
        Figure size (per figure).
    scatter_size : int
        Marker size for scatter points.
    cbar_ticksize : int
        Colorbar tick label size.
    save_dir : str or None
        If given, saves pdf + jpg into this directory. If None, does not save.
    show : bool
        Whether to call plt.show().

    Returns
    -------
    fig1, fig2 : matplotlib.figure.Figure
        Frequency comparison figure, duration comparison figure.
    """
    if europe_outline is None:
        raise ValueError(
            "europe_outline must be provided (Europe landmass GeoDataFrame)."
        )

    # ── Load data ────────────────────────────────────────────────────────
    figdata_path = os.path.join(figdata_dir, f"{savename}_data.json")
    with open(figdata_path, "r") as f:
        d = json.load(f)

    grdc_lons, grdc_lats, grdc_vals = d["grdc_lons"], d["grdc_lats"], d["grdc_vals"]
    hera_lons, hera_lats, hera_vals = d["hera_lons"], d["hera_lats"], d["hera_vals"]
    diff_lons, diff_lats, diff_vals = d["diff_lons"], d["diff_lats"], d["diff_vals"]
    grdc_dur_lons, grdc_dur_lats, grdc_dur_vals = (
        d["grdc_dur_lons"],
        d["grdc_dur_lats"],
        d["grdc_dur_vals"],
    )
    hera_dur_lons, hera_dur_lats, hera_dur_vals = (
        d["hera_dur_lons"],
        d["hera_dur_lats"],
        d["hera_dur_vals"],
    )
    diff_dur_lons, diff_dur_lats, diff_dur_vals = (
        d["diff_dur_lons"],
        d["diff_dur_lats"],
        d["diff_dur_vals"],
    )

    print(f"Reloaded figure data: {figdata_path}")

    # ── Fonts / style ────────────────────────────────────────────────────
    mpl.rcParams["font.sans-serif"] = ["Arial"]
    mpl.rcParams["font.family"] = "sans-serif"
    mpl.rcParams["pdf.fonttype"] = 42
    mpl.rcParams["ps.fonttype"] = 42
    plt.style.use("default")
    mpl.rcParams["font.sans-serif"] = ["Arial"]
    mpl.rcParams["font.family"] = "sans-serif"
    mpl.rcParams["pdf.fonttype"] = 42
    mpl.rcParams["ps.fonttype"] = 42
    sns.set_palette("husl")

    # ── Boundaries, colormaps, norms (identical to original) ──────────────
    boundaries_freq = np.concatenate(
        (np.array([-1e6]), np.arange(0.5, 2.5, 0.5), np.array([1e6])), axis=0
    )
    cmap_freq = plt.get_cmap("viridis", len(boundaries_freq) - 1)
    norm_freq = mcolors.BoundaryNorm(boundaries_freq, cmap_freq.N)

    freq_diff_min, freq_diff_max, freq_diff_gap = -0.6, 0.7, 0.2
    boundaries_freq_diff = np.concatenate(
        (
            np.array([-1e6]),
            np.arange(freq_diff_min, freq_diff_max, freq_diff_gap),
            np.array([1e6]),
        ),
        axis=0,
    )
    cmap_freq_diff = plt.get_cmap(cmc.vik, len(boundaries_freq_diff) - 1)
    norm_freq_diff = mcolors.BoundaryNorm(boundaries_freq_diff, cmap_freq_diff.N)

    duration_max = 6
    boundaries_duration = np.concatenate(
        (np.array([-1e6]), np.arange(1, duration_max, 1), np.array([1e6])), axis=0
    )
    cmap_duration = plt.get_cmap("viridis", len(boundaries_duration) - 1)
    norm_duration = mcolors.BoundaryNorm(boundaries_duration, cmap_duration.N)

    duration_diff_min, duration_diff_max, duration_diff_gap = -3, 3.5, 1
    boundaries_duration_diff = np.concatenate(
        (
            np.array([-1e6]),
            np.arange(duration_diff_min, duration_diff_max, duration_diff_gap),
            np.array([1e6]),
        ),
        axis=0,
    )
    cmap_duration_diff = plt.get_cmap(cmc.vik, len(boundaries_duration_diff) - 1)
    norm_duration_diff = mcolors.BoundaryNorm(
        boundaries_duration_diff, cmap_duration_diff.N
    )

    extent = [-9, 27, 35.6, 71]
    land_color = "#FFFFFF"
    coastline_color = "#000000"

    def add_map_features(ax):
        ax.set_extent(extent, crs=ccrs.PlateCarree())
        ax.add_geometries(
            europe_outline.geometry,
            ccrs.PlateCarree(),
            facecolor=land_color,
            edgecolor="none",
            zorder=1,
        )
        ax.add_geometries(
            europe_outline.boundary,
            ccrs.PlateCarree(),
            facecolor="none",
            edgecolor=coastline_color,
            linewidth=1,
            zorder=3,
        )
        ax.gridlines(draw_labels=False, lw=0)

    proj = ccrs.LambertConformal(
        central_latitude=53, central_longitude=12, standard_parallels=(45, 65)
    )

    # ========== FIGURE 1: MEAN ANNUAL FREQUENCY ==========
    fig1, axes1 = plt.subplots(1, 3, figsize=figsize, subplot_kw={"projection": proj})

    add_map_features(axes1[0])
    sc1 = axes1[0].scatter(
        grdc_lons,
        grdc_lats,
        c=grdc_vals,
        cmap=cmap_freq,
        norm=norm_freq,
        s=scatter_size,
        transform=ccrs.PlateCarree(),
    )
    axes1[0].set_title("GRDC", fontsize=14)
    cbar1 = plt.colorbar(
        sc1, ax=axes1[0], fraction=0.046, pad=0.04, orientation="horizontal"
    )
    cbar1.set_label("Events per Year", fontsize=11)
    cbar1.set_ticks(np.arange(0.5, 2.5, 0.5))
    cbar1.ax.tick_params(labelsize=cbar_ticksize, direction="in", length=4)

    add_map_features(axes1[1])
    sc2 = axes1[1].scatter(
        hera_lons,
        hera_lats,
        c=hera_vals,
        cmap=cmap_freq,
        norm=norm_freq,
        s=scatter_size,
        transform=ccrs.PlateCarree(),
    )
    axes1[1].set_title("HERA", fontsize=14)
    cbar2 = plt.colorbar(
        sc2, ax=axes1[1], fraction=0.046, pad=0.04, orientation="horizontal"
    )
    cbar2.set_label("Events per Year", fontsize=11)
    cbar2.set_ticks(np.arange(0.5, 2.5, 0.5))
    cbar2.ax.tick_params(labelsize=cbar_ticksize, direction="in", length=4)

    add_map_features(axes1[2])
    sc3 = axes1[2].scatter(
        diff_lons,
        diff_lats,
        c=diff_vals,
        cmap=cmap_freq_diff,
        norm=norm_freq_diff,
        s=scatter_size,
        transform=ccrs.PlateCarree(),
    )
    axes1[2].set_title("Difference (HERA - GRDC)", fontsize=14)
    cbar3 = plt.colorbar(
        sc3, ax=axes1[2], fraction=0.046, pad=0.04, orientation="horizontal"
    )
    cbar3.set_label("Difference (events/year)", fontsize=11)
    cbar3.set_ticks(np.arange(freq_diff_min, freq_diff_max, freq_diff_gap))
    cbar3.ax.tick_params(labelsize=cbar_ticksize, direction="in", length=4)

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        fig1.savefig(
            os.path.join(save_dir, f"{savename}_frequency.pdf"),
            format="pdf",
            bbox_inches="tight",
            dpi=300,
        )
        print(f"✓ Saved: {os.path.join(save_dir, f'{savename}_frequency.pdf')}")

    # ========== FIGURE 2: MEAN ANNUAL DURATION ==========
    fig2, axes2 = plt.subplots(1, 3, figsize=figsize, subplot_kw={"projection": proj})

    add_map_features(axes2[0])
    sc4 = axes2[0].scatter(
        grdc_dur_lons,
        grdc_dur_lats,
        c=grdc_dur_vals,
        cmap=cmap_duration,
        norm=norm_duration,
        s=scatter_size,
        transform=ccrs.PlateCarree(),
    )
    axes2[0].set_title("GRDC", fontsize=14)
    cbar4 = plt.colorbar(
        sc4, ax=axes2[0], fraction=0.046, pad=0.04, orientation="horizontal"
    )
    cbar4.set_label("Duration (days/event)", fontsize=11)
    cbar4.set_ticks(np.arange(1, duration_max, 1))
    cbar4.ax.tick_params(labelsize=cbar_ticksize, direction="in", length=4)

    add_map_features(axes2[1])
    sc5 = axes2[1].scatter(
        hera_dur_lons,
        hera_dur_lats,
        c=hera_dur_vals,
        cmap=cmap_duration,
        norm=norm_duration,
        s=scatter_size,
        transform=ccrs.PlateCarree(),
    )
    axes2[1].set_title("HERA", fontsize=14)
    cbar5 = plt.colorbar(
        sc5, ax=axes2[1], fraction=0.046, pad=0.04, orientation="horizontal"
    )
    cbar5.set_label("Duration (days/event)", fontsize=11)
    cbar5.set_ticks(np.arange(1, duration_max, 1))
    cbar5.ax.tick_params(labelsize=cbar_ticksize, direction="in", length=4)

    add_map_features(axes2[2])
    sc6 = axes2[2].scatter(
        diff_dur_lons,
        diff_dur_lats,
        c=diff_dur_vals,
        cmap=cmap_duration_diff,
        norm=norm_duration_diff,
        s=scatter_size,
        transform=ccrs.PlateCarree(),
    )
    axes2[2].set_title("Difference (HERA - GRDC)", fontsize=14)
    cbar6 = plt.colorbar(
        sc6, ax=axes2[2], fraction=0.046, pad=0.04, orientation="horizontal"
    )
    cbar6.set_label("Difference (days/event)", fontsize=11)
    cbar6.set_ticks(np.arange(duration_diff_min, duration_diff_max, duration_diff_gap))
    cbar6.ax.tick_params(labelsize=cbar_ticksize, direction="in", length=4)

    if save_dir:
        fig2.savefig(
            os.path.join(save_dir, f"{savename}_duration.pdf"),
            format="pdf",
            bbox_inches="tight",
            dpi=300,
        )
        print(f"✓ Saved: {os.path.join(save_dir, f'{savename}_duration.pdf')}")

    if show:
        plt.show()

    return fig1, fig2


fig1, fig2 = GRDC_HERA_compare_map(
    savename="FigS10",
    europe_outline=europe_outline,  # must be loaded in your session
    save_dir=str(FIGURE_DIR) + os.sep,
)

# Fig S11

Concurrent LF-HW Events with minimum overlap between low flow events and heatwave events of 3 days

In [ ]:
load_dir = str(FIGDATA_DIR) + os.sep

CE_3dayoverlap_Q10_TX90_1991to2020_FD = xr.open_dataset(
    load_dir + "FigS11_FD_1991to2020.nc",
    decode_times=False,
)
CE_3dayoverlap_Q10_TX90_1991to2020_Intensity = xr.open_dataset(
    load_dir + "FigS11_Intensity_1991to2020.nc"
)
CE_3dayoverlap_Q10_TX90_1961to1990_FD = xr.open_dataset(
    load_dir + "FigS11_FD_1961to1990.nc",
    decode_times=False,
)
CE_3dayoverlap_Q10_TX90_1961to1990_Intensity = xr.open_dataset(
    load_dir + "FigS11_Intensity_1961to1990.nc"
)

## abc

In [ ]:
# Create map features adder
add_map_features = create_map_features_adder()

# Example: Save the figure
fig = plot_three_panel_analysis(
    data_1961to1990=CE_3dayoverlap_Q10_TX90_1961to1990_FD,
    data_1991to2020=CE_3dayoverlap_Q10_TX90_1991to2020_FD,
    CE_MISSING_FLAG=-99999999,
    variable_name="mean_annual_frequency",
    cmap_sequential=viridis_mod,
    extend_colorbar="both",
    bins_map=[
        0.5,
        0.6,
        0.7,
        0.8,
        1,
    ],  # [0.5, 0.6, 0.7, 0.8, 0.9], #np.arange(0.25, 1.4, 0.25),
    bins_delta=np.array([-3, -2, -1, -0.025, 0.025, 1, 2, 3]),
    yellow_range=(-0.025, 0.025),
    kde_ylim=(0, 4),
    kde_xlim=(0, 1.9),
    kde_yticks=(0, 1, 2, 3),
    # kde_position_adjust=(0.05, -0.078, 0.00, 0.01), #(x_offset, y_offset, w_offset, h_offset) to adjust KDE panel position
    tick_positions_delta=[-3, -2, -1, 0, 1, 2, 3],
    # filename='FigS11_abc',  # Save with this name
    save_formats=["pdf"],
    dpi=300,
    titles={
        "panel1": "Events/Yr, 1991\u20132020",
        "panel2": "Relative Δ",  # "Relative \u0394 (91-20 vs 61-90)"
        "panel3": "Pan-European Distribution",
    },
    tick_interval=1,
    title_fontsize=12,
    cbar_tick_fontsize=12,
    add_map_features_func=add_map_features,
)
plt.show()

## def

In [ ]:
# Create map features adder
add_map_features = create_map_features_adder()

# Example: Save the figure
fig = plot_three_panel_analysis(
    data_1961to1990=convert_timedelta_var(
        CE_3dayoverlap_Q10_TX90_1961to1990_FD, "mean_annual_duration"
    ),
    data_1991to2020=convert_timedelta_var(
        CE_3dayoverlap_Q10_TX90_1991to2020_FD, "mean_annual_duration"
    ),
    CE_MISSING_FLAG=-99999999,
    variable_name="mean_annual_duration",
    cmap_sequential=viridis_mod,
    bins_map=[1, 1.25, 1.5, 1.75, 2.0],  # np.arange(0.5, 3.0, 0.5),
    extend_colorbar="both",
    bins_delta=np.array([-3, -2, -1, -0.025, 0.025, 1, 2, 3]),
    yellow_range=(-0.025, 0.025),
    kde_ylim=(0, 1.35),
    kde_yticks=(0, 1.0),
    kde_xlim=(0, 3.8),
    tick_positions_delta=[-3, -2, -1, 0, 1, 2, 3],
    # filename='FigS11_def',  # Save with this name
    panel_labels=["d", "e", "f"],
    save_formats=["pdf"],
    dpi=300,
    titles={
        "panel1": "Days/Event, 1991\u20132020",
        "panel2": "Relative \u0394",  # "Relative \u0394 (91-20 vs 61-90)"
        "panel3": "Pan-European Distribution",
    },
    tick_interval=1,
    title_fontsize=12,
    cbar_tick_fontsize=12,
    add_map_features_func=add_map_features,
)
plt.show()

## g

In [ ]:
# ── four separate stat dicts, one per column ─────────────────────────────────
stats_ce_freq = compute_region_stats(
    data_old=CE_3dayoverlap_Q10_TX90_1961to1990_FD,
    data_new=CE_3dayoverlap_Q10_TX90_1991to2020_FD,
    region_mask=europe_PRUDENCE_regions,
    fill_value=-99999999,
    var_name="mean_annual_frequency",
)
stats_ce_dur = compute_region_stats(
    data_old=convert_timedelta_var(
        CE_3dayoverlap_Q10_TX90_1961to1990_FD, "mean_annual_duration"
    ),
    data_new=convert_timedelta_var(
        CE_3dayoverlap_Q10_TX90_1991to2020_FD, "mean_annual_duration"
    ),
    region_mask=europe_PRUDENCE_regions,
    fill_value=-99999999,
    var_name="mean_annual_duration",
)
stats_ce_lowflwdef = compute_region_stats(
    data_old=CE_3dayoverlap_Q10_TX90_1961to1990_Intensity,
    data_new=CE_3dayoverlap_Q10_TX90_1991to2020_Intensity,
    region_mask=europe_PRUDENCE_regions,
    fill_value=-99999999,
    var_name="mean_annual_flow_intensity_scaled",
)
stats_ce_temExs = compute_region_stats(
    data_old=CE_3dayoverlap_Q10_TX90_1961to1990_Intensity,
    data_new=CE_3dayoverlap_Q10_TX90_1991to2020_Intensity,
    region_mask=europe_PRUDENCE_regions,
    fill_value=-99999999,
    var_name="mean_annual_temperature_intensity_scaled",
)

# ── define columns as a list — easy to reorder, comment out, or swap ──────────
# h_jitter : horizontal jitter as a fraction of the x-range (0.0 = off).
#             Use a small value (e.g. 0.02–0.05) for integer-valued or
#             low-cardinality columns where dots would otherwise stack.
columns_def = [
    dict(
        stats=stats_ce_freq,
        title="Events per year",
        xlabel="Events per year",
        xmin=0.0,
        xmax=1.5,
        major_tick=1.0,
        h_jitter=0.05,
    ),
    dict(
        stats=stats_ce_dur,
        title="Days per event",
        xlabel="Days per event",
        xmin=0.0,
        xmax=3,
        major_tick=1.0,
        h_jitter=0.0,
    ),
    dict(
        stats=stats_ce_lowflwdef,
        title="Flow Deficit (%)",
        xlabel="Flow Deficit (%)",
        xmin=0.0,
        xmax=22,
        major_tick=8.0,
        h_jitter=0.0,
    ),
    dict(
        stats=stats_ce_temExs,
        title="Temp. Excess (%)",
        xlabel="Temp. Excess (%)",
        xmin=0.0,
        xmax=10.3,
        major_tick=4.0,
        h_jitter=0.0,
    ),
]

# ── reorder by simply changing the list order ─────────────────────────────────
# e.g. swap frequency and duration:
# columns_def = [columns_def[2], columns_def[3], columns_def[0], columns_def[1]]

# ── unpack and plot ───────────────────────────────────────────────────────────
fig = plot_strip_8x4(
    **{f"stats_col{i+1}": c["stats"] for i, c in enumerate(columns_def)},
    **{f"col{i+1}_title": c["title"] for i, c in enumerate(columns_def)},
    **{f"col{i+1}_xlabel": c["xlabel"] for i, c in enumerate(columns_def)},
    **{f"xmin_col{i+1}": c["xmin"] for i, c in enumerate(columns_def)},
    **{f"xmax_col{i+1}": c["xmax"] for i, c in enumerate(columns_def)},
    **{f"major_tick_col{i+1}": c["major_tick"] for i, c in enumerate(columns_def)},
    **{f"h_jitter_col{i+1}": c.get("h_jitter", 0.0) for i, c in enumerate(columns_def)},
    dot_stat="mean",  # or 'median'
    jitter_frac=0.50,  # vertical jitter, same for all columns
    col_header_fs=10,
    annot_fs=9,
    row_height_mm=11,
    region_alpha={
        "FR": 0.05,
        "ME": 0.05,
        "AL": 0.05,
        "EA": 0.05,
        "MD": 0.05,
        "SC": 0.015,
        "IP": 0.05,
        "BI": 0.05,
    },
    panel_label="(g)",
    # filename    = 'FigS11_g_strip_8x4',
    save_formats=["pdf"],
)

# Fig S12

In [ ]:
load_dir = str(FIGDATA_DIR) + os.sep

CE_Q20_TX80_1991to2020_FD = xr.open_dataset(
    load_dir + "FigS12_FD_1991to2020.nc", decode_times=False
)
CE_Q20_TX80_1991to2020_Intensity = xr.open_dataset(
    load_dir + "FigS12_Intensity_1991to2020.nc"
)
CE_Q20_TX80_1961to1990_FD = xr.open_dataset(
    load_dir + "FigS12_FD_1961to1990.nc", decode_times=False
)
CE_Q20_TX80_1961to1990_Intensity = xr.open_dataset(
    load_dir + "FigS12_Intensity_1961to1990.nc"
)

## abc

In [ ]:
# Create map features adder
add_map_features = create_map_features_adder()

# Example: Save the figure
fig = plot_three_panel_analysis(
    data_1961to1990=CE_Q20_TX80_1961to1990_FD,
    data_1991to2020=CE_Q20_TX80_1991to2020_FD,
    CE_MISSING_FLAG=-99999999,
    variable_name="mean_annual_frequency",
    cmap_sequential=viridis_mod,
    extend_colorbar="both",
    bins_map=[0.8, 1.3, 1.8, 2.3, 2.8],
    bins_delta=np.array([-3, -2, -1, -0.025, 0.025, 1, 2, 3]),
    yellow_range=(-0.025, 0.025),
    kde_ylim=(0, 1.4),
    kde_xlim=(0, 3.9),
    kde_yticks=(
        0,
        1,
    ),
    # kde_position_adjust=(0.05, -0.078, 0.00, 0.01), #(x_offset, y_offset, w_offset, h_offset) to adjust KDE panel position
    tick_positions_delta=[-3, -2, -1, 0, 1, 2, 3],
    # filename='FigS12_abc',  # Save with this name
    save_formats=["pdf"],
    dpi=300,
    titles={
        "panel1": "Events/Yr, 1991\u20132020",
        "panel2": "Relative Δ",  # "Relative \u0394 (91-20 vs 61-90)"
        "panel3": "Pan-European Distribution",
    },
    tick_interval=1,
    title_fontsize=12,
    cbar_tick_fontsize=12,
    add_map_features_func=add_map_features,
)
plt.show()

## def

In [ ]:
# Create map features adder
add_map_features = create_map_features_adder()

# Example: Save the figure
fig = plot_three_panel_analysis(
    data_1961to1990=convert_timedelta_var(
        CE_Q20_TX80_1961to1990_FD, "mean_annual_duration"
    ),
    data_1991to2020=convert_timedelta_var(
        CE_Q20_TX80_1991to2020_FD, "mean_annual_duration"
    ),
    CE_MISSING_FLAG=-99999999,
    variable_name="mean_annual_duration",
    cmap_sequential=viridis_mod,
    bins_map=[1.5, 2.5, 3.0, 3.5, 4.0],
    extend_colorbar="both",
    bins_delta=np.array([-3, -2, -1, -0.025, 0.025, 1, 2, 3]),
    yellow_range=(-0.025, 0.025),
    kde_ylim=(0, 1),
    kde_yticks=(0, 1.0),
    kde_xlim=(0, 5.8),
    tick_positions_delta=[-3, -2, -1, 0, 1, 2, 3],
    # filename='FigS12_def',  # Save with this name
    panel_labels=["d", "e", "f"],
    save_formats=["pdf"],
    dpi=300,
    titles={
        "panel1": "Days/Event, 1991\u20132020",
        "panel2": "Relative \u0394",  # "Relative \u0394 (91-20 vs 61-90)"
        "panel3": "Pan-European Distribution",
    },
    tick_interval=1,
    title_fontsize=12,
    cbar_tick_fontsize=12,
    add_map_features_func=add_map_features,
)
plt.show()

## g

In [ ]:
# ── four separate stat dicts, one per column ─────────────────────────────────
stats_ce_freq = compute_region_stats(
    data_old=CE_Q20_TX80_1961to1990_FD,
    data_new=CE_Q20_TX80_1991to2020_FD,
    region_mask=europe_PRUDENCE_regions,
    var_name="mean_annual_frequency",
)
stats_ce_dur = compute_region_stats(
    data_old=convert_timedelta_var(CE_Q20_TX80_1961to1990_FD, "mean_annual_duration"),
    data_new=convert_timedelta_var(CE_Q20_TX80_1991to2020_FD, "mean_annual_duration"),
    region_mask=europe_PRUDENCE_regions,
    var_name="mean_annual_duration",
)
stats_ce_lowflwdef = compute_region_stats(
    data_old=CE_Q20_TX80_1961to1990_Intensity,
    data_new=CE_Q20_TX80_1991to2020_Intensity,
    region_mask=europe_PRUDENCE_regions,
    var_name="mean_annual_flow_intensity_scaled",
)
stats_ce_temExs = compute_region_stats(
    data_old=CE_Q20_TX80_1961to1990_Intensity,
    data_new=CE_Q20_TX80_1991to2020_Intensity,
    region_mask=europe_PRUDENCE_regions,
    var_name="mean_annual_temperature_intensity_scaled",
)

# ── define columns as a list — easy to reorder, comment out, or swap ──────────
# h_jitter : horizontal jitter as a fraction of the x-range (0.0 = off).
#             Use a small value (e.g. 0.02–0.05) for integer-valued or
#             low-cardinality columns where dots would otherwise stack.
columns_def = [
    dict(
        stats=stats_ce_freq,
        title="Events per year",
        xlabel="Events per year",
        xmin=0.0,
        xmax=3.5,
        major_tick=1.0,
        h_jitter=0.05,
    ),
    dict(
        stats=stats_ce_dur,
        title="Days per event",
        xlabel="Days per event",
        xmin=0.0,
        xmax=6.3,
        major_tick=2.0,
        h_jitter=0.0,
    ),
    dict(
        stats=stats_ce_lowflwdef,
        title="Flow Deficit (%)",
        xlabel="Flow Deficit (%)",
        xmin=0.0,
        xmax=50.3,
        major_tick=15.0,
        h_jitter=0.0,
    ),
    dict(
        stats=stats_ce_temExs,
        title="Temp. Excess (%)",
        xlabel="Temp. Excess (%)",
        xmin=0.0,
        xmax=22.3,
        major_tick=5.0,
        h_jitter=0.0,
    ),
]

# ── reorder by simply changing the list order ─────────────────────────────────
# e.g. swap frequency and duration:
# columns_def = [columns_def[2], columns_def[3], columns_def[0], columns_def[1]]

# ── unpack and plot ───────────────────────────────────────────────────────────
fig = plot_strip_8x4(
    **{f"stats_col{i+1}": c["stats"] for i, c in enumerate(columns_def)},
    **{f"col{i+1}_title": c["title"] for i, c in enumerate(columns_def)},
    **{f"col{i+1}_xlabel": c["xlabel"] for i, c in enumerate(columns_def)},
    **{f"xmin_col{i+1}": c["xmin"] for i, c in enumerate(columns_def)},
    **{f"xmax_col{i+1}": c["xmax"] for i, c in enumerate(columns_def)},
    **{f"major_tick_col{i+1}": c["major_tick"] for i, c in enumerate(columns_def)},
    **{f"h_jitter_col{i+1}": c.get("h_jitter", 0.0) for i, c in enumerate(columns_def)},
    dot_stat="mean",  # or 'median'
    jitter_frac=0.50,  # vertical jitter, same for all columns
    col_header_fs=10,
    annot_fs=9,
    row_height_mm=11,
    region_alpha={
        "FR": 0.05,
        "ME": 0.05,
        "AL": 0.05,
        "EA": 0.05,
        "MD": 0.05,
        "SC": 0.015,
        "IP": 0.05,
        "BI": 0.05,
    },
    panel_label="(g)",
    # filename    = 'FigS12_g_strip_8x4',
    save_formats=["pdf"],
)

# Fig S13

In [ ]:
load_dir = str(FIGDATA_DIR) + os.sep

CE_Q5_TX95_1991to2020_FD = xr.open_dataset(
    load_dir + "FigS13_FD_1991to2020.nc", decode_times=False
)
CE_Q5_TX95_1991to2020_Intensity = xr.open_dataset(
    load_dir + "FigS13_Intensity_1991to2020.nc"
)
CE_Q5_TX95_1961to1990_FD = xr.open_dataset(
    load_dir + "FigS13_FD_1961to1990.nc", decode_times=False
)
CE_Q5_TX95_1961to1990_Intensity = xr.open_dataset(
    load_dir + "FigS13_Intensity_1961to1990.nc"
)

## abc

In [ ]:
# Create map features adder
add_map_features = create_map_features_adder()

# Example: Save the figure
fig = plot_three_panel_analysis(
    data_1961to1990=CE_Q5_TX95_1961to1990_FD,
    data_1991to2020=CE_Q5_TX95_1991to2020_FD,
    CE_MISSING_FLAG=-99999999,
    variable_name="mean_annual_frequency",
    cmap_sequential=viridis_mod,
    extend_colorbar="both",
    bins_map=[0.07, 0.16, 0.25, 0.34, 0.5],
    bins_delta=np.array([-3, -2, -1, -0.025, 0.025, 1, 2, 3]),
    yellow_range=(-0.025, 0.025),
    kde_ylim=(0, 12.4),
    kde_xlim=(0, 0.9),
    kde_yticks=(
        0,
        1,
    ),
    # kde_position_adjust=(0.05, -0.078, 0.00, 0.01), #(x_offset, y_offset, w_offset, h_offset) to adjust KDE panel position
    tick_positions_delta=[-3, -2, -1, 0, 1, 2, 3],
    # filename='FigS13_abc',  # Save with this name
    save_formats=["pdf"],
    dpi=300,
    titles={
        "panel1": "Events/Yr, 1991\u20132020",
        "panel2": "Relative Δ",  # "Relative \u0394 (91-20 vs 61-90)"
        "panel3": "Pan-European Distribution",
    },
    tick_interval=1,
    title_fontsize=12,
    cbar_tick_fontsize=12,
    add_map_features_func=add_map_features,
)
plt.show()

## def

In [ ]:
# Create map features adder
add_map_features = create_map_features_adder()

# Example: Save the figure
fig = plot_three_panel_analysis(
    data_1961to1990=convert_timedelta_var(
        CE_Q5_TX95_1961to1990_FD, "mean_annual_duration"
    ),
    data_1991to2020=convert_timedelta_var(
        CE_Q5_TX95_1991to2020_FD, "mean_annual_duration"
    ),
    CE_MISSING_FLAG=-99999999,
    variable_name="mean_annual_duration",
    cmap_sequential=viridis_mod,
    bins_map=[0.25, 0.45, 0.65, 0.9, 1.2],
    extend_colorbar="both",
    bins_delta=np.array([-3, -2, -1, -0.025, 0.025, 1, 2, 3]),
    yellow_range=(-0.025, 0.025),
    kde_ylim=(0, 4),
    kde_yticks=(0, 1.0),
    kde_xlim=(0, 2),
    tick_positions_delta=[-3, -2, -1, 0, 1, 2, 3],
    # filename='FigS13_def',  # Save with this name
    panel_labels=["d", "e", "f"],
    save_formats=["pdf"],
    dpi=300,
    titles={
        "panel1": "Days/Event, 1991\u20132020",
        "panel2": "Relative \u0394",  # "Relative \u0394 (91-20 vs 61-90)"
        "panel3": "Pan-European Distribution",
    },
    tick_interval=1,
    title_fontsize=12,
    cbar_tick_fontsize=12,
    add_map_features_func=add_map_features,
)
plt.show()

## g

In [ ]:
# ── four separate stat dicts, one per column ─────────────────────────────────
stats_ce_freq = compute_region_stats(
    data_old=CE_Q5_TX95_1961to1990_FD,
    data_new=CE_Q5_TX95_1991to2020_FD,
    region_mask=europe_PRUDENCE_regions,
    var_name="mean_annual_frequency",
)
stats_ce_dur = compute_region_stats(
    data_old=convert_timedelta_var(CE_Q5_TX95_1961to1990_FD, "mean_annual_duration"),
    data_new=convert_timedelta_var(CE_Q5_TX95_1991to2020_FD, "mean_annual_duration"),
    region_mask=europe_PRUDENCE_regions,
    var_name="mean_annual_duration",
)
stats_ce_lowflwdef = compute_region_stats(
    data_old=CE_Q5_TX95_1961to1990_Intensity,
    data_new=CE_Q5_TX95_1991to2020_Intensity,
    region_mask=europe_PRUDENCE_regions,
    var_name="mean_annual_flow_intensity_scaled",
)
stats_ce_temExs = compute_region_stats(
    data_old=CE_Q5_TX95_1961to1990_Intensity,
    data_new=CE_Q5_TX95_1991to2020_Intensity,
    region_mask=europe_PRUDENCE_regions,
    var_name="mean_annual_temperature_intensity_scaled",
)

# ── define columns as a list — easy to reorder, comment out, or swap ──────────
# h_jitter : horizontal jitter as a fraction of the x-range (0.0 = off).
#             Use a small value (e.g. 0.02–0.05) for integer-valued or
#             low-cardinality columns where dots would otherwise stack.
columns_def = [
    dict(
        stats=stats_ce_freq,
        title="Events per year",
        xlabel="Events per year",
        xmin=0.0,
        xmax=0.8,
        major_tick=0.5,
        h_jitter=0.05,
    ),
    dict(
        stats=stats_ce_dur,
        title="Days per event",
        xlabel="Days per event",
        xmin=0.0,
        xmax=2.1,
        major_tick=1.0,
        h_jitter=0.0,
    ),
    dict(
        stats=stats_ce_lowflwdef,
        title="Flow Deficit (%)",
        xlabel="Flow Deficit (%)",
        xmin=0.0,
        xmax=20.3,
        major_tick=10.0,
        h_jitter=0.0,
    ),
    dict(
        stats=stats_ce_temExs,
        title="Temp. Excess (%)",
        xlabel="Temp. Excess (%)",
        xmin=0.0,
        xmax=5.3,
        major_tick=2.0,
        h_jitter=0.0,
    ),
]

# ── reorder by simply changing the list order ─────────────────────────────────
# e.g. swap frequency and duration:
# columns_def = [columns_def[2], columns_def[3], columns_def[0], columns_def[1]]

# ── unpack and plot ───────────────────────────────────────────────────────────
fig = plot_strip_8x4(
    **{f"stats_col{i+1}": c["stats"] for i, c in enumerate(columns_def)},
    **{f"col{i+1}_title": c["title"] for i, c in enumerate(columns_def)},
    **{f"col{i+1}_xlabel": c["xlabel"] for i, c in enumerate(columns_def)},
    **{f"xmin_col{i+1}": c["xmin"] for i, c in enumerate(columns_def)},
    **{f"xmax_col{i+1}": c["xmax"] for i, c in enumerate(columns_def)},
    **{f"major_tick_col{i+1}": c["major_tick"] for i, c in enumerate(columns_def)},
    **{f"h_jitter_col{i+1}": c.get("h_jitter", 0.0) for i, c in enumerate(columns_def)},
    dot_stat="mean",  # or 'median'
    jitter_frac=0.50,  # vertical jitter, same for all columns
    col_header_fs=10,
    annot_fs=9,
    row_height_mm=11,
    region_alpha={
        "FR": 0.05,
        "ME": 0.05,
        "AL": 0.05,
        "EA": 0.05,
        "MD": 0.05,
        "SC": 0.015,
        "IP": 0.05,
        "BI": 0.05,
    },
    panel_label="(g)",
    # filename    = 'FigS13_g_strip_8x4',
    save_formats=["pdf"],
)

# Fig S14

In [ ]:
base_path = str(FIGDATA_DIR) + os.sep

spatial_annual_df = pd.read_parquet(
    base_path + "FigS14_data.parquet"
)

In [ ]:
# ============================================================================
# RELOAD DATA
# ============================================================================

warnings.filterwarnings("ignore")

# ============================================================================
# CONFIGURATION
# ============================================================================

PERIODS = {
    "Period 1": (1961, 1990),
    "Period 2": (1991, 2020),
    "Full Period": (1961, 2020),
}

WINDOW_SIZE = 10
ALPHA = 0.05
FIG_WIDTH = 7.29

# ============================================================================
# FUNCTION: TREND-FREE PREWHITENING
# ============================================================================


def trend_free_prewhitening(data, window_size=10):
    """
    Apply trend-free prewhitening with running window.
    """
    data = np.array(data)
    n = len(data)

    trend_component = uniform_filter1d(data, size=window_size, mode="nearest")
    detrended = data - trend_component

    if n > 1:
        autocorr = np.corrcoef(detrended[:-1], detrended[1:])[0, 1]
    else:
        autocorr = 0

    if abs(autocorr) > 0.1 and n > 1:
        prewhitened = detrended[1:] - autocorr * detrended[:-1]
        prewhitened = np.concatenate([[detrended[0]], prewhitened])
    else:
        prewhitened = detrended

    prewhitened_data = prewhitened + trend_component

    return prewhitened_data, trend_component


# ============================================================================
# FUNCTION: MODIFIED SEN'S SLOPE WITH MANN-KENDALL
# ============================================================================


def calculate_trend_statistics(
    years, data, alpha=0.05, apply_prewhitening=True, window_size=10
):
    """
    Calculate trend statistics using Modified Sen's Slope and Mann-Kendall test.
    """
    years = np.array(years)
    data = np.array(data)

    valid_idx = ~np.isnan(data)
    years_clean = years[valid_idx]
    data_clean = data[valid_idx]

    if len(data_clean) < 3:
        return {
            "slope": np.nan,
            "intercept": np.nan,
            "p_value": np.nan,
            "tau": np.nan,
            "trend": "insufficient data",
            "significant": False,
            "prewhitened_data": data_clean,
            "trend_component": np.zeros_like(data_clean),
        }

    if apply_prewhitening and len(data_clean) >= window_size:
        data_for_analysis, trend_component = trend_free_prewhitening(
            data_clean, window_size
        )
    else:
        data_for_analysis = data_clean
        trend_component = np.zeros_like(data_clean)

    mk_result = mk.original_test(data_for_analysis, alpha=alpha)

    n = len(data_for_analysis)
    slopes = []
    for i in range(n):
        for j in range(i + 1, n):
            if years_clean[j] != years_clean[i]:
                slope = (data_for_analysis[j] - data_for_analysis[i]) / (
                    years_clean[j] - years_clean[i]
                )
                slopes.append(slope)

    if slopes:
        sen_slope = np.median(slopes)
        medians_x = np.median(years_clean)
        medians_y = np.median(data_for_analysis)
        intercept = medians_y - sen_slope * medians_x
    else:
        sen_slope = 0
        intercept = np.median(data_for_analysis)

    if mk_result.p < alpha:
        if sen_slope > 0:
            trend_direction = "increasing"
        else:
            trend_direction = "decreasing"
        significant = True
    else:
        trend_direction = "no trend"
        significant = False

    tau_value = getattr(mk_result, "Tau", getattr(mk_result, "tau", np.nan))
    mk_stat = getattr(mk_result, "z", getattr(mk_result, "s", np.nan))

    return {
        "slope": sen_slope,
        "intercept": intercept,
        "p_value": mk_result.p,
        "tau": tau_value,
        "trend": trend_direction,
        "significant": significant,
        "mk_statistic": mk_stat,
        "prewhitened_data": data_for_analysis,
        "trend_component": trend_component,
        "original_data": data_clean,
        "years": years_clean,
    }


# ============================================================================
# MAIN ANALYSIS — panels a-f only: counts and durations, no ratios
# ============================================================================

variables = {
    "total_ce": {
        "label": "CE Count",
        "sd_col": "sd_ce_count",
        "ylabel": "Count",
        "unit": "events/yr",
    },
    "total_lf": {
        "label": "LFE Count",
        "sd_col": "sd_lf_count",
        "ylabel": "Count",
        "unit": "events/yr",
    },
    "total_hw": {
        "label": "HWE Count",
        "sd_col": "sd_hw_count",
        "ylabel": "Count",
        "unit": "events/yr",
    },
    "mean_ce_duration_all": {
        "label": "CE Duration (days/e)",
        "sd_col": "sd_ce_duration",
        "ylabel": "Duration (days)",
        "unit": "days/yr",
    },
    "mean_lf_duration_all": {
        "label": "LFE Duration (days/e)",
        "sd_col": "sd_lf_duration",
        "ylabel": "Duration (days)",
        "unit": "days/yr",
    },
    "mean_hw_duration_all": {
        "label": "HWE Duration (days/e)",
        "sd_col": "sd_hw_duration",
        "ylabel": "Duration (days)",
        "unit": "days/yr",
    },
}

all_results = {}

for var_name, var_info in variables.items():
    var_results = {}

    for period_name, (start_year, end_year) in PERIODS.items():
        period_df = spatial_annual_df[
            (spatial_annual_df["year"] >= start_year)
            & (spatial_annual_df["year"] <= end_year)
        ].copy()

        years = period_df["year"].values
        values = period_df[var_name].values

        if var_info["sd_col"] is not None and var_info["sd_col"] in period_df.columns:
            sd_values = period_df[var_info["sd_col"]].values
        else:
            sd_values = None

        trend_stats = calculate_trend_statistics(
            years, values, alpha=ALPHA, apply_prewhitening=True, window_size=WINDOW_SIZE
        )

        var_results[period_name] = {
            "years": years,
            "values": values,
            "sd_values": sd_values,
            "mean": np.nanmean(values),
            "std": np.nanstd(values),
            "median": np.nanmedian(values),
            "min": np.nanmin(values),
            "max": np.nanmax(values),
            "n": len(values),
            "trend": trend_stats,
        }

    all_results[var_name] = var_results

# ============================================================================
# CREATE 2x3 VISUALIZATION (panels a-f only)
# ============================================================================

plt.rcParams.update(plt.rcParamsDefault)

fig, axes = plt.subplots(2, 3, figsize=(FIG_WIDTH, FIG_WIDTH * 0.75))
fig.suptitle(
    "Trend Analysis: Modified Sen's Slope with Mann-Kendall Test",
    fontsize=10,
    fontweight="bold",
    y=0.995,
)

axes_flat = axes.flatten()

full_period_color = "#F18F01"

for idx, (var_name, var_info) in enumerate(variables.items()):
    if idx >= 6:
        break

    ax = axes_flat[idx]
    var_results = all_results[var_name]

    full_data = var_results["Full Period"]
    years = full_data["years"]
    values = full_data["values"]
    sd_values = full_data["sd_values"]

    if sd_values is not None:
        ax.errorbar(
            years,
            values,
            yerr=sd_values,
            fmt="o",
            markersize=4,
            alpha=0.5,
            color="#606060",
            ecolor="#B0B0B0",
            capsize=2,
            capthick=0.8,
            linewidth=0.8,
            zorder=2,
        )
    else:
        ax.scatter(years, values, s=25, alpha=0.7, color="#606060", zorder=2)

    trend_stats = var_results["Full Period"]["trend"]

    if not np.isnan(trend_stats["slope"]):
        start_year, end_year = PERIODS["Full Period"]
        x_trend = np.array([start_year, end_year])
        y_trend = trend_stats["slope"] * x_trend + trend_stats["intercept"]

        linestyle = "-" if trend_stats["significant"] else "--"
        linewidth = 2.5 if trend_stats["significant"] else 1.8
        alpha = 0.9 if trend_stats["significant"] else 0.6

        ax.plot(
            x_trend,
            y_trend,
            linestyle=linestyle,
            linewidth=linewidth,
            color=full_period_color,
            alpha=alpha,
            zorder=4,
            label="Full Period trend",
        )

        if "duration" in var_name.lower():
            slope_text = f"{trend_stats['slope']:.4f}"
        elif "ce_per" in var_name.lower():
            slope_text = f"{trend_stats['slope']:.5f}"
        else:
            slope_text = f"{trend_stats['slope']:.2f}"

        sig_marker = "*" if trend_stats["significant"] else "ns"

        ann_text = (
            f"Slope: {slope_text}\np = {trend_stats['p_value']:.3f} ({sig_marker})"
        )

        if np.nanmean(values[-10:]) > np.nanmean(values[:10]):
            ann_x, ann_y = 0.05, 0.95
            ha_pos = "left"
        else:
            ann_x, ann_y = 0.95, 0.95
            ha_pos = "right"

        ax.text(
            ann_x,
            ann_y,
            ann_text,
            transform=ax.transAxes,
            fontsize=6.5,
            verticalalignment="top",
            horizontalalignment=ha_pos,
            bbox=dict(
                boxstyle="round,pad=0.4",
                facecolor="white",
                alpha=0.9,
                edgecolor="gray",
                linewidth=0.8,
            ),
        )

    ax.set_xlabel("Year", fontsize=8)
    ax.set_title(
        f"{chr(97 + idx)}) {var_info['label']}",
        fontsize=9,
        fontweight="bold",
        loc="left",
    )
    ax.grid(True, alpha=0.15, linestyle="-", linewidth=0.5, color="gray")

    ax.set_xlim(1960, 2021)
    ax.set_xticks(np.arange(1960, 2021, 2))
    ax.set_xticklabels(
        [str(y) if y % 10 == 0 else "" for y in np.arange(1960, 2021, 2)], fontsize=7
    )

    ax.tick_params(axis="both", which="major", labelsize=7)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.subplots_adjust(
    left=0.08, right=0.96, top=0.92, bottom=0.08, hspace=0.35, wspace=0.35
)

save_dir = str(FIGURE_DIR) + os.sep
savename = "FigS14_annual_trend"

if savename:
    os.makedirs(save_dir, exist_ok=True)
    fig.savefig(
        os.path.join(save_dir, f"{savename}.pdf"), format="pdf", bbox_inches="tight"
    )
    print(f"✓ Figure saved: {savename}")

plt.show()

# Fig S15

In [ ]:
DATA_DIR = str(FIGDATA_DIR) + os.sep

grid = pd.read_csv(DATA_DIR + 'FigS15_data.csv')
top  = pd.read_csv(DATA_DIR + 'FigS15_top10_years.csv').sort_values('rank')
meta = json.load(open(DATA_DIR + 'FigS15_plot_params.json'))

# --- Europe outline (must be lon/lat to match transform=PlateCarree) ---
europe_outline = gpd.read_file(str(FIGDATA_DIR / "europe_outline.gpkg"))
#print("Outline CRS:", europe_outline.crs)
# if europe_outline.crs is not None and europe_outline.crs.to_epsg() != 4326:
#     europe_outline = europe_outline.to_crs(epsg=4326)

bins = np.array(meta['tick_bins'])
norm = BoundaryNorm(np.concatenate(([-1e6], bins, [1e6])), ncolors=meta['N_BINS'])
cmap = ListedColormap(meta['colors'])

# --- Projection (as used in the main figure) ---
projection = ccrs.LambertConformal(
    central_latitude=53,
    central_longitude=12,
    standard_parallels=(45, 65)
)

fig = plt.figure(figsize=(9, 9))
gs  = fig.add_gridspec(3, 4, hspace=0.1, wspace=0.05,
                       top=0.93, bottom=0.08, left=0.05, right=0.98)

for idx, (_, r) in enumerate(top.iterrows()):
    yr, rk = int(r['year']), int(r['rank'])
    ax = fig.add_subplot(gs[idx // 4, idx % 4], projection=projection)
    ax.set_extent(meta['map_extent'], crs=ccrs.PlateCarree())

    # no axes frame
    ax.spines['geo'].set_visible(False)
    ax.set_frame_on(False)

    da = (grid[grid['year'] == yr]
          .pivot_table(index='lat', columns='lon', values='CEMId')
          .pipe(lambda d: xr.DataArray(d.values,
                                       coords={'lat': d.index, 'lon': d.columns},
                                       dims=['lat', 'lon'])))

    im = da.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        add_colorbar=False,
        cmap=cmap,
        norm=norm,
        zorder=2,
        rasterized=True,
        antialiased=False,
        edgecolors='none',
        linewidth=0
    )

    europe_outline.boundary.plot(ax=ax, transform=ccrs.PlateCarree(),
                                 edgecolor='black', linewidth=0.5, zorder=3)

    ax.set_title(f'Rank {rk}: {yr}', fontsize=10, pad=4)

cax  = fig.add_axes([0.55, 0.2, 0.4, 0.02])
cbar = plt.colorbar(im, cax=cax, orientation='horizontal', spacing='uniform')
cbar.set_ticks(bins)
cbar.set_ticklabels([f'{b:.2f}' for b in bins])
cbar.set_label(meta['colorbar_label'], fontsize=10, fontweight='bold')
cbar.ax.tick_params(labelsize=10, width=0.8, length=5)
cbar.outline.set_linewidth(0.8)

fig.suptitle('Spatial Distribution of Annual Maximum CEMId for Top 10 Extreme Years',
             fontsize=12, fontweight='bold', y=1.01)

plt.savefig(FIGURE_DIR / 'FigS15.pdf', format='pdf', bbox_inches='tight')
plt.show()